# r10 figures and decks (Student-t primary model), v4 style

In [ ]:
# 1. Install, fonts and Drive
import subprocess, sys
subprocess.run("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections && "
               "apt-get -qq install -y ttf-mscorefonts-installer fonts-liberation > /dev/null 2>&1", shell=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-pptx==1.0.2", "python-docx==1.1.2"])
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
print("Arial available:", any("Arial" in f.name for f in fm.fontManager.ttflist))
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content")


In [ ]:
%%writefile /content/fig_common.py
import io, json, shutil, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image

BASE = Path("/content/drive/MyDrive/new_dysfunction")
SENS = BASE / "normalized_state_bhm_ri_hc_v1"
ROOT = SENS / "_primary_sensitivity"
PRIM = ROOT / "_primary_summaries"
PPCR = ROOT / "_ppc_reports"
SIM_ROOT = BASE / "k1_validation_fine4_mount5_cpu_v1"
ARCH = {"immune_coarse": "v2p2_g1_fixed_ctnu", "immune_fine": "v2p1c_g1_fixed_gnu", "nonimmune": "v2p1c_g1_fixed_gnu"}
FAMS = ["immune_coarse", "immune_fine", "nonimmune"]
FAM_LABEL = {"immune_coarse": "Immune coarse", "immune_fine": "Immune fine", "nonimmune": "Non-immune"}
FAM_PREFIX = {"immune_coarse": "C", "immune_fine": "F", "nonimmune": "N"}
ALLREF = {"immune_coarse": "allprofiles_long", "immune_fine": "allprofiles", "nonimmune": "allprofiles_ta995"}
RS_ALL = {"immune_coarse": "readscale_all", "immune_fine": "readscale_all_long", "nonimmune": "readscale_all"}
INT_WIDE = {"immune_coarse": "int_wide", "immune_fine": "int_wide", "nonimmune": "int_wide_ta995"}

OUT = BASE / "paper_A4_figures" / "r10"
TIFF_DIR, SVG_DRIVE = OUT / "tiff", OUT / "svg"
SVG_LOCAL = Path("/content/r10_svg")
for d in (TIFF_DIR, SVG_DRIVE, SVG_LOCAL):
    d.mkdir(parents=True, exist_ok=True)
MANIFEST, VALUES = OUT / "manifest.csv", OUT / "values.json"

BLUE, LIGHT_BLUE, MAGENTA, LIGHT_MAGENTA, TEAL = "#2C5AA0", "#A9BEDF", "#D81B60", "#F2B6CB", "#0E7C7B"
INK, MID_GREY, BAND_GREY, GRID_GREY, EMPTY_CELL, ZEBRA = "#1F1F1F", "#6F7378", "#E8E8EA", "#DCDDE0", "#FAF9F6", "#F4F4F6"
GREY, LINE = MID_GREY, "#B9BCC2"
FAM_COLOR = {"immune_coarse": BLUE, "immune_fine": MAGENTA, "nonimmune": TEAL}
DIVERGING = mpl.colors.LinearSegmentedColormap.from_list("bmi_diverging", [BLUE, "#FFFFFF", MAGENTA])
FONT = 10
SIZE = {"value": 9, "axis_line": 0.9, "grid_line": 0.45, "interval_line": 1.9, "zero_line": 1.0, "marker": 32}
EFFECT_LABEL = "Change in pathway score (SD)\nper 5 kg/m² higher BMI"
CONTRAST_LABEL = "Group difference in\npathway score (SD)"
EFFECT_SHORT = "Change in score (SD)\nper 5 kg/m²"
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"], "svg.fonttype": "none",
    "font.size": FONT, "axes.labelsize": FONT, "xtick.labelsize": FONT, "ytick.labelsize": FONT, "legend.fontsize": FONT,
    "axes.edgecolor": INK, "axes.linewidth": SIZE["axis_line"], "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelpad": 4, "xtick.major.size": 3.5, "ytick.major.size": 3.5, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "xtick.color": INK, "ytick.color": INK, "text.color": INK, "axes.labelcolor": INK, "legend.frameon": False,
    "grid.color": GRID_GREY, "grid.linewidth": SIZE["grid_line"], "figure.facecolor": "white", "savefig.facecolor": "white"})
MM = 1 / 25.4

def panel(w_mm, h_mm, left=16, bottom=13, right=3, top=3):
    """Figure drawn at its final size; margins (mm) hold tick and axis labels. No titles."""
    fig = plt.figure(figsize=(w_mm * MM, h_mm * MM))
    ax = fig.add_axes([left / w_mm, bottom / h_mm, 1 - (left + right) / w_mm, 1 - (bottom + top) / h_mm])
    return fig, ax

def style_axis(ax, grid_axis="x"):
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(SIZE["axis_line"]); ax.spines[side].set_color(INK)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID_GREY, linewidth=SIZE["grid_line"], zorder=0); ax.set_axisbelow(True)

def row_axis(ax):
    """Forest-style y axis: no spine, no ticks."""
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0, pad=4)

def zebra(ax, ys, half=0.5, horizontal=True):
    for k, y in enumerate(ys):
        if k % 2 == 0:
            (ax.axhspan if horizontal else ax.axvspan)(y - half, y + half, color=ZEBRA, zorder=0, lw=0)

def dir_colors(mean):
    return np.where(np.asarray(mean, float) >= 0, MAGENTA, BLUE)

def points(ax, x, y, colors, filled, marker="o", size=None):
    """v4 markers: filled with a white edge for strong directional evidence, open otherwise."""
    size = size or SIZE["marker"]
    x, y, colors, filled = (np.asarray(v) for v in (x, y, colors, filled))
    filled = filled.astype(bool)
    if filled.any():
        ax.scatter(x[filled], y[filled], s=size, c=colors[filled], marker=marker, edgecolor="white", linewidth=0.7, zorder=4)
    if (~filled).any():
        ax.scatter(x[~filled], y[~filled], s=size, facecolor="white", edgecolor=colors[~filled], marker=marker,
                   linewidth=1.2, zorder=4)

def strong(p):
    p = np.asarray(p, float); return (p >= 0.95) | (p <= 0.05)

def forest(ax, y, mean, lo, hi, p, lw=None, colors=None, marker="o", size=None):
    colors = dir_colors(mean) if colors is None else np.asarray(colors)
    ax.hlines(y, lo, hi, colors=colors, linewidth=lw or SIZE["interval_line"], zorder=2, capstyle="round")
    points(ax, mean, y, colors, strong(p), marker=marker, size=size)

def symmetric_span(*arrays, pad=1.1):
    v = np.concatenate([np.asarray(a, float).ravel() for a in arrays]); v = v[np.isfinite(v)]
    return float(np.max(np.abs(v)) * pad) if len(v) else 1.0

def right_column(ax, y, texts, header=None, colors=None, x=1.02):
    for k, (yy, t) in enumerate(zip(y, texts)):
        ax.text(x, yy, t, transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=SIZE["value"],
                color=INK if colors is None else colors[k])
    if header:
        ax.text(x, 1.0, header, transform=ax.transAxes, ha="left", va="bottom", fontsize=SIZE["value"], color=MID_GREY)

def evidence_handles():
    def h(color, filled, label):
        return Line2D([0], [0], color=color, lw=SIZE["interval_line"], marker="o", markersize=6,
                      markerfacecolor=color if filled else "white", markeredgecolor="white" if filled else color,
                      markeredgewidth=0.7 if filled else 1.2, label=label)
    return [h(MAGENTA, True, "Positive posterior mean, P(>0) ≥ 0.95"), h(MAGENTA, False, "Positive posterior mean, P(>0) < 0.95"),
            h(BLUE, True, "Negative posterior mean, P(>0) ≤ 0.05"), h(BLUE, False, "Negative posterior mean, P(>0) > 0.05")]

def _manifest_upsert(row):
    m = pd.read_csv(MANIFEST) if MANIFEST.exists() else pd.DataFrame()
    if len(m):
        m = m[~((m.figure == row["figure"]) & (m.slot == row["slot"])) & (m.file != row["file"])]
    pd.concat([m, pd.DataFrame([row])], ignore_index=True).to_csv(MANIFEST, index=False)

def fit_inside(fig, pad_in=0.02, tries=4):
    """PowerPoint clips an SVG at its canvas edge; pull every axes inward until all drawn content (tick labels,
    rotated labels, colour-bar text) lies inside the canvas. Size of the canvas (= slot) is unchanged."""
    W, H = fig.get_size_inches()
    for _ in range(tries):
        fig.canvas.draw(); bb = fig.get_tightbbox(fig.canvas.get_renderer())
        l, r = max(0.0, -bb.x0), max(0.0, bb.x1 - W); b, t = max(0.0, -bb.y0), max(0.0, bb.y1 - H)
        if max(l, r, b, t) <= 0.005:
            return True
        l, r, b, t = [v + pad_in if v > 0 else 0.0 for v in (l, r, b, t)]
        sx, sy = (W - l - r) / W, (H - b - t) / H
        for ax in fig.axes:
            p = ax.get_position()
            ax.set_position([l / W + p.x0 * sx, b / H + p.y0 * sy, p.width * sx, p.height * sy])
    print(f"WARNING: content still outside the canvas after {tries} passes")
    return False

def save(fig, figure, slot, stem, source, kind="panel"):
    name = f"{figure}_{slot}_{stem}"
    if kind != "key" and not str(stem).endswith("schematic"):
        fit_inside(fig)
    svg = SVG_LOCAL / f"{name}.svg"
    fig.savefig(svg, format="svg")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=600); buf.seek(0)
    Image.open(buf).convert("RGB").save(TIFF_DIR / f"{name}.tiff", format="TIFF", compression="tiff_lzw", dpi=(600, 600))
    shutil.copy2(svg, SVG_DRIVE / svg.name)
    w, h = fig.get_size_inches() / MM
    plt.close(fig)
    _manifest_upsert(dict(figure=figure, slot=slot, kind=kind, file=svg.name, tiff=f"{name}.tiff",
                          width_mm=round(w, 2), height_mm=round(h, 2), source=str(source), created=time.strftime("%Y-%m-%d %H:%M")))
    print("saved", svg.name, f"{w:.0f} x {h:.0f} mm")

def save_key(figure, slot, stem, handles, labels, w_mm, h_mm, ncol=1, source="key"):
    """Key drawn inside its own box: columns are reduced until the key fits the width (PowerPoint crops anything outside)."""
    fig = plt.figure(figsize=(w_mm * MM, h_mm * MM))
    for n in range(min(ncol, len(labels)), 0, -1):
        leg = fig.legend(handles, labels, loc="center", ncol=n, frameon=False, handlelength=1.8, handleheight=0.9,
                         columnspacing=1.2, handletextpad=0.5, labelspacing=0.45, borderaxespad=0)
        fig.canvas.draw(); bb = leg.get_window_extent()
        if bb.width <= fig.bbox.width * 0.98 or n == 1:
            break
        leg.remove()
    if bb.width > fig.bbox.width * 0.98 or bb.height > fig.bbox.height * 0.98:
        print(f"WARNING {figure} {slot}: key {bb.width / fig.dpi / MM:.0f} x {bb.height / fig.dpi / MM:.0f} mm "
              f"does not fit its {w_mm} x {h_mm} mm box; enlarge the slot in cell 3")
    save(fig, figure, slot, stem, source, kind="key")

def save_colorbar(figure, slot, stem, cmap, vmin, vmax, label, w_mm, h_mm, vertical=True):
    fig = plt.figure(figsize=(w_mm * MM, h_mm * MM))
    cax = fig.add_axes([0.08, 0.06, 0.16, 0.88] if vertical else [0.15, 0.62, 0.7, 0.22])
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin, vmax), cmap=cmap), cax=cax,
                      orientation="vertical" if vertical else "horizontal")
    cb.outline.set_linewidth(0.9); cb.ax.tick_params(length=3, width=0.8, labelsize=FONT)
    cb.set_label(label, fontsize=FONT, labelpad=6)
    fit_inside(fig); save(fig, figure, slot, stem, "colourbar", kind="key")

def set_value(**kw):
    v = json.loads(VALUES.read_text()) if VALUES.exists() else {}
    for k, x in kw.items():
        v[k] = float(x) if isinstance(x, np.floating) else int(x) if isinstance(x, np.integer) else x
    VALUES.write_text(json.dumps(v, indent=2, ensure_ascii=False))

CT_NAMES = {"CD4+ T CELLS": "CD4⁺ T cells", "CD8_T_GENERAL": "CD8⁺ T cells", "CD8_T_EXHAUSTED": "Exh. CD8⁺ T",
            "CD4_T_REGULATORY": "Regulatory T", "T_CELLS": "T cells", "T_GAMMA_DELTA": "γδ T cells",
            "B_CELLS": "B cells", "B_CELLS_NAIVE": "Naive B cells", "NK_CELLS": "NK cells", "BASOPHILS": "Basophils",
            "DENDRITIC_CELLS": "Dendritic cells", "MDC": "mDC", "PDC": "pDC", "TAM": "TAM", "MONOCYTES": "Monocytes",
            "MONOCYTE_CLASSICAL": "Class. mono.", "MONOCYTE_NON_CLASSICAL": "Non-class. mono.",
            "ADIPOCYTES": "Adipocytes", "NORMAL_DUCTAL": "Normal ductal", "STELLATE": "Stellate", "TUMOR_EPITHELIAL": "Tumour epithelial"}
def ct_label(name, prefix=None):
    raw = str(name).split("||")[-1]
    lab = CT_NAMES.get(raw, raw.replace("_", " ").capitalize())
    return f"{prefix}: {lab}" if prefix else lab

def sig_label(feature):
    return str(feature).split("||")[-1].replace("_Signature", "").replace("_", " ")

def celltype_order():
    out = []
    for fam in FAMS:
        out += [(fam, x) for x in pd.read_csv(PRIM / f"{fam}_celltype.csv").celltype]
    return out

def block_positions(order, gap=1.0):
    pos, x, prev = [], 0.0, None
    for fam, _ in order:
        if prev is not None and fam != prev:
            x += gap
        pos.append(x); x += 1.0; prev = fam
    return np.array(pos)

def load_primary():
    PF = pd.concat([pd.read_csv(PRIM / f"{f}_feature.csv").assign(family=f) for f in FAMS], ignore_index=True)
    PC = pd.concat([pd.read_csv(PRIM / f"{f}_celltype.csv").assign(family=f) for f in FAMS], ignore_index=True)
    PF["celltype"] = PF.feature.str.split("||", regex=False).str[:2].str.join("||")
    PF["signature"] = PF.feature.map(sig_label)
    return PF, PC

def load_sim2(refresh=False):
    """Per-replicate metrics from SIM_ROOT/tasks_v2; cached as sim2_by_replicate.csv."""
    cache = SIM_ROOT / "sim2_by_replicate.csv"
    if cache.exists() and not refresh:
        return pd.read_csv(cache)
    rows = []
    for p in sorted((SIM_ROOT / "tasks_v2").glob("*.json")):
        r = json.loads(p.read_text()); base = dict(scenario=r["scenario"], model=r["model"], replicate=r["replicate"])
        if r["model"] in ("ols", "ols_group"):
            blocks = [(None, r["feature"])] if r["model"] == "ols" else list(r["feature"].items())
            for contrast, m in blocks:
                for tag in ("unadj", "bh"):
                    rows.append(dict(base, level="feature", contrast=contrast, method_tag=tag, converged_first=True, converged=True,
                                     coverage=m["coverage"] if tag == "unadj" else np.nan, width=m["mean_width"] if tag == "unadj" else np.nan,
                                     rmse=m["rmse"], false=m[f"{tag}_false"], n_null=m["n_null"], true=m[f"{tag}_true"],
                                     n_nonnull=m["n_nonnull"], calls=m[f"{tag}_calls"], minutes=0.0))
            continue
        a = r["attempts"][r["final_attempt"]]
        for level in ("feature", "celltype"):
            blocks = list(a[level].items()) if r["model"] == "group_t" else [(None, a[level])]
            for contrast, m in blocks:
                rows.append(dict(base, level=level, contrast=contrast, method_tag="hdi", converged_first=r["converged_first"],
                                 converged=r["converged"], coverage=m["coverage"], width=m["mean_width"], rmse=m["rmse"],
                                 false=m["false_calls"], n_null=m["n_null"], true=m["true_calls"], n_nonnull=m["n_nonnull"],
                                 calls=m["n_calls"], minutes=r["minutes"], n_attempts=len(r["attempts"])))
    df = pd.DataFrame(rows); df.to_csv(cache, index=False); return df

def hc_t_by_replicate(scenarios):
    """First-study Student-t rows in the same shape as load_sim2() (feature and cell-type levels)."""
    M = pd.read_csv(SIM_ROOT / "task_metrics_verified.csv"); H = M[M.model.eq("hc_t") & M.scenario.isin(scenarios)]
    out = []
    for level in ("feature", "celltype"):
        f = lambda c: H[f"final_{level}_{c}"].to_numpy()
        out.append(pd.DataFrame(dict(scenario=H.scenario.to_numpy(), model="hc_t", replicate=H.replicate.to_numpy(), level=level,
                                     contrast=None, method_tag="hdi", converged_first=H.converged_first.to_numpy(),
                                     converged=H.converged_final.to_numpy(), coverage=f("coverage"), width=f("mean_width"),
                                     rmse=f("rmse"), false=f("false_calls"), n_null=f("n_null"), true=f("true_calls"),
                                     n_nonnull=f("n_nonnull"), calls=f("n_calls"))))
    return pd.concat(out, ignore_index=True)

PUB = BLUE
T_NAME, O_NAME = "Student-t intercept", "Orthogonal patient-effect"
METHOD_STYLE = {T_NAME: (MAGENTA, "o"), O_NAME: (BLUE, "o"), "OLS": (INK, "^"), "OLS + BH": (INK, "v"),
                "Composite": (MID_GREY, "D"), "Group model": (TEAL, "o"), "OLS by group": (INK, "^"), "OLS by group + BH": (INK, "v"),
                "One-factor": (MID_GREY, "^")}
SLUG = {T_NAME: "t", O_NAME: "orth", "OLS": "ols", "OLS + BH": "bh"}

MO = BASE / "multiomics_extensions"
R_DIR = MO / "rna_protein_original_signatures_v2/shards_4/aggregate/publication/056205a4e2ca"
P_DIR = MO / "protein_ptm_paired_v2/publication/2687095c8b4e1e44/shards_4/aggregate"
DA_DIR = P_DIR / "deeper_analysis_20260925T152006651162Z" / "tables"
X_DIR = BASE / "paper_A4_figures/PTM_Extended_Figures_20261004T161611_905289Z/tables"
B_DIR = BASE / "paper_A4_figures/bulk_PTM_plotting/PDAC_Bulk_PTM_Plots_20261011T215214_372590Z/tables"
DRIVERS = BASE / "paper_A4_figures/headline_gene_drivers_student_t"
TABLES = OUT / "tables"; TABLES.mkdir(parents=True, exist_ok=True)
PTM_UNIT = "Change (SD) per 5 kg/m²\nwithin plex"
RP_UNIT = "Change (SD) per 5 kg/m²"
ASSAY_LABEL = {"phospho": "Phosphosites", "glyco": "Glycoforms"}
ASSAY_COLOR = {"phospho": BLUE, "glyco": MAGENTA}

def rd(path, **kw):
    """CSV reader that keeps literal strings such as 'null' (a scenario name) instead of turning them into NaN."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False, **kw)

def tb(s):
    return pd.Series(s).astype(str).str.strip().str.lower().isin(["true", "1", "1.0"]).to_numpy()

def hdi_forest(ax, y, mean, lo, hi, excl, colors=None, marker="o", size=None, lw=None):
    """Forest row for HDI-based outputs: filled marker when the 95% HDI excludes zero, open otherwise."""
    colors = dir_colors(mean) if colors is None else np.asarray(colors)
    ax.hlines(y, lo, hi, colors=colors, linewidth=lw or SIZE["interval_line"], zorder=2, capstyle="round")
    points(ax, mean, y, colors, np.asarray(excl, bool), marker=marker, size=size)

def hdi_handles(pos="Positive, 95% HDI excludes zero", neg="Negative, 95% HDI excludes zero"):
    def h(color, filled, label):
        return Line2D([0], [0], color=color, lw=SIZE["interval_line"], marker="o", markersize=6,
                      markerfacecolor=color if filled else "white", markeredgecolor="white" if filled else color,
                      markeredgewidth=0.7 if filled else 1.2, label=label)
    return [h(MAGENTA, True, pos), h(BLUE, True, neg), h(MID_GREY, False, "95% HDI includes zero")]

def programme_labels():
    """RNA–protein gene sets: label from the first source signature; number of source features sharing the gene set."""
    cov = rd(R_DIR / "inputs_used" / "coverage.csv")
    cov = cov[tb(cov.included)]
    lab, n = {}, cov.groupby("programme").size().to_dict()
    for r in cov.itertuples():
        lab.setdefault(r.programme, f"{FAM_PREFIX[r.family]}: {ct_label(r.CellType)} · {sig_label(r.Signature)}")
    return lab, n

def rp_effects():
    """Primary RNA–protein effects, one row per gene set and quantity."""
    e = rd(R_DIR / "results" / "all_effects_95HDI.csv"); e = e[e.variant.eq("primary")].copy()
    e["excl"] = (e.hdi_95_lower > 0) | (e.hdi_95_upper < 0)
    return e

def ptm_primary():
    """Primary protein–PTM fits: PTM_given_Protein term bmi_5_within, every eligible feature (1,648) with diagnostics."""
    a = rd(DA_DIR / "all_primary_features.csv")
    a["adequate"] = tb(a.numerical_checks_pass); a["excl"] = tb(a.hdi_excludes_zero)
    a["lfsr"] = np.minimum(a.probability_positive, 1 - a.probability_positive)
    return a

def site_label(gene, feature_id):
    """Phosphosite 'GENE S252'; glycoform 'GENE N2H5F0S0' (glycan composition, no peptide)."""
    f = str(feature_id)
    if "-N" in f and "[" in f:
        comp = f.rsplit("-", 1)[-1]; comp = comp[:-2] if comp.endswith("G0") else comp
        return f"{gene} {comp}"
    return f"{gene} {f.rsplit('_', 1)[-1]}"

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def group_bands(ax, y, groups, color="#F7F7F8"):
    """Very light grey band behind every other group of rows (e.g. analysis blocks), instead of per-row stripes."""
    y = np.asarray(y, float); groups = list(groups); k = 0; start = 0
    for i in range(1, len(groups) + 1):
        if i == len(groups) or groups[i] != groups[start]:
            if k % 2 == 0:
                ax.axhspan(y[i - 1] - 0.5, y[start] + 0.5, color=color, zorder=0, lw=0)
            k += 1; start = i

def heat(ax, values, vmax, row_labels, col_labels, rings=None, crosses=None, row_colors=None, col_rotation=90, fontsize=None,
         cmap=None, vmin=None, text_fmt=None, col_breaks=()):
    """The single heatmap style of the paper (as Figure 3): diverging blue–white–pink, off-white empty cells,
    thin grey cell grid, white rings for marked cells, black frame."""
    v = np.asarray(values, float); n_r, n_c = v.shape; fs = fontsize or FONT - 1
    ax.set_facecolor(EMPTY_CELL)
    cm = cmap or DIVERGING; lo = -vmax if vmin is None else vmin
    ax.imshow(np.ma.masked_invalid(v), cmap=cm, vmin=lo, vmax=vmax, aspect="equal", interpolation="nearest", extent=(0, n_c, n_r, 0))
    ax.set_anchor("NW")
    if text_fmt:
        for r in range(n_r):
            for c in range(n_c):
                if np.isfinite(v[r, c]):
                    dark = (v[r, c] - lo) / (vmax - lo) > 0.75 or (v[r, c] - lo) / (vmax - lo) < 0.12 and cm is DIVERGING
                    ax.text(c + 0.5, r + 0.5, text_fmt.format(v[r, c]), ha="center", va="center", fontsize=SIZE["value"],
                            color="white" if dark else INK, zorder=6)
    for b in col_breaks:
        ax.axvline(b, color=INK, lw=0.9, zorder=6)
    for (mask, kind) in [(rings, "ring"), (crosses, "cross")]:
        if mask is None: continue
        m = np.asarray(mask, bool)
        for r, c in zip(*np.where(m)):
            if kind == "ring":
                ax.add_patch(mpatches.Circle((c + 0.5, r + 0.5), 0.22, facecolor="white", edgecolor=INK, lw=0.7, zorder=5))
            else:
                ax.plot([c + 0.3, c + 0.7], [r + 0.3, r + 0.7], color=INK, lw=0.9, zorder=5)
                ax.plot([c + 0.3, c + 0.7], [r + 0.7, r + 0.3], color=INK, lw=0.9, zorder=5)
    ax.set_xticks(np.arange(n_c + 1), minor=True); ax.set_yticks(np.arange(n_r + 1), minor=True)
    ax.grid(which="minor", color="#9A9DA2", linewidth=0.3); ax.tick_params(which="both", length=0, pad=2)
    ax.set_xticks(np.arange(n_c) + 0.5, col_labels, rotation=col_rotation, ha="center" if col_rotation == 90 else "right", fontsize=fs)
    ax.set_yticks(np.arange(n_r) + 0.5, row_labels, fontsize=fs, fontweight="bold")
    if row_colors is not None:
        for lab, c in zip(ax.get_yticklabels(), row_colors): lab.set_color(c)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.9); sp.set_edgecolor(INK)

def shared_row_axes(w_mm, h_mm, left, n, gap=4, right=3, bottom=13, top=2, widths=None):
    """One figure, n side-by-side axes sharing the same rows (labels drawn on the first only)."""
    fig = plt.figure(figsize=(w_mm * MM, h_mm * MM)); avail = w_mm - left - right - gap * (n - 1)
    widths = widths or [1] * n; tot = sum(widths); axes, x = [], left
    for wv in widths:
        cw = avail * wv / tot
        axes.append(fig.add_axes([x / w_mm, bottom / h_mm, cw / w_mm, (h_mm - bottom - top) / h_mm])); x += cw + gap
    return fig, axes

def dumbbell(ax, y, series, xmax=1.0, pct=True, connector=LIGHT_MAGENTA, size=110, vfs=6.5):
    """Previous-version comparison style: open circles with the value printed inside, joined by a pale connector.
    Where circles of one row overlap, they are left overlapping and their values are printed just outside instead
    (first above, second below, any further to the right), each in its method's colour. series = [(values, colour), ...]."""
    y = np.asarray(y, float); vals = np.array([np.asarray(v, float) for v, _ in series]); cols = [c for _, c in series]
    ax.set_xlim(-0.04 * xmax, xmax * 1.04)
    lo, hi = np.nanmin(vals, axis=0), np.nanmax(vals, axis=0)
    ax.hlines(y, lo, hi, color=connector, lw=3.2, zorder=1, alpha=0.8)
    fig = ax.figure; w_mm = ax.get_position().width * fig.get_size_inches()[0] / MM
    d_data = (np.sqrt(size) / 72 * 25.4) * 1.05 * (ax.get_xlim()[1] - ax.get_xlim()[0]) / w_mm
    fmt = (lambda v: f"{v * 100:.0f}") if pct else (lambda v: f"{v:.2f}")
    for j, yy in enumerate(y):
        v = vals[:, j]; ok = np.where(np.isfinite(v))[0]
        for k in ok:
            ax.scatter([v[k]], [yy], s=size, facecolor="white", edgecolor=cols[k], lw=1.1, zorder=3 + k * 0.01)
        order = ok[np.argsort(v[ok])]
        clusters, cur = [], ([order[0]] if len(order) else [])
        for k in order[1:]:
            if v[k] - v[cur[-1]] < d_data: cur.append(k)
            else: clusters.append(cur); cur = [k]
        if cur: clusters.append(cur)
        for cl in clusters:
            if len(cl) == 1:
                k = cl[0]; ax.text(v[k], yy, fmt(v[k]), ha="center", va="center", fontsize=vfs, color=cols[k], zorder=4)
                continue
            mid = float(np.mean(v[cl]))
            for n, k in enumerate(cl):
                if n == 0: ax.text(mid, yy + 0.30, fmt(v[k]), ha="center", va="bottom", fontsize=vfs, color=cols[k], zorder=4)
                elif n == 1: ax.text(mid, yy - 0.30, fmt(v[k]), ha="center", va="top", fontsize=vfs, color=cols[k], zorder=4)
                else: ax.text(v[cl].max() + d_data * 0.7 * (n - 1), yy, fmt(v[k]), ha="left", va="center", fontsize=vfs, color=cols[k], zorder=4)


In [ ]:
import json
from fig_common import OUT, MANIFEST, VALUES
RESET = True
if RESET:
    for p in (MANIFEST, VALUES):
        if p.exists(): p.unlink()
PAGE = dict(width_mm=210, height_mm=297, margin_mm=15, letter_pt=10, caption_pt=9)
def S(slot, x, y, w, h, letter=None):
    d = dict(slot=slot, x=x, y=y, w=w, h=h)
    if letter: d["letter"] = letter
    return d
LAYOUT = {
 "Figure 1": dict(deck="main", number="1", caption_y=222, slots=[
     S("p01", 13.5, 15, 182.9, 64.01, "a"), S("p02", 13.5, 83, 182.9, 65.0, "b"), S("p03", 13.5, 152, 182.9, 65.0, "c")]),
 "Figure 2": dict(deck="main", number="2", caption_y=241, slots=[
     S("p01", 15, 15, 88, 44, "a"), S("p02", 107, 15, 88, 44, "b"), S("p03", 15, 62, 88, 44, "c"), S("p04", 107, 62, 88, 44, "d"),
     S("p05", 15, 109, 97, 100, "e"), S("p06", 114, 109, 81, 100, "f"), S("k01", 15, 211, 180, 17), S("k02", 15, 230, 180, 9)]),
 "Figure 3": dict(deck="main", number="3", caption_y=252, slots=[
     S("p01", 15, 15, 140, 118, "a"), S("k01", 158, 15, 37, 58), S("k02", 158, 76, 37, 57),
     S("p02", 15, 137, 92, 112, "b"), S("p03", 110, 137, 85, 72, "c"), S("k03", 110, 212, 85, 37)]),
 "Figure 4": dict(deck="main", number="4", caption_y=234, slots=[
     S("p01", 15, 15, 180, 125, "a"),
     S("p03", 15, 145, 58, 58, "b"), S("p04", 76, 145, 58, 58, "c"), S("p05", 137, 145, 58, 58, "d"), S("k01", 15, 206, 180, 24)]),
 "Figure S1": dict(deck="supp", number="1", caption_y=85, slots=[S("p01", 13.5, 15, 182.9, 64.01)]),
 "Figure S2": dict(deck="supp", number="2", caption_y=214, slots=[
     S("p01", 15, 15, 66, 56, "a"), S("p02", 85, 15, 66, 56, "b"), S("k01", 154, 18, 42, 46),
     S("p03", 15, 76, 88, 52, "c"), S("p04", 107, 76, 88, 60, "d"),
     S("p05", 15, 140, 88, 60, "e"), S("p06", 107, 140, 88, 36, "f"), S("k02", 15, 202, 180, 10)]),
 "Figure S2 (continued 2)": dict(deck="supp", number="2", caption_y=170, slots=[
     S("p07", 15, 15, 88, 30, "g"), S("p08", 107, 15, 88, 30, "h"), S("k03", 15, 47, 180, 8),
     S("p09", 15, 60, 88, 60, "i"), S("p10", 107, 60, 88, 52, "j"), S("k04", 107, 113, 88, 11),
     S("p11", 15, 126, 88, 34, "k")]),
 "Figure S3": dict(deck="supp", number="3", caption_y=155, slots=[
     S("p01", 15, 15, 100, 125, "a"), S("p02", 118, 15, 77, 125, "b"), S("k01", 15, 143, 180, 8)]),
 "Figure S4": dict(deck="supp", number="4", caption_y=145, slots=[
     S("p01", 15, 15, 97, 112, "a"), S("p02", 114, 15, 81, 112, "b"), S("k01", 15, 130, 180, 11)]),
 "Figure S5": dict(deck="supp", number="5", caption_y=170, slots=[
     S("p01", 15, 15, 110, 150, "a"), S("p02", 128, 15, 67, 60, "b"), S("k01", 128, 80, 67, 40)]),
 "Figure S7": dict(deck="supp", number="7", caption_y=204, slots=[
     S("p01", 15, 15, 88, 55, "a"), S("p02", 107, 15, 88, 55, "b"), S("p03", 15, 74, 180, 108, "c"), S("k01", 15, 185, 100, 16)]),
 "Figure S8": dict(deck="supp", number="8", caption_y=208, slots=[
     S("p01", 15, 15, 95, 125, "a"), S("p02", 114, 15, 81, 60, "b"), S("k01", 114, 79, 81, 58),
     S("p03", 15, 145, 58, 58, "c"), S("p04", 76, 145, 58, 58, "d"), S("p05", 137, 145, 58, 58, "e")]),
 "Figure S9": dict(deck="supp", number="9", caption_y=188, slots=[
     S("p01", 15, 15, 74, 150, "a"), S("p02", 91, 15, 52, 150, "b"), S("p03", 143, 15, 52, 150, "c"), S("k01", 15, 168, 180, 16)]),
 "Figure S10": dict(deck="supp", number="10", caption_y=158, slots=[
     S("p01", 15, 15, 88, 55, "a"), S("p02", 107, 15, 88, 55, "b"), S("p03", 15, 74, 88, 62, "c"), S("p04", 107, 74, 88, 62, "d"),
     S("k01", 15, 140, 180, 14)]),
 "Figure S12": dict(deck="supp", number="12", caption_y=96, slots=[
     S("p01", 15, 15, 100, 62, "a"), S("p02", 118, 15, 77, 62, "b"), S("k01", 15, 81, 180, 10)]),
}
EV = "Filled markers, P(effect > 0) ≥ 0.95 or ≤ 0.05; open markers, otherwise; pink, posterior mean above zero; blue, below."
CAPTIONS = {
 "Figure 1": dict(title="Analysis design.", panels=[
  "(a) Cellular model. Bulk tumour RNA-seq from {n_patients} CPTAC PDAC patients was deconvolved separately into immune coarse, immune fine and non-immune cell types; 8, 14 and 4 cell types were retained for modelling, and curated signatures were scored per patient and cell type (306, 496 and 112 features). Each analysis was fitted with a hierarchical model in which feature BMI effects are partially pooled towards their cell-type mean, with a Student-t patient × cell-type intercept, Student-t residuals with feature residual scales pooled within cell type, and adjustment for age and sex. Outputs are posterior means, 95% HDIs and P(effect > 0).",
  "(b) Paired bulk RNA–protein model: 122 matched gene sets scored in RNA and TMT proteomics from identical, fully observed genes, each fitted with separate RNA and protein BMI effects and a bivariate Student-t likelihood; estimates are mapped back to their source signature labels.",
  "(c) Protein-conditioned PTM model: each glycosylation or phosphorylation feature paired with its parent protein, fitted with separate protein and PTM BMI effects; the PTM effect conditional on protein is reported."]),
 "Figure 2": dict(title="Calibration of the cellular model in simulation and predictive checks on the observed data.", panels=[
  "(a–d) Simulation on the immune fine design ({sim_n_rep} datasets per scenario; Figure S1), comparing the Student-t intercept model, the orthogonal patient-effect model, OLS per feature (HC3 intervals) and, for cell types, a patient-level composite; summaries use fits that passed the convergence criteria. Bands, 95% Monte Carlo intervals; dashed lines, nominal levels. (a) Coverage of feature effects; the orthogonal model covered {pub_cov_min}–{pub_cov_max}. (b) Coverage of cell-type effects. (c) Proportion of null effects called (square-root scale; per effect, not per dataset); under the null, {fcr_null_corr_published_feature} for the orthogonal model and {t_null_fcr} for the Student-t intercept model, which made at least one call in {t_null_any_call} of null datasets. (d) Interval width; Student-t intercept intervals were {feature_width_ratio_min}–{feature_width_ratio_max} times the OLS width.",
  "(e, f) Predictive checks of the primary fits by cell type: residual pairs with |r| ≥ 0.9 (e) and features with studentized residual tails outside their predictive range (f); points, observed; bands, 95% interval of replicated data."]),
 "Figure 3": dict(title="BMI associations with cellular pathway activity under the Student-t intercept model.", panels=[
  "(a) Posterior mean BMI effects for the {n_heat_rows} signatures with the highest posterior probability of direction in any cell type; no feature-level 95% HDI excluded zero. Rings mark P(effect > 0) ≥ 0.95 or ≤ 0.05 ({n_ring} of {n_features} features); this directional threshold is less stringent than HDI exclusion. Column prefixes: C, immune coarse; F, immune fine; N, non-immune. Signature labels are coloured by analysis, or black when the signature is shared across analyses. All feature estimates are shown in Figure S5; signatures scored in more than one cell type are shown in Figure S6.",
  "(b) Cell-type mean BMI effects with 95% HDIs; numbers give P(effect > 0). {n_ct_positive} of {n_ct} posterior means were above zero.",
  "(c) Tumour-associated macrophage signatures with the highest P(effect > 0). " + EV]),
 "Figure 4": dict(title="Robustness of the cellular BMI estimates.", panels=[
  "(a) Left, cell-type mean BMI effects in the primary fit (points, 95% HDI) and the range of posterior means across {n_sens_fits} sensitivity fits per analysis (grey bar): BMI hierarchy priors halved and doubled, intercept-scale prior halved and doubled, all profiles without read exclusion, and a read-dependent residual scale on both profile sets. Correlations with the primary estimates, median HDI-width ratios and changes in cell-type P(effect > 0) for each fit are in Figure S7. The immune fine read-dependent fit on all profiles did not meet the convergence criterion (Figure S8).",
  "Right, obese-versus-normal-weight cell-type contrasts from the WHO BMI-group model fitted to all profiles, with 95% HDIs, for the same cell types; shaded blocks separate the three analyses. Other contrasts are in Figure S9.",
  "(b–d) Feature-level obese-versus-normal contrasts from the group model against the continuous BMI effect rescaled to the observed obese–normal BMI difference, both fitted to all profiles: immune coarse (b; r = {r_cat_coarse}), immune fine (c; r = {r_cat_fine}) and non-immune (d; r = {r_cat_nonimmune}). r, Pearson correlation of posterior means; dashed line, identity. Marker fill and colour follow the group model (y axis). " + EV]),
 "Figure S1": dict(title="Simulation design.", panels=[
  "Pathway scores were simulated on the design of the immune fine analysis: the 127 patients with their recorded BMI, age and sex, four cell types (TAM, CD8⁺ T cells, B cells, NK cells) with their 148 features, and the observed low-read exclusions. True BMI effects, residual scales and within-cell-type residual correlations were taken from the fitted model. Six scenarios (null; fitted effects; sparse, one cell type shifted by 0.12 SD per SD of BMI and 15% of signatures deviating; equicorrelated residuals, ρ = 0.6; 5% outlier profiles and 1% outlier observations; 80 of the 127 patients) and an effect-size grid (cell-type BMI effect 0.05, 0.10, 0.20 or 0.30 SD per SD of BMI, with the fitted between-feature spread) were each simulated 20 times, with Student-t patient effects (ν = 4). All methods were fitted to the same datasets. Bayesian fits used the same sampler and convergence criteria (R-hat < 1.01, bulk and tail ESS ≥ 400, no divergences, BFMI ≥ 0.3) and were refitted once with longer tuning on failure; summaries use fits that passed. Truth for a WHO-group contrast was the simulated BMI effect multiplied by the group’s mean BMI difference from normal weight in that dataset. The simulation covers one analysis design and does not by itself establish calibration for every cell type in the three analyses."]),
 "Figure S2": dict(title="Simulation comparisons of the cellular model.", panels=[
  "(a) Power (non-null feature effects called with the correct sign) and (b) coverage of 95% intervals across the effect-size grid: Student-t intercept model and orthogonal patient-effect model (95% HDI excluding zero), OLS per feature (95% CI) and OLS with Benjamini–Hochberg q < 0.05. At 0.10 SD, power was {grid_power_t_10}, {grid_power_orth_10}, {grid_power_ols_10} and {grid_power_bh_10}; the orthogonal model covered {grid_coverage_orth_20}–{grid_coverage_orth_05}, and Student-t intercept coverage decreased to {grid_coverage_t_30} at the largest simulated effect. Bands, ±1.96 Monte Carlo SE.",
  "(c) Power, (d) coverage and (e) root-mean-square error of feature effects (SD × 100) by scenario; values are printed inside the markers.",
  "(f) False discoveries: null datasets with any call, sparse datasets with any false call, and the mean false-discovery proportion among sparse-scenario calls (datasets without calls count as zero). Under the null the Student-t intercept model called {t_null_fcr} of effects and made at least one call in {t_null_any_call} of datasets; estimates are based on 20 datasets per scenario. Summaries use fits that passed the convergence criteria."]),
 "Figure S2 (continued 2)": dict(title="(continued).", append=True, panels=[
  "(g) Power and (h) share of null datasets with any call for the WHO group model and OLS by group with Benjamini–Hochberg, by contrast; with 18 obese patients, feature-level power of the group model reached at most {group_power_max}.",
  "(i) Coverage of obese-versus-normal contrasts by scenario; equicorrelated group-model summaries use {group_n_copula_used} of 20 datasets.",
  "(j) Power for cell-type mean effects: Student-t intercept model and the patient-level composite; Student-t intercept cell-type intervals were {ct_width_ratio_min}–{ct_width_ratio_max} times the composite width.",
  "(k) Fits passing the convergence criteria at the first attempt, counting every fit, with the share after at most one retry in brackets. In the equicorrelated scenario {group_conv_first_copula} of {group_n_copula} group-model fits passed at the first attempt and {group_conv_final_copula} after the retry; across all scenarios the one-factor model passed in {k1_converged_all} of {k1_attempted_all} fits at the first attempt and {k1_converged_final_all} after retries."]),
 "Figure S3": dict(title="Sampling diagnostics of the primary fits.", panels=[
  "(a) Maximum R-hat by parameter family in each analysis; dashed line, 1.01.",
  "(b) Minimum bulk (filled) and tail (open) effective sample size by parameter family; dashed line, 400. Four chains of 4,000 draws after 2,000 tuning iterations."]),
 "Figure S4": dict(title="Further posterior predictive checks of the primary fits.", panels=[
  "(a) Raw-outcome signature pairs with |r| ≥ 0.9 and (b) features whose residual standard deviation fell outside its predictive range, for each cell type; observed counts were above zero in {n_ct_disp_out} of {n_ct_disp} cell types. Points, observed; bands, 95% interval of replicated data."]),
 "Figure S5": dict(title="All feature-level estimates of the cellular model.", panels=[
  "(a) Posterior mean BMI effect for each of the {n_features} features, grouped by cell type. " + EV,
  "(b) Distribution of feature-level P(effect > 0) in each analysis. This is descriptive; without a null reference fitted to these data it is not evidence of enrichment."]),
 "Figure S6": dict(title="Signatures scored in more than one cell type.", panels=[
  "Posterior mean BMI effects for the {n_shared_signatures} signatures scored in at least two cell types; signatures specific to one cell type are shown in Figure S5 and listed in the supplementary data table. Rings mark P(effect > 0) ≥ 0.95 or ≤ 0.05; empty cells, signature not scored for that cell type. Column prefixes: C, immune coarse; F, immune fine; N, non-immune."]),
 "Figure S7": dict(title="Prior and profile sensitivity of the cellular estimates.", panels=[
  "(a) Pearson correlation of feature-level posterior means between each sensitivity fit and the primary fit.",
  "(b) Median ratio of 95% HDI width, sensitivity / primary.",
  "(c) Change in cell-type P(effect > 0), sensitivity − primary. Vertical lines separate the three analyses. The largest absolute change in a cell-type posterior mean across all fits was {max_sens_celltype_shift} SD per 5 kg/m². The immune fine read-dependent fit on all profiles did not meet the convergence criterion and is included as a sensitivity analysis (Figure S8)."]),
 "Figure S8": dict(title="Read-dependent residual scale.", panels=[
  "(a) Cell-type coefficients relating log residual scale to log allocated reads (standardized within cell type), fitted on profiles with ≥ 1,000 reads and on all profiles, with 95% HDIs. Negative values indicate noisier low-read profiles.",
  "(b) Upper-tail posterior predictive probability for studentized residual tails: reference models (open) and read-dependent residual scale (filled), on profiles with ≥ 1,000 reads (circles) and all profiles (squares). Reference: the primary fit for ≥ 1,000 reads and the all-profiles fit without a read term for all profiles. Dashed line, 0.025.",
  "(c–e) Feature-level BMI effects from the all-profiles reference fit (no read term) against the primary fit for immune coarse (c), immune fine (d) and non-immune (e) analyses; dashed line, identity; marker fill and colour follow the all-profiles fit (y axis). In the all-profiles read-dependent fit for immune fine, R-hat exceeded 1.01 only for feature intercepts and the global log-residual-scale location (maximum {rs_fine_all_rhat}); all reported quantities had R-hat ≤ 1.001. This fit did not meet the prespecified convergence criterion and is presented as a sensitivity analysis only; it is included in Figure 4a and Figure S7."]),
 "Figure S9": dict(title="WHO BMI-group contrasts by cell type.", panels=[
  "(a) Overweight versus normal weight, (b) obese versus normal weight and (c) obese versus overweight, from the group model fitted to all profiles, with 95% HDIs; cell types in the same order in all panels. {n_cat_celltype_excl} cell-type contrasts excluded zero. " + EV]),
 "Figure S10": dict(title="Checks of the WHO BMI-group model.", panels=[
  "(a) Upper-tail posterior predictive probability for studentized residual tails, group model on all profiles (circles) and on profiles with ≥ 1,000 reads (squares); dashed line, 0.025.",
  "(b) Pearson correlation between feature-level group contrasts and the continuous BMI effect, both fitted to all profiles, for each contrast and analysis.",
  "(c) Patient leverage in the group design against standardized BMI; dashed lines, 2r/n and |BMI z| = 2.",
  "(d) Patients most often responsible for the largest residual in a tail-flagged feature (all-profiles fits, all analyses)."]),
 "Figure S12": dict(title="Screen-level sign uncertainty across layers.", panels=[
  "(a) Proportion of features or gene sets with local false sign rate (lfsr) < 0.05 in each layer. The dashed line marks 10%, the proportion expected if P(effect > 0) were uniform; it is a nominal reference, not a calibrated null for these models, and features within a layer are not independent. PTM rows include the {n_ptm_pass} of {n_ptm_all} PTM features whose fits passed numerical checks; ‘Protein (PTM pairs)’ is the protein effect from each PTM feature’s paired fit, so a parent protein can appear more than once.",
  "(b) Distribution of P(effect > 0) for the PTM effect conditional on protein and for protein minus RNA. Dashed line, uniform density."]),
}
(OUT / "layout.json").write_text(json.dumps(dict(page=PAGE, figures=LAYOUT), indent=2, ensure_ascii=False))
(OUT / "captions.json").write_text(json.dumps(CAPTIONS, indent=2, ensure_ascii=False))
SLOT = {fig: {s["slot"]: s for s in d["slots"]} for fig, d in LAYOUT.items()}
from fig_common import set_value; set_value(n_patients=127)
print("layout written:", list(LAYOUT))


In [ ]:
import json
from fig_common import OUT
def S(slot, x, y, w, h, letter=None):
    d = dict(slot=slot, x=x, y=y, w=w, h=h)
    if letter: d["letter"] = letter
    return d
LMO = {
 "Figure 5": dict(deck="main", number="5", caption_y=247, slots=[
     S("p01", 15, 15, 80, 80, "a"), S("p02", 100, 15, 95, 62, "b"), S("k01", 100, 78, 95, 10), S("k03", 100, 88, 95, 10),
     S("p03", 15, 99, 180, 56, "c"), S("p04", 15, 158, 180, 78, "d"), S("k02", 15, 238, 180, 8)]),
 "Figure 6": dict(deck="main", number="6", caption_y=178, slots=[
     S("p01", 15, 15, 58, 58, "a"), S("p02", 76, 15, 58, 58, "b"), S("p03", 137, 15, 58, 58, "c"), S("k01", 15, 75, 180, 9),
     S("p04", 15, 88, 88, 74, "d"), S("p05", 107, 88, 88, 74, "e"), S("k02", 15, 164, 180, 9)]),
 "Figure 7": dict(deck="main", number="7", caption_y=140, slots=[
     S("p01", 15, 15, 56, 58, "a"), S("p02", 74, 15, 60, 58, "b"), S("p03", 137, 15, 58, 58, "c"), S("k01", 15, 76, 90, 28),
     S("p04", 110, 78, 85, 52, "d"), S("k02", 15, 108, 90, 12)]),
 "Figure S11": dict(deck="supp", number="11", caption_y=172, slots=[
     S("p01", 15, 15, 88, 55, "a"), S("p02", 107, 15, 88, 55, "b"), S("k01", 15, 72, 180, 8),
     S("p03", 15, 84, 88, 70, "c"), S("p04", 107, 84, 88, 70, "d"), S("k02", 15, 156, 88, 12), S("k03", 107, 156, 88, 12)]),
 "Figure S13": dict(deck="supp", number="13", caption_y=150, slots=[
     S("p01", 15, 15, 70, 55, "a"), S("p02", 88, 15, 107, 55, "b"), S("k01", 88, 71, 107, 9),
     S("p03", 15, 84, 80, 60, "c"), S("p04", 98, 84, 80, 60, "d"), S("k02", 180, 88, 15, 52)]),
 "Figure S14": dict(deck="supp", number="14", caption_y=257, slots=[
     S("p01", 15, 15, 72, 62, "a"), S("p02", 95, 15, 100, 56, "b"), S("k01", 15, 79, 72, 9),
     S("p03", 15, 94, 76, 74, "c"), S("k02", 15, 170, 76, 8),
     S("p04", 15, 180, 130, 66, "d"), S("k03", 15, 247, 130, 8)]),
 "Figure S15": dict(deck="supp", number="15", caption_y=200, slots=[
     S("p01", 15, 15, 140, 72, "a"), S("k01", 15, 89, 140, 8),
     S("p02", 15, 102, 88, 80, "b"), S("p03", 107, 102, 76, 80, "c"), S("k02", 185, 108, 12, 64), S("k03", 15, 184, 180, 8)]),
}
for k, metric in [(16, "coverage"), (17, "bias"), (18, "rmse")]:
    LMO[f"Figure S{k}"] = dict(deck="supp", number=str(k), caption_y=210, slots=[
        S("p01", 15, 15, 88, 58, "a"), S("p02", 107, 15, 88, 58, "b"), S("p03", 15, 76, 88, 58, "c"),
        S("p04", 107, 76, 88, 58, "d"), S("p05", 15, 137, 88, 58, "e"), S("p06", 107, 137, 88, 58, "f"), S("k01", 15, 197, 180, 9)])
LMO.update({
 "Figure S19": dict(deck="supp", number="19", caption_y=160, slots=[
     S("p01", 15, 15, 58, 42, "a"), S("p02", 76, 15, 58, 42), S("p03", 137, 15, 58, 42),
     S("p04", 15, 60, 58, 42), S("p05", 76, 60, 58, 42), S("p06", 137, 60, 58, 42), S("k01", 15, 104, 180, 9),
     S("p07", 15, 118, 100, 34, "b"), S("k02", 118, 122, 77, 18)]),
 "Figure S20": dict(deck="supp", number="20", caption_y=170, slots=[
     S("p01", 15, 15, 88, 70, "a"), S("p02", 107, 15, 88, 70), S("k01", 15, 87, 180, 9),
     S("p03", 15, 100, 100, 62, "b"), S("k02", 118, 110, 77, 22)]),
 "Figure S21": dict(deck="supp", number="21", caption_y=262, slots=[
     S("p01", 15, 15, 88, 58, "a"), S("p02", 107, 15, 88, 34, "b"), S("p03", 15, 78, 88, 170, "c"), S("p04", 107, 54, 88, 160, "d"),
     S("k01", 107, 220, 88, 24)]),
 "Figure S22": dict(deck="supp", number="22", caption_y=168, slots=[
     S("p01", 15, 15, 88, 145, "a"), S("k01", 106, 20, 16, 70), S("k02", 126, 30, 69, 22),
     S("p02", 108, 100, 87, 46, "b"), S("k03", 108, 148, 87, 8)]),
})
for k, cmp_slots in [(23, ["primary → fixed_plex", "primary → stage_subset_base", "stage_subset_base → stage_adjusted", "primary → wes_subset_base"]),
                     (24, ["wes_subset_base → wes_adjusted", "primary → mutation_adjusted", "primary → exclude_recorded_weight_loss",
                           "primary → exclude_adenosquamous"])]:
    LMO[f"Figure S{k}"] = dict(deck="supp", number=str(k), caption_y=250, comparisons=cmp_slots, slots=[
        S("p01", 15, 15, 88, 110, "a"), S("p02", 107, 15, 88, 110, "b"), S("p03", 15, 128, 88, 110, "c"),
        S("p04", 107, 128, 88, 110, "d"), S("k01", 15, 240, 180, 8)])
LMO.update({
 "Figure S25": dict(deck="supp", number="25", caption_y=150, slots=[
     S("p01", 15, 15, 88, 50, "a"), S("p02", 107, 15, 88, 120, "b"), S("k01", 15, 70, 88, 16)]),
 "Figure S26": dict(deck="supp", number="26", caption_y=250, slots=[
     S("p01", 15, 15, 88, 80, "a"), S("p02", 107, 15, 88, 45, "b"), S("k01", 107, 62, 88, 16),
     S("p03", 15, 100, 130, 132, "c"), S("k02", 15, 234, 130, 9)]),
})
HDI = "Filled markers, 95% HDI excludes zero; open markers, includes zero; pink, posterior mean above zero; blue, below."
PU = "SD per 5 kg/m² within plex"
CMO = {
 "Figure 5": dict(title="RNA–protein agreement differs from a paired BMI-slope contrast.", panels=[
  "(a) Posterior mean RNA and protein BMI slopes for the {n_rp} matched gene sets (Spearman’s ρ = {rp_rho}), with the marginal distributions of RNA (top) and protein (right) slopes.",
  "(b) Correlations of BMI slopes from the linear benchmark, with within-plex patient-bootstrap 95% CIs: cell-type RNA scores against bulk RNA and against bulk protein within each cellular analysis, and bulk RNA against bulk protein. Linear benchmark, not cellular-model posteriors.",
  "(c) Protein-minus-RNA slope contrasts (left) and residual RNA–protein correlations (right) for the ten sensitivity targets, with 95% HDIs.",
  "(d) The 16 gene sets with the largest absolute RNA or protein slope: RNA slope, protein slope and protein-minus-RNA contrast, with 95% HDIs. Labels give the first source signature (C, immune coarse; F, immune fine; N, non-immune); labels record gene-list origin, not protein-producing cells. Units, SD per 5 kg/m², standardized separately per layer. In c and d, filled markers, 95% HDI excludes zero; open markers, includes zero; pink, posterior mean above zero; blue, below."]),
 "Figure 6": dict(title="Parent-protein conditioning changes which PTM features stand out.", panels=[
  "(a, b) Marginal versus protein-conditioned posterior mean BMI slopes for all numerically adequate phosphosites (a; n = {n_phospho_adequate}) and glycoforms (b; n = {n_glyco_adequate}); dashed line, identity.",
  "(c) 95% HDI classification before and after conditioning. Phosphosites: {phospho_marg_cand} marginal candidates, {phospho_retained} retained and {phospho_added} added, giving {phospho_cand}. Glycoforms: {glyco_marg_cand} marginal candidates, {glyco_retained} retained and {glyco_added} added, giving {glyco_cand}. No resolved interval changed sign.",
  "(d, e) Marginal (grey squares) and protein-conditioned slopes with 95% HDIs for all {phospho_cand} phosphosite candidates (d) and the 20 of {glyco_cand} glycoform candidates with the largest absolute protein-conditioned slope (e). Units, " + PU + ". Glycoform labels give gene and glycan composition; full identities are in the PDAC Obesity Atlas and all candidates are in Figure S20. " + HDI]),
 "Figure 7": dict(title="PTM interval exclusions, sign uncertainty and patient influence.", panels=[
  "(a) Observed protein-conditioned 95% HDI exclusions among numerically adequate phosphosites and glycoforms, against the number expected if every slope were null at 5% (hatched; {phospho_exp_excl} and {glyco_exp_excl}).",
  "(b) Local false sign rate (lfsr) by feature percentile within assay; dashed and dotted lines, lfsr = 0.05 and 0.01.",
  "(c) Mean lfsr among the top-ranked features, by assay and for all adequate features. Panels a–c are descriptive, with nominal references; they do not provide false-discovery-rate control.",
  "(d) Protein-conditioned slopes (SD per 5 kg/m² within plex) with and without the flagged influential patient for the three candidates with a deletion refit ({deletion_genes}); points, posterior means; lines, 95% HDIs."]),
 "Figure S11": dict(title="Extra checks of the cellular model.", panels=[
  "(a) Proportion of feature posterior means above zero in the primary fit (real signatures) and in a fit of the same model in which every signature was replaced by a size-matched random gene set from the same cell type, scored identically (one draw per signature).",
  "(b) Features with local false sign rate < 0.05 in the same two fits. Random gene sets reached {rand_lfsr_coarse}, {rand_lfsr_fine} and {rand_lfsr_non} features in the immune coarse, immune fine and non-immune analyses, against {real_lfsr_coarse}, {real_lfsr_fine} and {real_lfsr_non} for the real signatures. The immune fine random-gene fit reached minimum ESS {rand_fine_ess} (criterion 400; R-hat {rand_fine_rhat}, no divergences) and is shown as a sensitivity analysis.",
  "(c) Cell-type mean BMI effects (SD per 5 kg/m²) with real signatures against random gene sets; dashed line, identity.",
  "(d) BMI slopes of the five TAM signatures with lfsr < 0.05 in the primary fit, from ordinary least squares on the primary profiles (points) and the range after leaving out each gene in turn (bars); the gene whose removal changed the slope most is named. OLS slopes are on the score scale of the decomposition and are not the posterior estimates."]),
 "Figure S13": dict(title="RNA–protein coverage, direction agreement and observed score pairs.", panels=[
  "(a) Cellular features eligible for the RNA–protein model (at least five complete matched genes and 50% coverage), by analysis.",
  "(b) Sign agreement of linear-benchmark BMI slopes, with within-plex patient-bootstrap 95% CIs; ticks, agreement expected from the marginal proportions of positive slopes alone. Duplicate features do not represent independent gene sets.",
  "(c, d) Matched RNA and protein scores for tumour-epithelial-labelled ECM Remodeling (c) and exhausted-CD8-labelled Metabolic Imbalance (d), coloured by BMI; each point is one patient. Trends are unadjusted; model-based contrasts and residual correlations are in Figure 5."]),
 "Figure S14": dict(title="RNA–protein model fit, matched-gene scoring and patient influence.", panels=[
  "(a) R-hat and minimum ESS across {n_rp_fits} primary, sensitivity and patient-deletion fits; maximum R-hat {rp_max_rhat}, {rp_div} divergences.",
  "(b) Posterior predictive upper-tail probabilities for the {n_rp} primary fits by check (five checks per gene set); {rp_flags} of {rp_checks} checks were flagged.",
  "(c) Matched-gene versus full-source RNA slopes from the linear benchmark ({n_cov_sets} gene sets; Spearman’s ρ = {cov_rho}); filled, bootstrap interval of the difference excludes zero.",
  "(d) Patient-deletion refits for the four flagged gene sets: posterior means and 95% HDIs after deletion, with ticks at the original means. The largest RNA or protein slope change was {del_max_slope} SD per 5 kg/m² and the largest residual-correlation change {del_max_corr}."]),
 "Figure S15": dict(title="RNA and protein slopes of the ten sensitivity targets.", panels=[
  "(a) RNA (blue) and protein (pink) BMI posterior means and 95% HDIs for ten targets fixed without using bulk BMI estimates, in SD per 5 kg/m² within plex, adjusted for age and sex; filled, HDI excludes zero.",
  "(b, c) Protein (b) and RNA (c) slopes under fixed-plex, purity-subset, purity-adjusted, stage-subset, stage-adjusted and RNA-fraction models; RNA-fraction models are specific to each target. Colour, posterior mean; white circles, HDI excludes zero; off-white, not modelled. Patient-deletion refits are in Figure S14d. Adjusted estimates are compared with unadjusted fits in the same patients. Cell-type labels record gene-list origin, not the protein-producing cells."]),
 "Figure S16": dict(title="Interval coverage of the protein–PTM model in simulation.", panels=[
  "(a–f) Coverage of nominal 95% HDIs for six model quantities across five scenarios (null, protein-only, conditional signal, heavy tails and prior-predictive simulation-based calibration, SBC), for all completed (blue) and numerically adequate (pink) replicates. Bars, 95% Wilson intervals for the coverage proportion; dashed line, 0.95. Simulations used the covariate and missing-data templates of one phosphosite and one glycoform and do not establish false-discovery control across the screen."]),
 "Figure S17": dict(title="Bias of the protein–PTM model in simulation.", panels=[
  "(a–f) Mean bias with approximate ±1.96 Monte Carlo SE for the six quantities and five scenarios of Figure S16, all completed (blue) and numerically adequate (pink) replicates; dashed line, zero. Units differ across quantities."]),
 "Figure S18": dict(title="Estimation error of the protein–PTM model in simulation.", panels=[
  "(a–f) Root-mean-square error for the six quantities and five scenarios of Figure S16, all completed (blue) and numerically adequate (pink) replicates. Units differ across quantities."]),
 "Figure S19": dict(title="Rank calibration and completion of the protein–PTM simulations.", panels=[
  "(a) Simulation-based calibration rank histograms for the six quantities, with the expected count (dashed) and pointwise 95% binomial reference limits (grey), which are not simultaneous bands or a formal calibration test.",
  "(b) {sim_completed} simulated datasets, of which {sim_adequate} fits met the numerical adequacy criteria."]),
 "Figure S20": dict(title="Protein-conditioned PTM slopes and candidate diagnostics.", panels=[
  "(a) Protein-conditioned BMI slopes for all numerically adequate primary fits: {n_phospho_adequate} phosphosites and {n_glyco_adequate} glycoforms, ranked by estimate, with 95% HDIs (grey); pink and blue, HDI excludes zero.",
  "(b) Candidate diagnostics: maximum R-hat and minimum ESS for the {n_candidates} candidates; filled, covariate-sensitivity fits available ({n_cand_sens} candidates: the 24 post hoc selected features in Figure S22 and three ASAH1 glycoforms)."]),
 "Figure S21": dict(title="Glycoforms of the same protein can have opposite protein-conditioned slopes.", panels=[
  "(a–d) All numerically adequate glycoforms of CTSD (a), HLA-DRA (b) and LAMC1 (c) and of the displayed POSTN peptide (d), ranked by estimate and chosen after inspecting the primary results to illustrate heterogeneity. Protein-conditioned BMI posterior means (" + PU + ") with 95% HDIs. Glycoforms of one protein are correlated and are not independent replications; no gene-level effect, glycan conversion or occupancy is estimated. " + HDI]),
 "Figure S22": dict(title="Sensitivity of post hoc selected PTM candidates.", panels=[
  "(a) Protein-conditioned BMI posterior means for the 24 candidates from CTSD, GSDMD, HLA-DRA, LAMC1 and POSTN under the primary fit and eight sensitivity variants, in " + PU + "; circles, 95% HDI excludes zero; ×, numerical failure. E01–E24 map to feature identities in the PDAC Obesity Atlas.",
  "(b) Interval classifications retained or changed in each comparison, counting only pairs in which both fits were numerically adequate. Features were selected after inspecting the primary results; variant-level estimates are in Figures S23 and S24."]),
 "Figure S23": dict(title="Post hoc PTM candidates under fixed-plex, stage and purity-subset variants.", panels=[
  "Posterior means and 95% HDIs for the 24 features of Figure S22, reference estimate in blue and sensitivity estimate in pink. (a) Fixed plex versus primary. (b) Stage subset versus primary. (c) Stage adjustment versus the matched stage subset. (d) Purity subset versus primary. Open markers, HDI includes zero; grey crosses, failed numerical checks."]),
 "Figure S24": dict(title="Post hoc PTM candidates under purity adjustment, mutation adjustment and patient exclusions.", panels=[
  "Posterior means and 95% HDIs for the 24 features of Figure S22, reference estimate in blue and sensitivity estimate in pink. (a) Purity adjustment versus the matched purity subset. (b) Mutation adjustment versus primary. (c) Exclusion of patients with recorded weight loss versus primary. (d) Exclusion of adenosquamous tumours versus primary. Open markers, HDI includes zero; grey crosses, failed numerical checks."]),
 "Figure S25": dict(title="Feature-level estimates for STAT3 phosphosites and ASAH1 glycoforms.", panels=[
  "(a) STAT3 Y705 and S727: total STAT3 protein, marginal phosphosite and protein-conditioned phosphosite slopes.",
  "(b) All {n_asah1} ASAH1 protein-conditioned glycoform slopes; A01–A{n_asah1} map to peptide-plus-glycan identities in the PDAC Obesity Atlas, and identical compositions on different peptides remain distinct. Units, " + PU + ". " + HDI]),
 "Figure S26": dict(title="Sensitivity analyses and bootstrap benchmark for STAT3 and ASAH1 features.", panels=[
  "(a) STAT3 Y705 and S727 continuous sensitivity fits (reference, open grey; sensitivity, coloured).",
  "(b) Categorical contrasts (overweight and obese versus normal weight) for the same sites, in SD.",
  "(c) Fixed-plex linear bootstrap benchmarks for the STAT3 sites and the ASAH1 glycoforms; bootstrap intervals are confidence intervals, not Bayesian refits. {stat3_note}"]),
}
L = json.loads((OUT / "layout.json").read_text()); C = json.loads((OUT / "captions.json").read_text())
L["figures"].update({k: {kk: vv for kk, vv in v.items() if kk != "comparisons"} for k, v in LMO.items()}); C.update(CMO)
(OUT / "layout.json").write_text(json.dumps(L, indent=2, ensure_ascii=False)); (OUT / "captions.json").write_text(json.dumps(C, indent=2, ensure_ascii=False))
SLOT.update({fig: {s["slot"]: s for s in d["slots"]} for fig, d in LMO.items()})
COMPARISONS = {k: v["comparisons"] for k, v in LMO.items() if "comparisons" in v}
print("multiomics layout added:", list(LMO))


In [ ]:
from fig_common import *
from matplotlib.patches import FancyBboxPatch, Rectangle, Ellipse
FS = 1.15
N_PATIENTS = 127
DECONVOLUTIONS = [("Immune coarse", 8, 306), ("Immune fine", 14, 496), ("Non-immune", 4, 112)]
N_FEATURES = sum(d[2] for d in DECONVOLUTIONS)
MIN_READS = "1,000"
N_PLEXES = 24
N_COVERED_LABELS, N_UNIQUE_SCORES = 398, 122
N_PTM = 1648
BLUE, LIGHT_BLUE, MAGENTA, LIGHT_MAGENTA = "#2C5AA0", "#DCE5F3", "#D81B60", "#FBE3EC"
MID_BLUE, MID_PINK = "#7E9CCF", "#EC8DB0"
INK, MID_GREY, PALE, LINE = "#1F1F1F", "#6F7378", "#F4F4F6", "#B9BCC2"
DIVERGING = mpl.colors.LinearSegmentedColormap.from_list("d", [BLUE, "white", MAGENTA])

W, COL_H = 7.2, 2.30
H = TOP = BODY_TOP = BOTTOM = None
COLS = [(0.10, 1.12), (1.38, 1.34), (2.88, 1.38), (4.42, 1.46), (6.04, 1.06)]

def new_canvas(height):
    """Canvas of the given height; the stage columns sit at its top."""
    global H, TOP, BODY_TOP, BOTTOM
    H = height
    TOP = H - 0.13
    BODY_TOP = TOP - 0.24
    BOTTOM = TOP - COL_H
    fig = plt.figure(figsize=(W, H))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.axis("off")
    return fig, ax

def box(ax, x, y, w, h, face="white", edge=MID_GREY, lw=0.8, r=0.05, z=2, ls="-"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                                facecolor=face, edgecolor=edge, linewidth=lw, linestyle=ls, zorder=z))

def text(ax, x, y, s, size=6.8, color=INK, weight="normal", ha="center", style="normal"):
    ax.text(x, y, s, ha=ha, va="center", fontsize=size * FS, color=color, fontweight=weight, style=style,
            zorder=6, linespacing=1.3)

def stage(ax, i, title, color, face, edge=None):
    x, w = COLS[i]
    box(ax, x, BODY_TOP, w, 0.24, face=color, edge=color, z=3)
    text(ax, x + w / 2, BODY_TOP + 0.12, title, 7.6, "white", "bold")
    box(ax, x, BOTTOM, w, COL_H - 0.24, face=face, edge=edge or face, lw=0.9 if edge else 0.8)
    return x, w

def connectors(ax, y):
    for (xa, wa), (xb, _) in zip(COLS[:-1], COLS[1:]):
        ax.plot([xa + wa + 0.02, xb - 0.02], [y, y], color=LINE, lw=1.4, solid_capstyle="butt", zorder=1)

RNG = np.random.default_rng(7)

def g_waffle(ax, x, y, w, h, n=127, cols=16):
    """One square per patient."""
    rows = int(np.ceil(n / cols))
    cell = min(w / cols, h / rows)
    x0 = x + (w - cell * cols) / 2
    for k in range(n):
        r, c = divmod(k, cols)
        ax.add_patch(Rectangle((x0 + c * cell + cell * 0.12, y + h - (r + 1) * cell + cell * 0.12),
                               cell * 0.76, cell * 0.76, facecolor=BLUE if (k // cols) % 2 == 0 else MID_BLUE,
                               edgecolor="none", zorder=5))

def g_stacked(ax, x, y, w, h, n=7):
    """Per-patient cell-type fractions as stacked bars."""
    colours = [BLUE, MID_BLUE, MID_PINK, MAGENTA]
    bw = w / n
    for k in range(n):
        frac = RNG.dirichlet([3, 2, 2, 1.5])
        base = y
        for f, c in zip(frac, colours):
            ax.add_patch(Rectangle((x + k * bw + bw * 0.12, base), bw * 0.76, f * h, facecolor=c,
                                   edgecolor="white", lw=0.3, zorder=5))
            base += f * h

def g_heatmap(ax, x, y, w, h, rows=4, cols=9, seed=3):
    vals = np.random.default_rng(seed).normal(size=(rows, cols))
    for r in range(rows):
        for c in range(cols):
            ax.add_patch(Rectangle((x + c * w / cols, y + r * h / rows), w / cols, h / rows,
                                   facecolor=DIVERGING(0.5 + vals[r, c] / 5), edgecolor="white", lw=0.4, zorder=5))

def g_shrinkage(ax, x, y, w, h):
    """Partial pooling: each feature estimated alone (left) is pulled towards its cell-type slope (right)."""
    xl, xr = x + 0.12 * w, x + 0.88 * w
    for xx in (xl, xr):
        ax.plot([xx, xx], [y, y + h], color=INK, lw=0.6, zorder=4)
    groups = [([0.95, 0.78, 0.60, 0.50], BLUE), ([0.42, 0.25, 0.12, 0.02], MAGENTA)]
    for raw, colour in groups:
        mean = float(np.mean(raw))
        ax.plot([xr - 0.07, xr + 0.07], [y + mean * h] * 2, color=colour, lw=2.0, zorder=5, solid_capstyle="butt")
        for r in raw:
            pooled = mean + 0.35 * (r - mean)
            ax.plot([xl, xr], [y + r * h, y + pooled * h], color=colour, lw=0.8, alpha=0.75, zorder=4)
            ax.scatter([xl], [y + r * h], s=10, facecolor="white", edgecolor=colour, lw=0.8, zorder=6)
            ax.scatter([xr], [y + pooled * h], s=10, facecolor=colour, edgecolor="white", lw=0.3, zorder=6)
    text(ax, xl, y - 0.08, "alone", 5.6, MID_GREY)
    text(ax, xr, y - 0.08, "pooled", 5.6, MID_GREY)

def g_forest(ax, x, y, w, h):
    zx = x + w / 2
    ax.plot([zx, zx], [y, y + h], color=INK, lw=0.7, zorder=4)
    rows = [(0.34, 0.16, True), (0.14, 0.20, True), (-0.08, 0.18, True), (-0.30, 0.15, True)]
    for i, (m, half, filled) in enumerate(rows):
        yy = y + h - (i + 0.5) * h / len(rows)
        colour = MAGENTA if m > 0 else BLUE
        ax.plot([zx + (m - half) * w, zx + (m + half) * w], [yy, yy], color=colour, lw=1.3, zorder=5,
                solid_capstyle="round")
        ax.scatter([zx + m * w], [yy], s=12, facecolor=colour if filled else "white", edgecolor=colour, lw=0.9, zorder=6)

def g_scatter_ellipse(ax, x, y, w, h):
    """Paired RNA and protein scores with a correlation ellipse (drawn in the box's own proportions)."""
    th = np.linspace(0, 2 * np.pi, 200)
    u, v = 0.46 * np.cos(th), 0.20 * np.sin(th)
    c, s_ = np.cos(np.pi / 4), np.sin(np.pi / 4)
    ex, ey = u * c - v * s_, u * s_ + v * c
    ax.fill(x + (ex + 0.5) * w, y + (ey + 0.5) * h, facecolor=LIGHT_MAGENTA, edgecolor=MAGENTA, lw=0.8, zorder=4)
    pts = RNG.multivariate_normal([0, 0], [[1, 0.75], [0.75, 1]], 24) / 5.5
    pts = pts.clip(-0.42, 0.42)
    ax.scatter(x + (pts[:, 0] + 0.5) * w, y + (pts[:, 1] + 0.5) * h, s=5, color=BLUE, zorder=5, lw=0)
    ax.plot([x, x + w], [y, y], color=INK, lw=0.6, zorder=5)
    ax.plot([x, x], [y, y + h], color=INK, lw=0.6, zorder=5)

def g_paired_intervals(ax, x, y, w, h):
    """For each signature: RNA (blue) and protein (pink) slope, each with its interval, offset vertically."""
    zx = x + w * 0.42
    ax.plot([zx, zx], [y, y + h], color=INK, lw=0.7, zorder=4)
    rows = [((0.30, 0.14), (0.18, 0.16)), ((0.20, 0.12), (-0.02, 0.15)), ((-0.14, 0.12), (-0.26, 0.13))]
    for i, (rna, prot) in enumerate(rows):
        yc = y + h - (i + 0.5) * h / len(rows)
        for (m, half), colour, off in ((rna, BLUE, 0.045), (prot, MAGENTA, -0.045)):
            yy = yc + off
            ax.plot([zx + (m - half) * w, zx + (m + half) * w], [yy, yy], color=colour, lw=1.2, zorder=5,
                    solid_capstyle="round")
            ax.scatter([zx + m * w], [yy], s=10, color=colour, lw=0, zorder=6)

# ---- tile icons for the checks row
def i_simulation(ax, x, y, w, h):
    t = np.linspace(0, 1, 60)
    for curve, colour in ((1 - np.exp(-4.5 * t), MAGENTA), (1 - np.exp(-2.2 * t), BLUE)):
        ax.plot(x + t * w, y + curve * h * 0.9, color=colour, lw=1.1, zorder=5)
    ax.plot([x, x + w], [y, y], color=INK, lw=0.5, zorder=5)

def i_lowread(ax, x, y, w, h):
    heights = [0.15, 0.25, 0.55, 0.7, 0.85, 0.95, 0.8]
    bw = w / len(heights)
    for k, v in enumerate(heights):
        colour = LINE if v < 0.3 else MID_BLUE
        ax.add_patch(Rectangle((x + k * bw + bw * 0.12, y), bw * 0.76, v * h, color=colour, zorder=5))
    ax.plot([x, x + w], [y + 0.3 * h, y + 0.3 * h], color=MAGENTA, lw=0.9, ls=(0, (2, 1.5)), zorder=6)

def i_ppc(ax, x, y, w, h):
    heights = [0.2, 0.45, 0.8, 1.0, 0.75, 0.4, 0.15]
    bw = w * 0.8 / len(heights)
    for k, v in enumerate(heights):
        ax.add_patch(Rectangle((x + k * bw, y), bw * 0.9, v * h * 0.9, color=LIGHT_BLUE, zorder=5))
    ax.plot([x + w * 0.88, x + w * 0.88], [y, y + h * 0.9], color=MAGENTA, lw=1.3, zorder=6)

def i_groups(ax, x, y, w, h):
    for k, (v, colour) in enumerate(((0.45, MID_BLUE), (0.65, MID_PINK), (0.9, MAGENTA))):
        ax.add_patch(Rectangle((x + w * (0.1 + 0.3 * k), y), w * 0.22, v * h, color=colour, zorder=5))

def i_agreement(ax, x, y, w, h):
    t = np.linspace(0.1, 0.9, 14)
    ax.plot([x, x + w], [y, y + h], color=LINE, lw=0.8, zorder=4)
    ax.scatter(x + t * w, y + (t + RNG.normal(0, 0.07, t.size)).clip(0, 1) * h, s=5, color=MAGENTA, lw=0, zorder=5)

def i_readdepth(ax, x, y, w, h):
    t = np.linspace(0.05, 1, 22)
    spread = 0.45 * (1 - t) + 0.06
    ax.scatter(x + t * w, y + (0.5 + RNG.normal(0, 1, t.size) * spread / 2).clip(0, 1) * h, s=5, color=BLUE,
               lw=0, zorder=5)
    ax.plot([x, x + w], [y + h / 2, y + h / 2], color=LINE, lw=0.8, zorder=4)

def checks_row(ax, title, items, icons=True):
    cy, ch = 0.10, 1.10 if icons else 0.90
    box(ax, 0.10, cy, W - 0.20, ch, face="white", edge=LINE, lw=0.8, r=0.06)
    text(ax, 0.24, cy + ch - 0.15, title, 7.4, weight="bold", ha="left")
    cw = (W - 0.44) / len(items)
    small = len(items) >= 7
    for i, item in enumerate(items):
        name, sub = item[0], item[1]
        xx = 0.22 + i * cw
        tile_h = 0.78 if icons else 0.52
        box(ax, xx + 0.03, cy + 0.08, cw - 0.10, tile_h, face=PALE, edge=PALE)
        mid = xx + cw / 2 - 0.02
        if icons and len(item) > 2:
            iw = min(0.52, cw - 0.26)
            item[2](ax, mid - iw / 2, cy + 0.54, iw, 0.24)
            text(ax, mid, cy + 0.36, name, 6.2 if small else 6.4, weight="bold")
            text(ax, mid, cy + 0.19, sub, 5.5 if small else 5.7, MID_GREY)
        else:
            text(ax, mid, cy + 0.41, name, 6.2 if small else 6.5, weight="bold")
            text(ax, mid, cy + 0.23, sub, 5.6 if small else 5.8, MID_GREY)

fig, ax = new_canvas(2.52)
connectors(ax, BOTTOM + (COL_H - 0.24) / 2)

x, w = stage(ax, 0, "Cohort", BLUE, LIGHT_BLUE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "CPTAC PDAC", 7.2, weight="bold")
g_waffle(ax, x + 0.12, BODY_TOP - 0.86, w - 0.24, 0.50)
text(ax, cx, BODY_TOP - 1.02, f"{N_PATIENTS} patients", 7.0)
text(ax, cx, BODY_TOP - 1.40, "Bulk tumour\nRNA-seq", 6.6)
text(ax, cx, BOTTOM + 0.26, "Recorded BMI,\nage and sex", 6.4, MID_GREY)

x, w = stage(ax, 1, "Deconvolution", BLUE, "white")
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.15, "BayesPrism, run separately", 6.1, MID_GREY)
g_stacked(ax, x + 0.22, BODY_TOP - 0.58, w - 0.44, 0.34)
bh, gap = 0.40, 0.06
for i, (label, n_ct, n_ft) in enumerate(DECONVOLUTIONS):
    yy = BODY_TOP - 0.68 - (i + 1) * bh - i * gap
    box(ax, x + 0.05, yy, w - 0.10, bh, face="white", edge=BLUE, lw=0.9)
    text(ax, cx, yy + bh * 0.66, label, 6.9, BLUE, "bold")
    text(ax, cx, yy + bh * 0.27, f"{n_ct} cell types modelled", 6.0)

x, w = stage(ax, 2, "Pathway features", BLUE, PALE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "Curated signatures", 7.0, weight="bold")
text(ax, cx, BODY_TOP - 0.38, "unsigned gene lists per cell type", 5.9, MID_GREY)
g_heatmap(ax, x + 0.28, BODY_TOP - 0.94, w - 0.56, 0.38)
text(ax, cx, BODY_TOP - 1.05, "patients × features", 5.8, MID_GREY)
text(ax, cx, BODY_TOP - 1.45, "Within-compartment log-CPM,\ngene z-scores capped at ±5,\naveraged and standardized", 6.0)
text(ax, cx, BOTTOM + 0.14, " / ".join(str(d[2]) for d in DECONVOLUTIONS) + " features", 6.6, BLUE, "bold")

x, w = stage(ax, 3, "Hierarchical model", MAGENTA, LIGHT_MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.18, "One fit per deconvolution", 6.8, weight="bold")
g_shrinkage(ax, x + 0.20, BODY_TOP - 0.90, w - 0.40, 0.58)
text(ax, cx, BODY_TOP - 1.18, "feature slopes pulled towards\ntheir cell-type slope", 6.0, MAGENTA)
text(ax, cx, BODY_TOP - 1.56, "Student-t residuals; feature\nscales pooled in cell type", 6.0)
text(ax, cx, BOTTOM + 0.20, "Patient × cell-type intercept,\nStudent-t; age and sex adjusted", 6.0)

x, w = stage(ax, 4, "Outputs", MAGENTA, "white", edge=MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "BMI slopes", 7.0, weight="bold")
text(ax, cx, BODY_TOP - 0.38, "per 5 kg/m²", 6.3, MID_GREY)
g_forest(ax, x + 0.14, BODY_TOP - 1.18, w - 0.28, 0.66)
text(ax, cx, BODY_TOP - 1.40, "feature and cell-type", 6.0)
text(ax, cx, BOTTOM + 0.22, "Posterior mean,\n95% HDI, P(effect > 0)", 6.1)

save(fig, "F1", "p01", "cellular_pipeline_schematic", "schematic")

fig, ax = new_canvas(COL_H + 0.26)
connectors(ax, BOTTOM + (COL_H - 0.24) / 2)

x, w = stage(ax, 0, "Same patients", BLUE, LIGHT_BLUE)
cx = x + w / 2
g_waffle(ax, x + 0.12, BODY_TOP - 0.66, w - 0.24, 0.50)
text(ax, cx, BODY_TOP - 0.82, f"{N_PATIENTS} patients", 7.0, weight="bold")
text(ax, cx, BODY_TOP - 1.14, "Bulk tumour\nRNA-seq", 6.4)
text(ax, cx, BODY_TOP - 1.52, f"TMT proteomics,\n{N_PLEXES} plexes", 6.4)
text(ax, cx, BOTTOM + 0.14, "BMI, age and sex", 6.2, MID_GREY)

x, w = stage(ax, 1, "Original signatures", BLUE, PALE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, f"{N_FEATURES} cell-type features", 6.8, weight="bold")
g_heatmap(ax, x + 0.24, BODY_TOP - 0.74, w - 0.48, 0.36, seed=11)
text(ax, cx, BODY_TOP - 1.05, f"{N_COVERED_LABELS} with enough\nprotein coverage", 6.3)
text(ax, cx, BODY_TOP - 1.50, f"{N_UNIQUE_SCORES} unique matched-\ngene sets, each fitted\nonce and linked back\nto its labels", 6.1)

x, w = stage(ax, 2, "Paired bulk scores", BLUE, PALE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "Same genes in both layers", 6.6, weight="bold")
hw = (w - 0.34) / 2
g_heatmap(ax, x + 0.12, BODY_TOP - 0.76, hw, 0.36, cols=5, seed=5)
g_heatmap(ax, x + 0.22 + hw, BODY_TOP - 0.76, hw, 0.36, cols=5, seed=6)
text(ax, x + 0.12 + hw / 2, BODY_TOP - 0.87, "RNA", 6.0, BLUE, "bold")
text(ax, x + 0.22 + 1.5 * hw, BODY_TOP - 0.87, "Protein", 6.0, MAGENTA, "bold")
text(ax, cx, BODY_TOP - 1.22, "Fully observed proteins;\nat least 5 genes and\nhalf the signature", 6.0)
text(ax, cx, BOTTOM + 0.22, "Gene z-scores capped at ±5,\naveraged and standardized", 5.9)

x, w = stage(ax, 3, "Paired model", MAGENTA, LIGHT_MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.18, "One fit per gene set", 6.8, weight="bold")
g_scatter_ellipse(ax, x + 0.40, BODY_TOP - 0.88, w - 0.80, 0.54)
text(ax, x + 0.36, BODY_TOP - 0.61, "Protein", 5.6, MAGENTA, ha="right")
text(ax, cx, BODY_TOP - 0.98, "RNA", 5.6, BLUE)
text(ax, cx, BODY_TOP - 1.24, "Separate RNA and protein\nBMI slopes; bivariate Student-t", 6.0, MAGENTA)
text(ax, cx, BOTTOM + 0.22, "Plex intercepts; within- and\nbetween-plex covariates", 5.9)

x, w = stage(ax, 4, "Outputs", MAGENTA, "white", edge=MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.24, "BMI slopes,\n95% HDI", 6.1, weight="bold")
g_paired_intervals(ax, x + 0.12, BODY_TOP - 0.98, w - 0.24, 0.50)
text(ax, x + w * 0.30, BODY_TOP - 1.10, "RNA", 5.8, BLUE, "bold")
text(ax, x + w * 0.70, BODY_TOP - 1.10, "Protein", 5.8, MAGENTA, "bold")
text(ax, cx, BODY_TOP - 1.38, "Protein − RNA", 6.0)
text(ax, cx, BODY_TOP - 1.62, "Residual correlation", 6.0)
text(ax, cx, BOTTOM + 0.18, "Mapped to source\nsignature labels", 5.9, MAGENTA)
save(fig, "F1", "p02", "rna_protein_schematic", "schematic")

fig, ax = new_canvas(COL_H + 0.26)
connectors(ax, BOTTOM + (COL_H - 0.24) / 2)
x, w = stage(ax, 0, "Same patients", BLUE, LIGHT_BLUE)
cx = x + w / 2
g_waffle(ax, x + 0.12, BODY_TOP - 0.66, w - 0.24, 0.50)
text(ax, cx, BODY_TOP - 0.82, f"{N_PATIENTS} patients", 7.0, weight="bold")
text(ax, cx, BODY_TOP - 1.18, "TMT proteomics,\nglyco- and\nphosphoproteomics", 6.2)
text(ax, cx, BOTTOM + 0.14, "BMI, age and sex", 6.2, MID_GREY)
x, w = stage(ax, 1, "PTM features", BLUE, PALE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, f"{N_PTM:,} PTM features", 6.8, weight="bold")
g_heatmap(ax, x + 0.24, BODY_TOP - 0.74, w - 0.48, 0.36, seed=17)
text(ax, cx, BODY_TOP - 1.08, "Glycosylation and\nphosphorylation sites", 6.2)
text(ax, cx, BODY_TOP - 1.50, "each paired with\nits parent protein", 6.2)
x, w = stage(ax, 2, "Paired scores", BLUE, PALE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "PTM and parent protein", 6.6, weight="bold")
hw = (w - 0.34) / 2
g_heatmap(ax, x + 0.12, BODY_TOP - 0.76, hw, 0.36, cols=5, seed=21)
g_heatmap(ax, x + 0.22 + hw, BODY_TOP - 0.76, hw, 0.36, cols=5, seed=22)
text(ax, x + 0.12 + hw / 2, BODY_TOP - 0.87, "Protein", 6.0, BLUE, "bold")
text(ax, x + 0.22 + 1.5 * hw, BODY_TOP - 0.87, "PTM", 6.0, MAGENTA, "bold")
text(ax, cx, BODY_TOP - 1.24, "Same patients and\nplexes in both", 6.0)
x, w = stage(ax, 3, "Paired model", MAGENTA, LIGHT_MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.18, "One fit per PTM feature", 6.8, weight="bold")
g_scatter_ellipse(ax, x + 0.40, BODY_TOP - 0.88, w - 0.80, 0.54)
text(ax, x + 0.36, BODY_TOP - 0.61, "PTM", 5.6, MAGENTA, ha="right")
text(ax, cx, BODY_TOP - 0.98, "Protein", 5.6, BLUE)
text(ax, cx, BODY_TOP - 1.24, "Separate protein and PTM\nBMI slopes; bivariate Student-t", 6.0, MAGENTA)
text(ax, cx, BOTTOM + 0.22, "Plex effects; age\nand sex adjusted", 5.9)
x, w = stage(ax, 4, "Outputs", MAGENTA, "white", edge=MAGENTA)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.24, "BMI slopes,\n95% HDI", 6.1, weight="bold")
g_paired_intervals(ax, x + 0.12, BODY_TOP - 0.98, w - 0.24, 0.50)
text(ax, x + w * 0.30, BODY_TOP - 1.10, "Protein", 5.8, BLUE, "bold")
text(ax, x + w * 0.70, BODY_TOP - 1.10, "PTM", 5.8, MAGENTA, "bold")
text(ax, cx, BODY_TOP - 1.42, "PTM given protein", 6.0)
text(ax, cx, BOTTOM + 0.18, "Linked to signature\ngenes", 5.9, MAGENTA)
save(fig, "F1", "p03", "ptm_schematic", "schematic")

SIM_PATIENTS, SIM_CT, SIM_FEATURES, SIM_OBS = 127, 4, 148, "18,796"
SCEN = [("Null", "no BMI effect"), ("Fitted effects", "as estimated"), ("Sparse", "one cell type"),
        ("Equicorrelated", "ρ = 0.6"), ("Contaminated", "5% outliers"), ("Small n", "80 patients")]
METHODS = [("Student-t intercept", "primary model", MAGENTA), ("Orthogonal", "patient-effect model", BLUE),
           ("OLS per feature", "with and without BH", INK), ("Cell-type composite", "OLS, HC3 intervals", MID_GREY),
           ("WHO group model", "Student-t intercept", "#0E7C7B"), ("OLS by group", "with and without BH", INK)]

def g_truth(ax, x, y, w, h):
    """Cell-type BMI effects (bars) with feature effects scattered around them."""
    means = [0.55, 0.35, 0.70, 0.25]
    cw = w / len(means)
    ax.plot([x, x + w], [y + 0.2 * h, y + 0.2 * h], color=INK, lw=0.6, zorder=4)
    rng = np.random.default_rng(5)
    for k, m in enumerate(means):
        cx_ = x + (k + 0.5) * cw; colour = MAGENTA if k % 2 == 0 else BLUE
        ax.plot([cx_ - cw * 0.32, cx_ + cw * 0.32], [y + m * h] * 2, color=colour, lw=2.0, solid_capstyle="butt", zorder=5)
        pts = (m + rng.normal(0, 0.11, 6)).clip(0.05, 0.98)
        ax.scatter(cx_ + rng.uniform(-0.22, 0.22, 6) * cw, y + pts * h, s=6, facecolor="white", edgecolor=colour, lw=0.6, zorder=6)

def item_box(ax, x, y, w, h, label, edge, sub=None, face="white"):
    box(ax, x, y, w, h, face=face, edge=edge, lw=0.9, r=0.04, z=4)
    if sub:
        text(ax, x + w / 2, y + h * 0.66, label, 5.9, edge if edge != MID_GREY else INK, "bold")
        text(ax, x + w / 2, y + h * 0.27, sub, 5.2, MID_GREY)
    else:
        text(ax, x + w / 2, y + h / 2, label, 5.9, edge if edge != MID_GREY else INK, "bold")

fig, ax = new_canvas(2.52)
connectors(ax, BOTTOM + (COL_H - 0.24) / 2)

x, w = stage(ax, 0, "Cohort design", BLUE, LIGHT_BLUE)
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.20, "From CPTAC PDAC", 6.8, weight="bold")
g_waffle(ax, x + 0.12, BODY_TOP - 0.86, w - 0.24, 0.50)
text(ax, cx, BODY_TOP - 1.02, f"{SIM_PATIENTS} patients", 7.0)
text(ax, cx, BODY_TOP - 1.28, "Observed BMI,\nage and sex", 6.4)
text(ax, cx, BODY_TOP - 1.56, "scores simulated", 6.0, MAGENTA, "bold")
text(ax, cx, BOTTOM + 0.24, f"{SIM_CT} immune fine cell\ntypes, {SIM_FEATURES} features", 6.0, MID_GREY)

x, w = stage(ax, 1, "Truth", BLUE, "white")
cx = x + w / 2
text(ax, cx, BODY_TOP - 0.18, "From the fitted model", 6.4, weight="bold")
g_truth(ax, x + 0.16, BODY_TOP - 0.86, w - 0.32, 0.56)
text(ax, cx, BODY_TOP - 1.04, "cell-type and feature\nBMI effects", 6.0, BLUE)
text(ax, cx, BODY_TOP - 1.44, "Residual scales and\nwithin-cell-type\ncorrelation", 6.0)
text(ax, cx, BOTTOM + 0.22, f"Low-read pattern kept\n({SIM_OBS} observations)", 5.8, MID_GREY)

x, w = stage(ax, 2, "Scenarios", BLUE, PALE)
cx = x + w / 2
bh, gap, top = 0.205, 0.03, BODY_TOP - 0.07
for i, (name, sub) in enumerate(SCEN):
    item_box(ax, x + 0.07, top - (i + 1) * bh - i * gap, w - 0.14, bh, name, BLUE, sub)
yy = top - 7 * bh - 6 * gap - 0.02
item_box(ax, x + 0.07, yy, w - 0.14, bh, "Effect-size grid", MAGENTA, "0.05 to 0.30 SD")
text(ax, cx, BOTTOM + 0.13, "20 datasets each", 6.0, weight="bold")

x, w = stage(ax, 3, "Methods", MAGENTA, LIGHT_MAGENTA)
cx = x + w / 2
bh, gap, top = 0.22, 0.05, BODY_TOP - 0.10
for i, (name, sub, colour) in enumerate(METHODS):
    item_box(ax, x + 0.08, top - (i + 1) * bh - i * gap, w - 0.16, bh, name, colour, sub)
text(ax, cx, BOTTOM + 0.22, "Same datasets, sampler\nand convergence criteria", 6.0)

x, w = stage(ax, 4, "Metrics", MAGENTA, "white", edge=MAGENTA)
cx = x + w / 2
i_simulation(ax, x + 0.18, BODY_TOP - 0.62, w - 0.36, 0.42)
text(ax, cx, BODY_TOP - 0.76, "power by effect size", 5.6, MID_GREY)
for k, line in enumerate(["Coverage", "False-call rate", "False-discovery\nproportion", "RMSE and width", "Convergence"]):
    text(ax, cx, BODY_TOP - 1.02 - k * 0.22 - (0.05 if k >= 3 else 0), line, 6.0)

save(fig, "S1", "p01", "simulation_engine_schematic", "schematic")


In [ ]:
from fig_common import *
S = SLOT["Figure 2"]
M = pd.read_csv(SIM_ROOT / "task_metrics_verified.csv")
SC = ["null_corr", "fitted_corr", "sparse_corr", "positive_copula", "contamination", "small_n"]
SCL = {"null_corr": "Null", "fitted_corr": "Fitted effects", "sparse_corr": "Sparse effects", "positive_copula": "Equicorrelated",
       "contamination": "Contaminated", "small_n": "Small n (80)"}
H, C = M[M.model.eq("hc_t") & M.scenario.isin(SC)], M[M.model.eq("composite") & M.scenario.isin(SC)]
set_value(sim_n_rep=int(H.groupby("scenario").replicate.nunique().min()))
def mean_ci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return (v.mean(), 1.96 * v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else (np.nan, np.nan)
TCOL, CCOL, OCOL = MAGENTA, MID_GREY, INK
O = pd.read_csv(SIM_ROOT / "ols_feature_benchmark_by_replicate.csv")
P2 = load_sim2(); P = P2[P2.model.eq("published") & P2.scenario.isin(SC) & P2.converged.astype(bool)]
PF_, PC_ = P[P.level.eq("feature")], P[P.level.eq("celltype")]

def scenario_rows(slot, series, xlabel, ref=None, xlim=None, stem="", rows=SC, xscale=None):
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=27, bottom=12, right=6, top=2)
    y = -np.arange(len(rows)); zebra(ax, y)
    for df, col, colour, marker, dy in series:
        for yy, s in zip(y, rows):
            m, ci = mean_ci(df[df.scenario.eq(s)][col]) if callable(col) is False else col(df, s)
            if not np.isfinite(m): continue
            ax.plot([max(m - ci, 0) if xscale else m - ci, m + ci], [yy + dy] * 2, color=colour, lw=4, alpha=0.35,
                    solid_capstyle="butt", zorder=2)
            ax.scatter([m], [yy + dy], s=SIZE["marker"], color=colour, marker=marker, edgecolor="white", lw=0.7, zorder=4)
    if ref is not None: ax.axvline(ref, color=INK, ls="--", lw=0.9, zorder=1)
    ax.set_yticks(y, [SCL[s] for s in rows]); ax.set_ylim(y.min() - 0.6, 0.6)
    if xscale == "sqrt":
        ax.set_xscale("function", functions=(lambda v: np.sqrt(np.clip(v, 0, None)), lambda v: v ** 2))
        ax.set_xticks([0, 0.01, 0.05, 0.1, 0.25, 0.5], ["0", "0.01", "0.05", "0.1", "0.25", "0.5"])
    if xlim: ax.set_xlim(*xlim)
    ax.set_xlabel(xlabel); style_axis(ax, "x"); row_axis(ax)
    save(fig, "F2", slot, stem, SIM_ROOT / "task_metrics_verified.csv")

scenario_rows("p01", [(H, "final_feature_coverage", TCOL, "o", 0.25), (PF_, "coverage", PUB, "o", 0.0), (O, "coverage", OCOL, "^", -0.25)],
              "Coverage, feature effects", 0.95, (0.3, 1.0), "feature_coverage")
scenario_rows("p02", [(H, "final_celltype_coverage", TCOL, "s", 0.25), (PC_, "coverage", PUB, "s", 0.0),
                      (C, "final_celltype_coverage", CCOL, "D", -0.25)],
              "Coverage, cell-type effects", 0.95, (0.3, 1.0), "celltype_coverage")
def fcr(level):
    def f(df, s):
        g = df[df.scenario.eq(s)]; nn = g[f"final_{level}_n_null"].astype(float)
        if nn.sum() == 0: return np.nan, np.nan
        rate = g[f"final_{level}_false_calls"].sum() / nn.sum()
        return rate, mean_ci(g[f"final_{level}_false_calls"] / nn.replace(0, np.nan))[1]
    return f
def ols_fcr(kind):
    def f(df, s):
        g = df[df.scenario.eq(s)]
        if g.n_null.sum() == 0: return np.nan, np.nan
        return g[f"{kind}_false"].sum() / g.n_null.sum(), mean_ci(g[f"{kind}_false"] / g.n_null.replace(0, np.nan))[1]
    return f
def pub_fcr(df, s):
    g = df[df.scenario.eq(s)]
    if g.n_null.sum() == 0: return np.nan, np.nan
    return g["false"].sum() / g.n_null.sum(), mean_ci(g["false"] / g.n_null.replace(0, np.nan))[1]
scenario_rows("p03", [(H, fcr("feature"), TCOL, "o", 0.36), (PF_, pub_fcr, PUB, "o", 0.24), (O, ols_fcr("unadj"), OCOL, "^", 0.12),
                      (O, ols_fcr("bh"), OCOL, "v", 0.0), (H, fcr("celltype"), TCOL, "s", -0.12), (PC_, pub_fcr, PUB, "s", -0.24),
                      (C, fcr("celltype"), CCOL, "D", -0.36)],
              "False-call rate among null effects", 0.05, (0, 0.62), "false_call_rate", rows=["null_corr", "sparse_corr"], xscale="sqrt")
for s in ["null_corr", "sparse_corr"]:
    for df, lev, tag in [(H, "feature", "hc_t_feature"), (H, "celltype", "hc_t_celltype"), (C, "celltype", "composite_celltype")]:
        r = fcr(lev)(df, s)[0]; set_value(**{f"fcr_{s}_{tag}": None if not np.isfinite(r) else round(float(r), 4)})
    for df, tag in [(PF_, "published_feature"), (PC_, "published_celltype")]:
        r = pub_fcr(df, s)[0]; set_value(**{f"fcr_{s}_{tag}": None if not np.isfinite(r) else round(float(r), 3)})
scenario_rows("p04", [(H, "final_feature_mean_width", TCOL, "o", 0.25), (PF_, "width", PUB, "o", 0.0), (O, "mean_width", OCOL, "^", -0.25)],
              "Interval width, feature effects (SD)", None, (0, None), "feature_width")
w = H.groupby("scenario").final_feature_mean_width.mean() / O.groupby("scenario").mean_width.mean()
cov_p = PF_.groupby("scenario").coverage.mean()
set_value(feature_width_ratio_min=round(float(w.min()), 2), feature_width_ratio_max=round(float(w.max()), 2),
          ols_fpr_null=round(float(O[O.scenario.eq("null_corr")].unadj_false.sum() / O[O.scenario.eq("null_corr")].n_null.sum()), 4),
          bh_fpr_null=round(float(O[O.scenario.eq("null_corr")].bh_false.sum() / O[O.scenario.eq("null_corr")].n_null.sum()), 4),
          pub_cov_min=f"{cov_p.min():.2f}", pub_cov_max=f"{cov_p.max():.2f}",
          pub_width_ratio=f"{(PF_.groupby('scenario').width.mean() / H.groupby('scenario').final_feature_mean_width.mean()).mean():.2f}")

order = celltype_order(); y = -block_positions(order, gap=0.8)
ppc = {f: pd.read_csv(SENS / f"{ARCH[f]}__{f}__lowread1000__ri_hc_t" / "publication" / "ppc_domain_checks.csv") for f in FAMS}
for slot, domain, xlabel, labels in [("p05", "conditional_residual_dependence", "Residual pairs with |r| ≥ 0.9", True),
                                     ("p06", "studentized_residual_tails", "Features outside tail range", False)]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=31 if labels else 4, bottom=12, right=3, top=2)
    zebra(ax, y)
    for (fam, ct), yy in zip(order, y):
        r = ppc[fam][ppc[fam].domain.eq(domain) & ppc[fam].celltype.eq(ct)].iloc[0]
        ax.plot([r.replicated_q025, max(r.replicated_q975, r.replicated_q025 + 0.15)], [yy, yy], color=BAND_GREY, lw=6,
                solid_capstyle="butt", zorder=2)
        over = r.observed_median > r.replicated_q975
        ax.scatter([r.observed_median], [yy], s=SIZE["marker"], color=MAGENTA if over else INK, edgecolor="white", lw=0.7, zorder=4)
    for fam in FAMS:
        g = ppc[fam]; set_value(**{f"tail_upper_p_{fam}": round(float(g[g.domain.eq("studentized_residual_tails") & g.celltype.eq("__GLOBAL__")].iloc[0].upper_p), 3)})
    ax.set_yticks(y, [ct_label(c, FAM_PREFIX[f]) for f, c in order] if labels else [""] * len(y), fontsize=FONT - 1)
    ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlim(0, None); ax.set_xlabel(xlabel); style_axis(ax, "x"); row_axis(ax)
    save(fig, "F2", slot, domain, "primary ppc_domain_checks.csv")

save_key("F2", "k01", "simulation_key",
         [Line2D([], [], marker="o", color=TCOL, ls="", ms=6, mec="white"), Line2D([], [], marker="o", color=PUB, ls="", ms=6, mec="white"),
          Line2D([], [], marker="^", color=OCOL, ls="", ms=6, mec="white"), Line2D([], [], marker="v", color=OCOL, ls="", ms=6, mec="white"),
          Line2D([], [], marker="D", color=CCOL, ls="", ms=5.5, mec="white"), Line2D([], [], marker="o", color=MID_GREY, ls="", ms=6, mec="white"),
          Line2D([], [], marker="s", color=MID_GREY, ls="", ms=6, mec="white"), Line2D([], [], color=INK, lw=4, alpha=0.35),
          Line2D([], [], color=INK, ls="--", lw=0.9)],
         [T_NAME, O_NAME, "OLS per feature", "OLS per feature + BH", "Composite (OLS)", "Circle, feature effect",
          "Square, cell-type effect", "95% Monte Carlo interval", "Nominal"],
         S["k01"]["w"], S["k01"]["h"], ncol=3)
save_key("F2", "k02", "ppc_key",
         [Line2D([], [], marker="o", color=INK, ls="", ms=6, mec="white"), Line2D([], [], marker="o", color=MAGENTA, ls="", ms=6, mec="white"),
          Line2D([], [], color=BAND_GREY, lw=6)],
         ["Observed, within replicated range", "Observed, above replicated range", "Replicated, 95% interval"],
         S["k02"]["w"], S["k02"]["h"], ncol=2)


In [ ]:
from fig_common import *
S = SLOT["Figure 3"]
PF, PC = load_primary()
order = celltype_order()
COLS = [ct for _, ct in order]
COL_LABEL = [ct_label(ct, FAM_PREFIX[f]) for f, ct in order]
PF["pdir"] = np.maximum(PF.p_positive, 1 - PF.p_positive)
PF["ring"] = strong(PF.p_positive)
set_value(n_features=len(PF), n_ring=int(PF.ring.sum()), n_feature_hdi_excl=int(PF.excludes_zero.sum()),
          n_ct=len(PC), n_ct_positive=int((PC["mean"] > 0).sum()))

def signature_rows(PF):
    g = PF.groupby("signature")
    rows = pd.DataFrame({"pdir": g.pdir.max(), "n_fam": g.family.nunique(), "fam": g.family.first(),
                         "extreme": g["mean"].agg(lambda v: v.loc[v.abs().idxmax()])})
    rows["home"] = np.where(rows.n_fam == 1, rows.fam, "shared")
    return rows

def draw_heatmap(rows_shown, figure, slot, w, h, stem, vmax):
    sub = PF[PF.signature.isin(rows_shown)]
    means = sub.pivot_table(index="signature", columns="celltype", values="mean", aggfunc="first").reindex(index=rows_shown, columns=COLS)
    rings = sub.pivot_table(index="signature", columns="celltype", values="ring", aggfunc="first").reindex_like(means)
    n_r, n_c = means.shape
    fig, ax = panel(w, h, left=47, bottom=35, right=2, top=2)
    ax.set_facecolor(EMPTY_CELL)
    ax.imshow(np.ma.masked_invalid(means.to_numpy(float)), cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto",
              interpolation="nearest", extent=(0, n_c, n_r, 0))
    for r in range(n_r):
        for c in range(n_c):
            if rings.iat[r, c] == True:
                ax.add_patch(mpatches.Circle((c + 0.5, r + 0.5), 0.22, facecolor="white", edgecolor=INK, lw=0.7, zorder=5))
    ax.set_xticks(np.arange(n_c + 1), minor=True); ax.set_yticks(np.arange(n_r + 1), minor=True)
    ax.grid(which="minor", color="#9A9DA2", linewidth=0.3); ax.tick_params(which="both", length=0, pad=2)
    ax.set_xticks(np.arange(n_c) + 0.5, COL_LABEL, rotation=90, ha="center", fontsize=FONT - 1)
    home = signature_rows(PF)["home"]
    ax.set_yticks(np.arange(n_r) + 0.5, rows_shown, fontsize=FONT - 1, fontweight="bold")
    for lab in ax.get_yticklabels():
        h_ = home.get(lab.get_text(), "shared"); lab.set_color(FAM_COLOR.get(h_, INK))
    for b in np.cumsum([sum(1 for f, _ in order if f == fam) for fam in FAMS])[:-1]:
        ax.axvline(b, color=INK, lw=0.9)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.9); sp.set_edgecolor(INK)
    save(fig, figure, slot, stem, PRIM)

rows = signature_rows(PF)
N_HEAT = 20
top = rows.sort_values(["pdir", "extreme"], ascending=False).head(N_HEAT)
ordered = []
for h_ in ["immune_coarse", "immune_fine", "nonimmune", "shared"]:
    ordered += top[top.home.eq(h_)].sort_values("extreme", ascending=False).index.tolist()
VMAX = float(np.ceil(PF["mean"].abs().max() * 50) / 50)
set_value(n_heat_rows=len(ordered), heat_vmax=VMAX)
draw_heatmap(ordered, "F3", "p01", S["p01"]["w"], S["p01"]["h"], "signature_heatmap", VMAX)
save_colorbar("F3", "k01", "heatmap_colourbar", DIVERGING, -VMAX, VMAX, "Posterior mean (SD per\n5 kg/m² higher BMI)",
              S["k01"]["w"], S["k01"]["h"])
save_key("F3", "k02", "heatmap_key",
         [Line2D([0], [0], marker="o", color=INK, mfc="white", ls="None", ms=6, mew=0.8), Patch(facecolor=EMPTY_CELL, edgecolor="#9A9DA2"),
          Patch(color=BLUE), Patch(color=MAGENTA), Patch(color=TEAL), Patch(color=INK)],
         ["P(>0) ≥ 0.95\nor ≤ 0.05", "Not scored", "Immune\ncoarse", "Immune fine", "Non-immune", "Shared"],
         S["k02"]["w"], S["k02"]["h"], ncol=1)

y = -block_positions(order, gap=0.8)
c = PC.set_index("celltype").reindex(COLS)
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=31, bottom=14, right=12, top=5)
zebra(ax, y); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
forest(ax, y, c["mean"], c.hdi95_low, c.hdi95_high, c.p_positive)
right_column(ax, y, [f"{p:.2f}" for p in c.p_positive], header="P(>0)")
span = symmetric_span(c.hdi95_low, c.hdi95_high); ax.set_xlim(-span, span)
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5, symmetric=True))
ax.set_yticks(y, [ct_label(ct, FAM_PREFIX[f]) for f, ct in order], fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
ax.set_xlabel(EFFECT_LABEL); style_axis(ax, "x"); row_axis(ax)
save(fig, "F3", "p02", "celltype_forest", PRIM)

tam = PF[PF.celltype.eq("immune_fine||TAM")].sort_values("p_positive", ascending=False).head(10)
yy = -np.arange(len(tam))
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=43, bottom=14, right=3, top=2)
zebra(ax, yy); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
forest(ax, yy, tam["mean"], tam.hdi95_low, tam.hdi95_high, tam.p_positive)
span = symmetric_span(tam.hdi95_low, tam.hdi95_high); ax.set_xlim(-span, span)
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5, symmetric=True))
ax.set_yticks(yy, [sig_label(f) for f in tam.feature], fontsize=FONT - 1); ax.set_ylim(yy.min() - 0.6, 0.6)
ax.set_xlabel(EFFECT_SHORT); style_axis(ax, "x"); row_axis(ax)
save(fig, "F3", "p03", "tam_forest", PRIM)
h = evidence_handles()
save_key("F3", "k03", "evidence_key", h, [x.get_label().replace(", P", ",\nP") for x in h], S["k03"]["w"], S["k03"]["h"], ncol=1)


In [ ]:
from fig_common import *
S = SLOT["Figure 4"]
PF, PC = load_primary()
order = celltype_order(); COLS = [ct for _, ct in order]; y = -block_positions(order, gap=0.8)
SENS_JOBS = lambda fam: ["bmi_wide", "bmi_narrow", INT_WIDE[fam], "int_narrow", ALLREF[fam], "readscale_lowread", RS_ALL[fam]]
rng_tab = pd.concat([pd.read_csv(ROOT / f"{fam}__{job}" / "celltype_summary.csv").query("estimand == 'per_5kg_m2'")
                     [["celltype", "mean"]].assign(family=fam, job=job) for fam in FAMS for job in SENS_JOBS(fam)])
set_value(n_sens_fits=len(SENS_JOBS("immune_coarse")))
mm = rng_tab.merge(PC[["celltype", "mean"]], on="celltype", suffixes=("", "_p"))
set_value(max_sens_celltype_shift=round(float((mm["mean"] - mm["mean_p"]).abs().max()), 3))

c = PC.set_index("celltype").reindex(COLS); g = rng_tab.groupby("celltype")["mean"]
lo_r, hi_r = g.min().reindex(COLS), g.max().reindex(COLS)
cat = pd.concat([pd.read_csv(ROOT / f"{f}__categorical_all" / "celltype_summary.csv") for f in FAMS])
o = cat[cat.estimand.eq("obese_vs_normal")].set_index("celltype").reindex(COLS)
fam_of = [f for f, _ in order]
fig, (ax1, ax2) = shared_row_axes(S["p01"]["w"], S["p01"]["h"], left=31, n=2, gap=7, bottom=14)
for ax, mean, lo, hi, p, xl, rng_ in [(ax1, c["mean"], c.hdi95_low, c.hdi95_high, c.p_positive, EFFECT_LABEL, True),
                                      (ax2, o["mean"], o.hdi95_low, o.hdi95_high, o.p_positive, "Obese vs normal weight,\npathway score (SD)", False)]:
    group_bands(ax, y, fam_of); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
    if rng_: ax.hlines(y, lo_r, hi_r, color=MID_GREY, lw=7, alpha=0.45, zorder=1.5)
    forest(ax, y, mean, lo, hi, p)
    span = symmetric_span(lo, hi); ax.set_xlim(-span, span); ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5, symmetric=True))
    ax.set_yticks(y, [ct_label(ct, FAM_PREFIX[f]) for f, ct in order] if ax is ax1 else [""] * len(y), fontsize=FONT - 1)
    ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(xl); style_axis(ax, "x"); row_axis(ax)
save(fig, "F4", "p01", "celltype_primary_range_and_obese_contrast", ROOT)
set_value(n_cat_celltype_excl=int(cat.excludes_zero.sum()))

for slot, fam in zip(["p03", "p04", "p05"], FAMS):
    d = ROOT / f"{fam}__categorical_all"
    pat = pd.read_csv(d / "ppc" / "patient_diagnostics.csv")
    dz = pat.groupby("bmi_group").bmi_z.mean(); per5 = json.loads((d / "diagnostics.json").read_text())["per5_factor"]
    k = float(dz["obese"] - dz["normal_weight"]) / per5
    cont = pd.read_csv(ROOT / f"{fam}__{ALLREF[fam]}" / "feature_summary.csv").query("estimand == 'per_5kg_m2'")
    f = pd.read_csv(d / "feature_summary.csv").query("estimand == 'obese_vs_normal'")
    m = f.merge(cont[["feature", "mean"]], on="feature", suffixes=("_cat", "_cont"))
    xv, yv = m.mean_cont * k, m.mean_cat
    r = float(np.corrcoef(xv, yv)[0, 1])
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=15, bottom=14, right=3, top=3)
    lo, hi = float(min(xv.min(), yv.min())), float(max(xv.max(), yv.max())); pad = 0.06 * (hi - lo)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=INK, ls="--", lw=0.9, zorder=1)
    ax.axhline(0, color=GRID_GREY, lw=0.8, zorder=0); ax.axvline(0, color=GRID_GREY, lw=0.8, zorder=0)
    points(ax, xv, yv, dir_colors(yv), strong(m.p_positive), size=12)
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad); ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Continuous (SD)"); ax.set_ylabel("Group model (SD)")
    ax.text(0.04, 0.96, f"r = {r:.2f}", transform=ax.transAxes, ha="left", va="top")
    tag = fam.replace("immune_", "")
    set_value(**{f"r_cat_{tag}": f"{r:.2f}", f"bmi_scale_obese_normal_{tag}": round(k, 3)})
    save(fig, "F4", slot, f"group_vs_continuous_allprofiles_{tag}", d)

h = evidence_handles() + [Line2D([0], [0], color=MID_GREY, lw=7, alpha=0.45)]
save_key("F4", "k01", "key", h, [x.get_label() for x in h[:-1]] + ["Range of sensitivity-fit posterior means"],
         S["k01"]["w"], S["k01"]["h"], ncol=2)


In [ ]:
from fig_common import *
SC = ["null_corr", "fitted_corr", "sparse_corr", "positive_copula", "contamination", "small_n"]
SCL = {"null_corr": "Null", "fitted_corr": "Fitted effects", "sparse_corr": "Sparse effects", "positive_copula": "Equicorrelated",
       "contamination": "Contaminated", "small_n": "Small n (80)"}
def mean_ci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return (v.mean(), 1.96 * v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else ((v.mean(), 0.0) if len(v) else (np.nan, np.nan))
K1C = MID_GREY
O = pd.read_csv(SIM_ROOT / "ols_feature_benchmark_by_replicate.csv")

S = {**SLOT["Figure S2"], **SLOT["Figure S2 (continued 2)"]}; M = pd.read_csv(SIM_ROOT / "task_metrics_verified.csv")
H, C = (M[M.model.eq(m) & M.scenario.isin(SC)] for m in ("hc_t", "composite"))
K_all = M[M.model.eq("factor_K1")]; K = K_all[K_all.scenario.isin(SC)]
set_value(k1_converged=int(K.converged_first.sum()), k1_fits=int(len(K)), k1_converged_all=int(K_all.converged_first.sum()),
          k1_attempted_all=int(len(K_all)), k1_converged_final_all=int(K_all.converged_final.sum()))
Hc = H[H.converged_final.astype(bool)]; P2 = load_sim2()
GRID = [0.05, 0.10, 0.20, 0.30]; GS = [f"grid_{e:.2f}" for e in GRID]
def rate(g, num, den):
    d = g[den].sum(); return g[num].sum() / d if d else np.nan
def sel(model, level="feature", tag=None, contrast=None, scen=None, all_fits=False):
    g = P2[P2.model.eq(model) & P2.level.eq(level)]
    if not all_fits: g = g[g.converged.astype(bool)]
    if tag: g = g[g.method_tag.eq(tag)]
    if contrast: g = g[g.contrast.eq(contrast)]
    if scen is not None: g = g[g.scenario.isin(scen)]
    return g
COL = {T_NAME: MAGENTA, O_NAME: TEAL, "OLS + BH": BLUE, "OLS": "#4A4A4A"}
LS = {T_NAME: "-", O_NAME: "-", "OLS + BH": "--", "OLS": ":"}
NONNULL = [s for s in SC if s != "null_corr"]
def rows_axis(slot, rows, xlabel, xmax=1.0, left=30, pct_ticks=True):
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=left, bottom=12, right=3, top=2)
    y = -np.arange(len(rows)); ax.set_yticks(y, rows); ax.set_ylim(y.min() - 0.6, 0.6)
    ax.set_xlabel(xlabel); style_axis(ax, "x"); row_axis(ax)
    if pct_ticks: ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v * 100:.0f}%"))
    return fig, ax, y

GRID_SERIES = [(T_NAME, sel("hc_t", scen=GS)), (O_NAME, sel("published", scen=GS)), ("OLS", sel("ols", tag="unadj", scen=GS)),
               ("OLS + BH", sel("ols", tag="bh", scen=GS))]
for slot, what, ylabel, ylim in [("p01", "power", "Power", (0, 1.02)), ("p02", "coverage", "Coverage", (0.4, 1.0))]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=16, bottom=13, right=3, top=2); style_axis(ax, "y")
    for name, g in GRID_SERIES:
        if what == "coverage" and name == "OLS + BH": continue
        vals, cis = [], []
        for gs in GS:
            gg = g[g.scenario.eq(gs)]
            if what == "power": vals.append(rate(gg, "true", "n_nonnull")); cis.append(mean_ci(gg["true"] / gg.n_nonnull)[1])
            else: m, ci = mean_ci(gg.coverage); vals.append(m); cis.append(ci)
        vals, cis = np.array(vals), np.array(cis)
        ax.fill_between(GRID, vals - cis, vals + cis, color=COL[name], alpha=0.28, lw=0, zorder=1)
        ax.plot(GRID, vals, color=COL[name], lw=1.5, ls=LS[name], marker="o", ms=3.5, zorder=3)
        set_value(**{f"grid_{what}_{SLUG[name]}_{round(e * 100):02d}": round(float(v), 3) for e, v in zip(GRID, vals)})
    if what == "coverage": ax.axhline(0.95, color=INK, ls=":", lw=0.9, zorder=1)
    ax.set_xticks(GRID, ["0.05", "0.1", "0.2", "0.3"]); ax.set_xlim(0.03, 0.32); ax.set_ylim(*ylim)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v * 100:.0f}%"))
    ax.set_xlabel("Mean true effect (SD per BMI SD)"); ax.set_ylabel(ylabel)
    save(fig, "S2", slot, f"grid_{what}", SIM_ROOT / "tasks_v2")
save_key("S2", "k01", "key_lines", [Line2D([], [], color=COL[n], ls=LS[n], lw=1.5) for n in [T_NAME, O_NAME, "OLS", "OLS + BH"]] +
         [Patch(color=MID_GREY, alpha=0.4), Line2D([], [], color=INK, ls=":", lw=0.9)],
         [T_NAME, O_NAME, "OLS (95% CI)", "OLS + Benjamini–Hochberg", "±1.96 Monte Carlo SE", "Nominal 95%"], S["k01"]["w"], S["k01"]["h"], ncol=1)

DB3 = [T_NAME, O_NAME, "OLS + BH"]
fig, ax, y = rows_axis("p03", [SCL[s] for s in NONNULL], "Power")
vals = {T_NAME: [rate(Hc[Hc.scenario.eq(s)], "final_feature_true_calls", "final_feature_n_nonnull") for s in NONNULL],
        O_NAME: [rate(sel("published", scen=[s]), "true", "n_nonnull") for s in NONNULL],
        "OLS + BH": [rate(O[O.scenario.eq(s)], "bh_true", "n_nonnull") for s in NONNULL]}
dumbbell(ax, y, [(vals[n], COL[n]) for n in DB3], xmax=1.0); save(fig, "S2", "p03", "power_scenarios", SIM_ROOT)
for n in DB3:
    set_value(**{f"power_{SLUG[n]}_{s}": round(float(v), 3) for s, v in zip(NONNULL, vals[n])})
fig, ax, y = rows_axis("p04", [SCL[s] for s in SC], "Coverage of 95% intervals")
cov = {T_NAME: [Hc[Hc.scenario.eq(s)].final_feature_coverage.mean() for s in SC], O_NAME: [sel("published", scen=[s]).coverage.mean() for s in SC],
       "OLS": [O[O.scenario.eq(s)].coverage.mean() for s in SC]}
dumbbell(ax, y, [(cov[n], COL[n]) for n in [T_NAME, O_NAME, "OLS"]], xmax=1.0); ax.axvline(0.95, color=INK, ls=":", lw=0.9, zorder=0)
save(fig, "S2", "p04", "coverage_scenarios", SIM_ROOT)
set_value(pub_cov_min=f"{min(cov[O_NAME]):.2f}", pub_cov_max=f"{max(cov[O_NAME]):.2f}")
fig, ax, y = rows_axis("p05", [SCL[s] for s in SC], "RMSE (SD × 100)")
rm = {T_NAME: [Hc[Hc.scenario.eq(s)].final_feature_rmse.mean() for s in SC], O_NAME: [sel("published", scen=[s]).rmse.mean() for s in SC],
      "OLS": [O[O.scenario.eq(s)].rmse.mean() for s in SC]}
xm = float(np.nanmax(np.array(list(rm.values())))) * 1.15
dumbbell(ax, y, [(rm[n], COL[n]) for n in [T_NAME, O_NAME, "OLS"]], xmax=xm)
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v * 100:.0f}")); save(fig, "S2", "p05", "rmse_scenarios", SIM_ROOT)
FROWS = [("Null: any call", "null_corr", "any"), ("Sparse: any false call", "sparse_corr", "anyfalse"), ("Sparse: mean FDP", "sparse_corr", "fdp")]
def fstat(false, calls, kind):
    f, c = np.asarray(false, float), np.asarray(calls, float)
    if kind == "any": return float((c > 0).mean())
    if kind == "anyfalse": return float((f > 0).mean())
    return float(np.mean(np.where(c > 0, f / np.where(c > 0, c, 1), 0.0)))
fig, ax, y = rows_axis("p06", [r[0] for r in FROWS], "Share of datasets / mean FDP", left=38)
fv = {T_NAME: [], O_NAME: [], "OLS + BH": []}
for _, s, kind in FROWS:
    h_, p_, o_ = Hc[Hc.scenario.eq(s)], sel("published", scen=[s]), O[O.scenario.eq(s)]
    fv[T_NAME].append(fstat(h_.final_feature_false_calls, h_.final_feature_n_calls, kind))
    fv[O_NAME].append(fstat(p_["false"], p_.calls, kind)); fv["OLS + BH"].append(fstat(o_.bh_false, o_.bh_calls, kind))
dumbbell(ax, y, [(fv[n], COL[n]) for n in DB3], xmax=1.0); save(fig, "S2", "p06", "false_discoveries", SIM_ROOT)
for n in DB3:
    set_value(**{f"false_{SLUG[n]}_{k}": round(v, 3) for (_, _, k), v in zip(FROWS, fv[n])})
set_value(t_null_any_call=f"{fv[T_NAME][0]:.2f}", t_null_fcr=f"{rate(Hc[Hc.scenario.eq('null_corr')], 'final_feature_false_calls', 'final_feature_n_null'):.3f}")
save_key("S2", "k02", "key_circles", [Line2D([], [], marker="o", ls="", mfc="white", mec=COL[n], ms=8, mew=1.1) for n in [T_NAME, O_NAME, "OLS", "OLS + BH"]] +
         [Line2D([], [], color=LIGHT_MAGENTA, lw=3.2), Line2D([], [], color=INK, ls=":", lw=0.9)],
         [T_NAME, O_NAME, "OLS (95% CI)", "OLS + Benjamini–Hochberg", "Range between methods", "Nominal 95%"], S["k02"]["w"], S["k02"]["h"], ncol=3)

GCOL = {"Group model": MAGENTA, "OLS by group": "#4A4A4A", "OLS by group + BH": BLUE}
CON = [("overweight_vs_normal", "Overweight vs normal"), ("obese_vs_normal", "Obese vs normal")]
fig, ax, y = rows_axis("p07", [l for _, l in CON], "Power")
gp = {"Group model": [rate(sel("group_t", contrast=c, scen=NONNULL), "true", "n_nonnull") for c, _ in CON],
      "OLS by group + BH": [rate(sel("ols_group", tag="bh", contrast=c, scen=NONNULL), "true", "n_nonnull") for c, _ in CON]}
dumbbell(ax, y, [(gp[n], GCOL[n]) for n in gp], xmax=0.3); save(fig, "S2", "p07", "group_power", SIM_ROOT / "tasks_v2")
fig, ax, y = rows_axis("p08", [l for _, l in CON], "Null datasets with any call")
gn = {"Group model": [float((sel("group_t", contrast=c, scen=["null_corr"]).calls > 0).mean()) for c, _ in CON],
      "OLS by group + BH": [float((sel("ols_group", tag="bh", contrast=c, scen=["null_corr"]).calls > 0).mean()) for c, _ in CON]}
dumbbell(ax, y, [(gn[n], GCOL[n]) for n in gn], xmax=1.0); save(fig, "S2", "p08", "group_null_calls", SIM_ROOT / "tasks_v2")
set_value(group_power_max=f"{max(gp['Group model']):.3f}", group_null_any=f"{max(gn['Group model']):.2f}")
fig, ax, y = rows_axis("p09", [SCL[s] for s in SC], "Coverage, obese vs normal")
gc = {"Group model": [sel("group_t", contrast="obese_vs_normal", scen=[s]).coverage.mean() for s in SC],
      "OLS by group": [sel("ols_group", tag="unadj", contrast="obese_vs_normal", scen=[s]).coverage.mean() for s in SC]}
dumbbell(ax, y, [(gc[n], GCOL[n]) for n in gc], xmax=1.0); ax.axvline(0.95, color=INK, ls=":", lw=0.9, zorder=0)
save(fig, "S2", "p09", "group_coverage", SIM_ROOT / "tasks_v2")
set_value(group_n_copula_used=int(sel("group_t", contrast="obese_vs_normal", scen=["positive_copula"]).replicate.nunique()))
save_key("S2", "k03", "key_group", [Line2D([], [], marker="o", ls="", mfc="white", mec=GCOL[n], ms=8, mew=1.1) for n in GCOL],
         ["WHO group model (Student-t intercept)", "OLS by group (95% CI)", "OLS by group + Benjamini–Hochberg"], S["k03"]["w"], S["k03"]["h"], ncol=3)
fig, ax, y = rows_axis("p10", [SCL[s] for s in NONNULL], "Cell-type power")
cd_ = {T_NAME: [rate(Hc[Hc.scenario.eq(s)], "final_celltype_true_calls", "final_celltype_n_nonnull") for s in NONNULL],
       "Composite": [rate(C[C.scenario.eq(s)], "final_celltype_true_calls", "final_celltype_n_nonnull") for s in NONNULL]}
dumbbell(ax, y, [(cd_[T_NAME], MAGENTA), (cd_["Composite"], INK)], xmax=1.0); save(fig, "S2", "p10", "celltype_power", SIM_ROOT)
save_key("S2", "k04", "key_celltype", [Line2D([], [], marker="o", ls="", mfc="white", mec=c, ms=8, mew=1.1) for c in (MAGENTA, INK)],
         [f"{T_NAME}, cell-type mean", "Cell-type composite (OLS)"], S["k04"]["w"], S["k04"]["h"], ncol=1)
gt_all = sel("group_t", "feature", contrast="obese_vs_normal", scen=SC, all_fits=True); pub_all = sel("published", scen=SC, all_fits=True)
CONV = [(T_NAME, H), (O_NAME, pub_all), ("WHO group model", gt_all), ("One-factor model", K)]
fig, ax, y = rows_axis("p11", [n for n, _ in CONV], "Fits converged at first attempt", left=40)
for yy, (n, df) in zip(y, CONV):
    v = float(df.converged_first.astype(bool).mean()); fin = float(df.converged_final.astype(bool).mean()) if "converged_final" in df else float(df.converged.astype(bool).mean())
    ax.barh(yy, 1.0, height=0.62, color=BAND_GREY, zorder=1); ax.barh(yy, v, height=0.62, color=MAGENTA, zorder=2)
    ax.text(1.02, yy, f"{v * 100:.0f}% ({fin * 100:.0f}%)", va="center", ha="left", fontsize=SIZE["value"], color=MAGENTA,
            transform=ax.get_yaxis_transform())
ax.set_xlim(0, 1); save(fig, "S2", "p11", "convergence", SIM_ROOT)
pc = gt_all[gt_all.scenario.eq("positive_copula")]
set_value(group_conv_first_copula=int(pc.converged_first.astype(bool).sum()), group_conv_final_copula=int(pc.converged.astype(bool).sum()),
          group_n_copula=int(len(pc)), pub_conv_first=int(pub_all.converged_first.astype(bool).sum()), pub_n=int(len(pub_all)))
w = H.groupby("scenario").final_celltype_mean_width.mean() / C.groupby("scenario").final_celltype_mean_width.mean()
set_value(ct_width_ratio_min=f"{w.min():.2f}", ct_width_ratio_max=f"{w.max():.2f}")

# ---------------- S3
S = SLOT["Figure S3"]
diag = pd.concat([pd.read_csv(SENS / f"{ARCH[f]}__{f}__lowread1000__ri_hc_t" / "publication" / "diagnostic_families.csv").assign(family=f) for f in FAMS])
fams_p = list(dict.fromkeys(diag.parameter_family)); yy = -np.arange(len(fams_p))
PNAME = {"feature_baseline": "Feature intercept", "feature_bmi_slope": "Feature BMI effect",
         "feature_bmi_raw": "Feature BMI deviation (std.)", "feature_sigma_bmi": "Between-feature BMI SD",
         "celltype_bmi_slope": "Cell-type mean BMI effect", "feature_covariate_slope": "Feature age and sex effects",
         "feature_covariate_raw": "Feature age/sex deviation (std.)", "feature_sigma_covariate": "Between-feature age/sex SD",
         "celltype_covariate_slope": "Cell-type age and sex effects", "celltype_intercept_w": "Patient × cell-type intercept",
         "celltype_intercept_sigma": "Intercept SD, by cell type", "intercept_nu": "Intercept degrees of freedom",
         "intercept_nu_minus_two": "Intercept degrees of freedom − 2", "sigma_feature": "Feature residual scale",
         "log_sigma_location": "Residual scale, overall (log)", "log_sigma_celltype_location": "Residual scale, cell type (log)",
         "log_sigma_celltype_location_sd": "Residual scale, between-cell-type SD", "log_sigma_celltype_raw": "Residual scale, cell-type dev. (std.)",
         "log_sigma_feature_scale": "Residual scale, between-feature SD", "log_sigma_raw": "Residual scale, feature dev. (std.)",
         "nu": "Residual degrees of freedom", "nu_minus_two": "Residual degrees of freedom − 2"}
for slot, cols, xlabel, ref, logx in [("p01", ["max_rhat"], "Maximum R-hat", 1.01, False), ("p02", ["min_ess_bulk", "min_ess_tail"], "Minimum ESS", 400, True)]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=50 if slot == "p01" else 4, bottom=12, right=3, top=2)
    zebra(ax, yy)
    for k, f in enumerate(FAMS):
        d = diag[diag.family.eq(f)].set_index("parameter_family").reindex(fams_p)
        for col, filled in zip(cols, [True, False]):
            ax.scatter(d[col], yy + (k - 1) * 0.24, s=18, marker="o", color=FAM_COLOR[f] if filled else "white",
                       edgecolor="white" if filled else FAM_COLOR[f], lw=0.7 if filled else 1.0, zorder=4)
    ax.axvline(ref, color=INK, ls="--", lw=0.9)
    ax.set_yticks(yy, [p.replace("_", " ") for p in fams_p] if slot == "p01" else [""] * len(yy), fontsize=FONT - 1)
    ax.set_ylim(yy.min() - 0.6, 0.6); ax.set_xlabel(xlabel); style_axis(ax, "x"); row_axis(ax)
    if logx: ax.set_xscale("log")
    save(fig, "S3", slot, "rhat" if slot == "p01" else "ess", "primary diagnostic_families.csv")
save_key("S3", "k01", "key", [Line2D([], [], marker="o", color=FAM_COLOR[f], ls="", ms=5, mec="white") for f in FAMS]
         + [Line2D([], [], marker="o", mfc="white", mec=INK, ls="", ms=5)], [FAM_LABEL[f] for f in FAMS] + ["Tail ESS (open)"],
         S["k01"]["w"], S["k01"]["h"], ncol=4)

# ---------------- S4
S = SLOT["Figure S4"]
order = celltype_order(); y = -block_positions(order, gap=0.8)
ppc = {f: pd.read_csv(SENS / f"{ARCH[f]}__{f}__lowread1000__ri_hc_t" / "publication" / "ppc_domain_checks.csv") for f in FAMS}
for slot, domain, xlabel, labels in [("p01", "raw_outcome_dependence", "Raw pairs with |r| ≥ 0.9", True),
                                     ("p02", "residual_dispersion", "Features outside dispersion range", False)]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=31 if labels else 4, bottom=12, right=3, top=2); zebra(ax, y)
    for (fam, ct), yy_ in zip(order, y):
        r = ppc[fam][ppc[fam].domain.eq(domain) & ppc[fam].celltype.eq(ct)].iloc[0]
        ax.plot([r.replicated_q025 - 0.08, max(r.replicated_q975, r.replicated_q025) + 0.08], [yy_, yy_], color="#CFD1D6", lw=6, solid_capstyle="butt", zorder=2)
        over = r.observed_median > r.replicated_q975
        ax.scatter([r.observed_median], [yy_], s=SIZE["marker"], color=MAGENTA if over else INK, edgecolor="white", lw=0.7, zorder=4)
    ax.set_yticks(y, [ct_label(c, FAM_PREFIX[f]) for f, c in order] if labels else [""] * len(y), fontsize=FONT - 1)
    ax.set_ylim(y.min() - 0.6, 0.6); xm = ax.get_xlim()[1]; ax.set_xlim(-0.05 * xm, xm)
    ax.set_xlabel(xlabel); style_axis(ax, "x"); row_axis(ax)
    save(fig, "S4", slot, domain, "primary ppc_domain_checks.csv")
_d = pd.concat([ppc[f][ppc[f].domain.eq("residual_dispersion")] for f in FAMS])
set_value(n_ct_disp_out=int((_d.observed_median > 0).sum()), n_ct_disp=int(len(_d)))
save_key("S4", "k01", "key", [Line2D([], [], marker="o", color=INK, ls="", ms=6, mec="white"), Line2D([], [], marker="o", color=MAGENTA, ls="", ms=6, mec="white"),
         Line2D([], [], color=BAND_GREY, lw=6)], ["Observed, within replicated range", "Observed, above replicated range", "Replicated, 95% interval"],
         S["k01"]["w"], S["k01"]["h"], ncol=2)


In [ ]:
from fig_common import *
PF, PC = load_primary()
order = celltype_order(); COLS = [ct for _, ct in order]; COL_LABEL = [ct_label(ct, FAM_PREFIX[f]) for f, ct in order]
PF["ring"] = strong(PF.p_positive)
set_value(n_features=len(PF))

S = SLOT["Figure S5"]; y = -block_positions(order, gap=0.8); rng = np.random.default_rng(1)
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=31, bottom=14, right=3, top=2)
zebra(ax, y); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
for (fam, ct), yy in zip(order, y):
    g = PF[PF.celltype.eq(ct)]
    points(ax, g["mean"], yy + rng.uniform(-0.3, 0.3, len(g)), dir_colors(g["mean"]), g.ring, size=10)
span = symmetric_span(PF["mean"]); ax.set_xlim(-span, span)
ax.set_yticks(y, [ct_label(c, FAM_PREFIX[f]) for f, c in order], fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
ax.set_xlabel(EFFECT_LABEL); style_axis(ax, "x"); row_axis(ax)
save(fig, "S5", "p01", "all_features", PRIM)
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=15, bottom=13, right=3, top=2)
for fam in FAMS:
    ax.hist(PF[PF.family.eq(fam)].p_positive, bins=np.linspace(0, 1, 21), density=True, histtype="step", color=FAM_COLOR[fam], lw=1.5)
    set_value(**{f"frac_pos_{fam.replace('immune_', '')}": round(float((PF[PF.family.eq(fam)].p_positive > 0.5).mean()), 2)})
ax.set_xlim(0, 1); ax.set_xlabel("P(effect > 0)"); ax.set_ylabel("Density"); style_axis(ax, "y")
save(fig, "S5", "p02", "p_positive_distribution", PRIM)
h = evidence_handles()
save_key("S5", "k01", "key", h + [Line2D([], [], color=FAM_COLOR[f], lw=1.5) for f in FAMS],
         [x.get_label() for x in h] + [f"{FAM_LABEL[f]} (b)" for f in FAMS], S["k01"]["w"], S["k01"]["h"], ncol=1)

g = PF.groupby("signature")
rows = pd.DataFrame({"n_fam": g.family.nunique(), "n_ct": g.celltype.nunique(), "fam": g.family.first(),
                     "extreme": g["mean"].agg(lambda v: v.loc[v.abs().idxmax()])})
rows = rows[rows.n_ct >= 2]
set_value(n_shared_signatures=len(rows))
PF[["family", "feature", "celltype", "signature", "mean", "hdi95_low", "hdi95_high", "p_positive", "excludes_zero"]].assign(
    lfsr=np.minimum(PF.p_positive, 1 - PF.p_positive)).to_csv(OUT / "Supplementary_Data_cellular_feature_effects.csv", index=False)
rows["home"] = np.where(rows.n_fam == 1, rows.fam, "shared")
ordered = []
for h_ in ["immune_coarse", "immune_fine", "nonimmune", "shared"]:
    ordered += rows[rows.home.eq(h_)].sort_values("extreme", ascending=False).index.tolist()
VMAX = float(np.ceil(PF["mean"].abs().max() * 50) / 50)
ROW_MM, PAGE_H = 4.3, 262
per_page = int((PAGE_H - 37) // ROW_MM)
first = int((PAGE_H - 37 - 40) // ROW_MM)
pages = [ordered[:first]] + [ordered[i:i + per_page] for i in range(first, len(ordered), per_page)]
L = json.loads((OUT / "layout.json").read_text()); CAPS = json.loads((OUT / "captions.json").read_text())
for k in [k for k in L["figures"] if k.startswith("Figure S6")]:
    L["figures"].pop(k); CAPS.pop(k, None) if k != "Figure S6" else None
for i, rows_shown in enumerate(pages, 1):
    key = "Figure S6" if i == 1 else f"Figure S6 (continued {i})"
    h_mm = 37 + ROW_MM * len(rows_shown)
    L["figures"][key] = dict(deck="supp", number="6", caption_y=round(15 + h_mm + 6, 1), slots=[
        dict(slot=f"pg{i}", x=15, y=15, w=145, h=round(h_mm, 1)), dict(slot=f"k1_{i}", x=162, y=15, w=33, h=58),
        dict(slot=f"k2_{i}", x=162, y=76, w=33, h=50)])
    if i > 1:
        CAPS[key] = dict(title="(continued).", panels=[""])
    sub = PF[PF.signature.isin(rows_shown)]
    means = sub.pivot_table(index="signature", columns="celltype", values="mean", aggfunc="first").reindex(index=rows_shown, columns=COLS)
    rings = sub.pivot_table(index="signature", columns="celltype", values="ring", aggfunc="first").reindex_like(means)
    n_r, n_c = means.shape
    fig, ax = panel(145, h_mm, left=50, bottom=35, right=2, top=2)
    ax.set_facecolor(EMPTY_CELL)
    ax.imshow(np.ma.masked_invalid(means.to_numpy(float)), cmap=DIVERGING, vmin=-VMAX, vmax=VMAX, aspect="auto",
              interpolation="nearest", extent=(0, n_c, n_r, 0))
    for r in range(n_r):
        for c in range(n_c):
            if rings.iat[r, c] == True:
                ax.add_patch(mpatches.Circle((c + 0.5, r + 0.5), 0.22, facecolor="white", edgecolor=INK, lw=0.7, zorder=5))
    ax.set_xticks(np.arange(n_c + 1), minor=True); ax.set_yticks(np.arange(n_r + 1), minor=True)
    ax.grid(which="minor", color="#9A9DA2", linewidth=0.3); ax.tick_params(which="both", length=0, pad=2)
    ax.set_xticks(np.arange(n_c) + 0.5, COL_LABEL, rotation=90, ha="center", fontsize=FONT - 1)
    ax.set_yticks(np.arange(n_r) + 0.5, rows_shown, fontsize=FONT - 1, fontweight="bold")
    for lab in ax.get_yticklabels():
        lab.set_color(FAM_COLOR.get(rows.home.get(lab.get_text(), "shared"), INK))
    for b in np.cumsum([sum(1 for f, _ in order if f == fam) for fam in FAMS])[:-1]:
        ax.axvline(b, color=INK, lw=0.9)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.9); sp.set_edgecolor(INK)
    save(fig, "S6", f"pg{i}", f"all_signatures_page{i}", PRIM)
    save_colorbar("S6", f"k1_{i}", f"colourbar_{i}", DIVERGING, -VMAX, VMAX, "Posterior mean (SD per\n5 kg/m² higher BMI)", 33, 58)
    save_key("S6", f"k2_{i}", f"key_{i}", [Line2D([0], [0], marker="o", color=INK, mfc="white", ls="None", ms=6, mew=0.8),
             Patch(facecolor=EMPTY_CELL, edgecolor="#9A9DA2"), Patch(color=BLUE), Patch(color=MAGENTA), Patch(color=TEAL), Patch(color=INK)],
             ["P(>0) ≥ 0.95\nor ≤ 0.05", "Not scored", "Coarse", "Fine", "Non-immune", "Shared"], 33, 50, ncol=1)
figs = L["figures"]
order_keys = [k for k in figs if not k.startswith("Figure S")] + sorted([k for k in figs if k.startswith("Figure S")],
             key=lambda k: (int(k.split("S")[1].split(" ")[0]), k))
L["figures"] = {k: figs[k] for k in order_keys}
(OUT / "layout.json").write_text(json.dumps(L, indent=2, ensure_ascii=False)); (OUT / "captions.json").write_text(json.dumps(CAPS, indent=2, ensure_ascii=False))
set_value(n_signatures=len(ordered), s6_pages=len(pages))
print(len(ordered), "signatures on", len(pages), "pages")


In [ ]:
from fig_common import *
PF, PC = load_primary()
order = celltype_order(); COLS = [ct for _, ct in order]
JOBL = [("BMI prior ×2", lambda f: "bmi_wide"), ("BMI prior ×0.5", lambda f: "bmi_narrow"),
        ("Intercept prior ×2", lambda f: INT_WIDE[f]), ("Intercept prior ×0.5", lambda f: "int_narrow"),
        ("All profiles", lambda f: ALLREF[f]), ("Read scale, ≥1,000", lambda f: "readscale_lowread"), ("Read scale, all", lambda f: RS_ALL[f])]
def feat(f, job):
    t = pd.read_csv(ROOT / f"{f}__{job}" / "feature_summary.csv"); return t[t.estimand.eq("per_5kg_m2")]
def ctab(f, job):
    t = pd.read_csv(ROOT / f"{f}__{job}" / "celltype_summary.csv"); return t[t.estimand.eq("per_5kg_m2")].set_index("celltype")

S = SLOT["Figure S7"]
R = np.zeros((len(JOBL), 3)); W = np.zeros_like(R); dP = np.zeros((len(JOBL), len(COLS)))
for i, (_, jf) in enumerate(JOBL):
    for j, f in enumerate(FAMS):
        m = feat(f, jf(f)).merge(PF[PF.family.eq(f)], on="feature", suffixes=("", "_p"))
        R[i, j] = np.corrcoef(m["mean"], m["mean_p"])[0, 1]
        W[i, j] = np.median((m.hdi95_high - m.hdi95_low) / (m.hdi95_high_p - m.hdi95_low_p))
    for k, (f, ct) in enumerate(order):
        dP[i, k] = ctab(f, jf(f)).loc[ct, "p_positive"] - PC.set_index("celltype").loc[ct, "p_positive"]
for slot, mat, cmap, vmin, vmax in [("p01", R, mpl.colors.LinearSegmentedColormap.from_list("w_b", ["#FFFFFF", BLUE]), 0.5, 1.0),
                                    ("p02", W, DIVERGING, 0.8, 1.2)]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=36 if slot == "p01" else 4, bottom=8, right=2, top=2)
    heat(ax, mat, vmax, [j for j, _ in JOBL] if slot == "p01" else [""] * len(JOBL), ["Coarse", "Fine", "Non-imm."],
         cmap=cmap, vmin=vmin, text_fmt="{:.2f}", col_rotation=0)
    save(fig, "S7", slot, "r_vs_primary" if slot == "p01" else "width_ratio", ROOT)
lim = max(0.1, float(np.ceil(np.abs(dP).max() * 20) / 20))
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=36, bottom=35, right=2, top=2)
heat(ax, dP, lim, [j for j, _ in JOBL], [ct_label(c_, FAM_PREFIX[f]) for f, c_ in order],
     col_breaks=np.cumsum([sum(1 for f, _ in order if f == fam) for fam in FAMS])[:-1])
save(fig, "S7", "p03", "celltype_delta_p", ROOT)
save_colorbar("S7", "k01", "delta_p_colourbar", DIVERGING, -lim, lim, "ΔP(effect > 0), sensitivity − primary", S["k01"]["w"], S["k01"]["h"], vertical=False)
mm = pd.concat([ctab(f, jf(f))[["mean"]].reset_index().assign(f=f) for _, jf in JOBL for f in FAMS]).merge(PC[["celltype", "mean"]], on="celltype", suffixes=("", "_p"))
set_value(max_sens_celltype_shift=round(float((mm["mean"] - mm["mean_p"]).abs().max()), 3))

# ---------------- S8
S = SLOT["Figure S8"]; y = -block_positions(order, gap=0.8)
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=31, bottom=14, right=3, top=2)
zebra(ax, y); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
for job_of, colour, dy in [(lambda f: "readscale_lowread", INK, 0.17), (lambda f: RS_ALL[f], MID_GREY, -0.17)]:
    g = pd.concat([pd.read_csv(ROOT / f"{f}__{job_of(f)}" / "read_scale_slope.csv") for f in FAMS]).set_index("celltype").reindex(COLS)
    ax.hlines(y + dy, g.hdi95_low, g.hdi95_high, color=colour, lw=1.4, zorder=2)
    ax.scatter(g["mean"], y + dy, s=22, color=colour, edgecolor="white", lw=0.6, zorder=4)
ax.set_yticks(y, [ct_label(c, FAM_PREFIX[f]) for f, c in order], fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
ax.set_xlabel("Change in log residual scale\nper SD of log allocated reads"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S8", "p01", "read_scale_coefficients", ROOT)
s8 = pd.read_csv(PPCR / "S8_readscale_ppc_vs_reference.csv"); s8 = s8[s8.domain.eq("studentized_residual_tails")]
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=17, bottom=12, right=3, top=2)
for k, f in enumerate(FAMS):
    for prof, mk, dx in [("reads >= 1,000", "o", -0.18), ("all profiles", "s", 0.18)]:
        g = s8[(s8.family == f) & (s8.profiles == prof)].set_index("model").upper_p
        ax.scatter([k + dx - 0.06], [g["reference"]], marker=mk, s=26, facecolor="white", edgecolor=FAM_COLOR[f], lw=1.2, zorder=4)
        ax.scatter([k + dx + 0.06], [g["read-dependent scale"]], marker=mk, s=26, color=FAM_COLOR[f], edgecolor="white", lw=0.6, zorder=4)
ax.axhline(0.025, color=INK, ls="--", lw=0.9)
ax.set_xticks(range(3), ["Coarse", "Fine", "Non-immune"]); ax.set_xlim(-0.6, 2.6); ax.set_ylim(0, None)
ax.set_ylabel("Tail probability"); style_axis(ax, "y")
save(fig, "S8", "p02", "tail_check", PPCR)
save_key("S8", "k01", "key", [Line2D([], [], color=INK, lw=1.4, marker="o", ms=4.5, mec="white"), Line2D([], [], color=MID_GREY, lw=1.4, marker="o", ms=4.5, mec="white"),
         Line2D([], [], marker="o", mfc="white", mec=INK, ls="", ms=6), Line2D([], [], marker="o", color=INK, ls="", ms=6, mec="white"),
         Line2D([], [], marker="o", mfc="white", mec=INK, ls="", ms=6), Line2D([], [], marker="s", mfc="white", mec=INK, ls="", ms=6)],
         ["a: fitted on ≥1,000 reads", "a: fitted on all profiles", "b: reference model (open)", "b: read-dependent scale (filled)",
          "b: ≥1,000 reads (circle)", "b: all profiles (square)"], S["k01"]["w"], S["k01"]["h"], ncol=1)
set_value(rs_fine_all_rhat=round(float(json.loads((ROOT / f"immune_fine__{RS_ALL['immune_fine']}" / "diagnostics.json").read_text())["max_rhat"]), 3))
for slot, f in zip(["p03", "p04", "p05"], FAMS):
    m = feat(f, ALLREF[f]).merge(PF[PF.family.eq(f)], on="feature", suffixes=("_all", "_p"))
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=17, bottom=14, right=3, top=3)
    lo, hi = float(min(m.mean_all.min(), m.mean_p.min())), float(max(m.mean_all.max(), m.mean_p.max())); pad = 0.06 * (hi - lo)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=INK, ls="--", lw=0.9, zorder=1)
    points(ax, m.mean_p, m.mean_all, dir_colors(m.mean_all), strong(m.p_positive_all), size=12)
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad); ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Primary (SD)"); ax.set_ylabel("All profiles (SD)")
    ax.text(0.04, 0.96, f"r = {np.corrcoef(m.mean_p, m.mean_all)[0, 1]:.2f}", transform=ax.transAxes, ha="left", va="top")
    save(fig, "S8", slot, f"allprofiles_vs_primary_{f}", ROOT)


In [ ]:
from fig_common import *
PF, PC = load_primary()
order = celltype_order(); COLS = [ct for _, ct in order]; y = -block_positions(order, gap=0.8)

# ---------------- S9
S = SLOT["Figure S9"]
cat = pd.concat([pd.read_csv(ROOT / f"{f}__categorical_all" / "celltype_summary.csv") for f in FAMS])
span = symmetric_span(cat.hdi95_low, cat.hdi95_high)
for slot, est, lab in [("p01", "overweight_vs_normal", "Overweight vs normal (SD)"), ("p02", "obese_vs_normal", "Obese vs normal (SD)"),
                       ("p03", "obese_vs_overweight", "Obese vs overweight (SD)")]:
    o = cat[cat.estimand.eq(est)].set_index("celltype").reindex(COLS)
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=31 if slot == "p01" else 3, bottom=14, right=3, top=2)
    zebra(ax, y); ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
    forest(ax, y, o["mean"], o.hdi95_low, o.hdi95_high, o.p_positive, lw=1.6)
    ax.set_xlim(-span, span); ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4, symmetric=True))
    ax.set_yticks(y, [ct_label(c, FAM_PREFIX[f]) for f, c in order] if slot == "p01" else [""] * len(y), fontsize=FONT - 1)
    ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(lab); style_axis(ax, "x"); row_axis(ax)
    save(fig, "S9", slot, est, ROOT)
set_value(n_cat_celltype_excl=int(cat.excludes_zero.sum()))
h = evidence_handles()
save_key("S9", "k01", "key", h, [x.get_label() for x in h], S["k01"]["w"], S["k01"]["h"], ncol=2)

# ---------------- S10
S = SLOT["Figure S10"]
s10 = pd.read_csv(PPCR / "S10_categorical_ppc.csv"); s10 = s10[s10.domain.eq("studentized_residual_tails")]
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=17, bottom=12, right=3, top=2)
for k, f in enumerate(FAMS):
    for prof, mk, dx in [("all profiles", "o", -0.14), ("reads >= 1,000", "s", 0.14)]:
        ax.scatter([k + dx], [s10[(s10.family == f) & (s10.profiles == prof)].upper_p.iloc[0]], marker=mk, s=30,
                   color=FAM_COLOR[f], edgecolor="white", lw=0.6, zorder=4)
ax.axhline(0.025, color=INK, ls="--", lw=0.9)
ax.set_xticks(range(3), ["Coarse", "Fine", "Non-immune"]); ax.set_xlim(-0.6, 2.6); ax.set_ylim(0, None)
ax.set_ylabel("Tail probability"); style_axis(ax, "y")
save(fig, "S10", "p01", "group_tail_check", PPCR)
CON = [("overweight_vs_normal", "Overweight\nvs normal"), ("obese_vs_normal", "Obese\nvs normal"), ("obese_vs_overweight", "Obese vs\noverweight")]
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=17, bottom=14, right=3, top=2)
for k, f in enumerate(FAMS):
    cont = pd.read_csv(ROOT / f"{f}__{ALLREF[f]}" / "feature_summary.csv").query("estimand == 'per_5kg_m2'")
    gfe = pd.read_csv(ROOT / f"{f}__categorical_all" / "feature_summary.csv")
    v = []
    for est, _ in CON:
        m = gfe[gfe.estimand.eq(est)].merge(cont[["feature", "mean"]], on="feature", suffixes=("_g", "_c"))
        v.append(np.corrcoef(m.mean_g, m.mean_c)[0, 1])
    ax.bar(np.arange(3) + (k - 1) * 0.26, v, width=0.24, color=FAM_COLOR[f], zorder=2)
ax.axhline(0, color=INK, lw=0.9); ax.set_xticks(range(3), [l for _, l in CON], fontsize=FONT - 1)
ax.set_ylabel("Pearson r with continuous"); style_axis(ax, "y")
save(fig, "S10", "p02", "group_continuous_agreement_allprofiles", ROOT)
pat = pd.read_csv(ROOT / "immune_coarse__categorical_all" / "ppc" / "patient_diagnostics.csv")
GC = {"normal_weight": LIGHT_BLUE, "overweight": LIGHT_MAGENTA, "obese": MAGENTA}
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=17, bottom=13, right=3, top=2)
for g, colour in GC.items():
    d = pat[pat.bmi_group.eq(g)]; ax.scatter(d.bmi_z, d.leverage, s=16, color=colour, edgecolor=INK if g == "obese" else "none", lw=0.3, zorder=3)
ax.axhline(pat.leverage_threshold.iloc[0], color=INK, ls="--", lw=0.9)
for v in (-2, 2): ax.axvline(v, color=INK, ls="--", lw=0.9)
ax.set_xlabel("BMI (z)"); ax.set_ylabel("Leverage"); style_axis(ax, None)
save(fig, "S10", "p03", "leverage", ROOT)
ext = pd.read_csv(PPCR / "categorical_residual_extremes.csv"); ext = ext[ext.profiles.eq("all profiles") & ext.exceeds_threshold]
top = ext.groupby(["patient", "bmi_group"]).size().sort_values(ascending=False).head(10).reset_index(name="n")
fig, ax = panel(S["p04"]["w"], S["p04"]["h"], left=22, bottom=13, right=3, top=2)
yy = -np.arange(len(top)); zebra(ax, yy)
ax.barh(yy, top.n, color=[GC.get(g, MID_GREY) for g in top.bmi_group], height=0.65, zorder=2)
ax.set_yticks(yy, top.patient, fontsize=FONT - 1); ax.set_ylim(yy.min() - 0.6, 0.6)
ax.set_xlabel("Tail-flagged features"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S10", "p04", "residual_extremes", PPCR)
save_key("S10", "k01", "key", [Line2D([], [], marker=m, color=INK, ls="", ms=6, mec="white") for m in ("o", "s")]
         + [Patch(color=FAM_COLOR[f]) for f in FAMS] + [Patch(color=c) for c in GC.values()],
         ["a: all profiles", "a: ≥1,000 reads"] + [FAM_LABEL[f] for f in FAMS] + ["Normal weight", "Overweight", "Obese"],
         S["k01"]["w"], S["k01"]["h"], ncol=4)

# ---------------- S12
S = SLOT["Figure S12"]
summ = pd.read_csv(PPCR / "screen_lfsr_svalue_summary.csv")
LAB = {("cellular RNA (Student-t)", "BMI per 5 kg/m2"): "Cellular RNA", ("PTM (paired protein)", "PTM_given_Protein"): "PTM given protein",
       ("PTM (paired protein)", "PTM"): "PTM", ("PTM (paired protein)", "Protein"): "Protein (PTM pairs)",
       ("RNA-protein (bulk)", "Protein_minus_RNA_BMI_per5"): "Protein − RNA", ("RNA-protein (bulk)", "Protein_BMI_per5"): "Protein (bulk)",
       ("RNA-protein (bulk)", "RNA_BMI_per5"): "RNA (bulk)"}
summ["label"] = [LAB.get((a, b), b) for a, b in zip(summ.layer, summ.quantity)]; summ["prop"] = summ.lfsr_lt_005 / summ.n
ptm_all = pd.read_csv(BASE / "multiomics_extensions/protein_ptm_paired_v2/publication/2687095c8b4e1e44/shards_4/aggregate/all_effects_95HDI.csv")
ptm_all = ptm_all[ptm_all.variant.eq("primary") & ptm_all.quantity.eq("PTM_given_Protein") & ptm_all.term.astype(str).str.lower().str.contains("bmi")]
set_value(n_ptm_all=int(ptm_all.key.nunique()), n_ptm_pass=int(summ[summ.label.eq("PTM given protein")].n.iloc[0]))
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=36, bottom=13, right=20, top=2)
yy = -np.arange(len(summ)); zebra(ax, yy)
cl = [TEAL if "RNA-protein" in l else MAGENTA if "PTM" in l else BLUE for l in summ.layer]
ax.barh(yy, summ.prop, color=cl, height=0.6, zorder=2)
right_column(ax, yy, [f"{a}/{b:,}" for a, b in zip(summ.lfsr_lt_005, summ.n)])
ax.axvline(0.10, color=INK, ls="--", lw=0.9)
ax.set_yticks(yy, summ.label, fontsize=FONT - 1); ax.set_ylim(yy.min() - 0.6, 0.6)
ax.set_xlabel("Proportion with lfsr < 0.05"); ax.set_xlim(0, max(0.2, float(summ.prop.max()) * 1.15)); style_axis(ax, "x"); row_axis(ax)
save(fig, "S12", "p01", "lfsr_proportions", PPCR)
feat = pd.read_csv(PPCR / "screen_lfsr_svalue_features.csv", low_memory=False)
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=15, bottom=13, right=3, top=2)
for (layer, q), colour in [(("PTM (paired protein)", "PTM_given_Protein"), MAGENTA), (("RNA-protein (bulk)", "Protein_minus_RNA_BMI_per5"), TEAL)]:
    col = "probability_positive" if "PTM" in layer else "p_positive"
    ax.hist(feat[(feat.layer == layer) & (feat.quantity == q)][col].dropna(), bins=np.linspace(0, 1, 21), density=True, histtype="step", color=colour, lw=1.5)
ax.axhline(1, color=INK, ls="--", lw=0.9); ax.set_xlim(0, 1); ax.set_xlabel("P(effect > 0)"); ax.set_ylabel("Density"); style_axis(ax, "y")
save(fig, "S12", "p02", "p_positive_distributions", PPCR)
save_key("S12", "k01", "key", [Patch(color=BLUE), Patch(color=MAGENTA), Patch(color=TEAL), Line2D([], [], color=MAGENTA, lw=1.5),
         Line2D([], [], color=TEAL, lw=1.5), Line2D([], [], color=INK, ls="--", lw=0.9)],
         ["Cellular RNA", "PTM layer", "RNA–protein layer", "b: PTM given protein", "b: Protein − RNA", "Nominal reference"],
         S["k01"]["w"], S["k01"]["h"], ncol=3)


In [ ]:
from fig_common import *
from scipy.stats import gaussian_kde
S = SLOT["Figure 5"]
E = rp_effects(); LAB, NSRC = programme_labels()
W = E.pivot(index="programme", columns="quantity", values="posterior_mean")
X = E.set_index(["programme", "quantity"])
rho = W.RNA_BMI_per5.rank().corr(W.Protein_BMI_per5.rank())
set_value(n_rp=int(len(W)), rp_rho=f"{rho:.2f}")

w, h = S["p01"]["w"], S["p01"]["h"]; fig = plt.figure(figsize=(w * MM, h * MM))
L, B, mw, mh = 17, 14, w - 17 - 16, h - 14 - 16
ax = fig.add_axes([L / w, B / h, mw / w, mh / h]); axt = fig.add_axes([L / w, (B + mh + 1) / h, mw / w, 13 / h])
axr = fig.add_axes([(L + mw + 1) / w, B / h, 13 / w, mh / h])
exR = X.loc[[(p, "RNA_BMI_per5") for p in W.index], "excl"].to_numpy(); exP = X.loc[[(p, "Protein_BMI_per5") for p in W.index], "excl"].to_numpy()
both, one = exR & exP, exR ^ exP
lim = symmetric_span(W.RNA_BMI_per5, W.Protein_BMI_per5)
ax.axhline(0, color=INK, lw=0.6, zorder=1); ax.axvline(0, color=INK, lw=0.6, zorder=1)
ax.scatter(W.RNA_BMI_per5[~both & ~one], W.Protein_BMI_per5[~both & ~one], s=10, color="#B8BBC1", lw=0, zorder=2)
ax.scatter(W.RNA_BMI_per5[one], W.Protein_BMI_per5[one], s=14, color=INK, lw=0, zorder=3)
ax.scatter(W.RNA_BMI_per5[both], W.Protein_BMI_per5[both], s=18, color=MAGENTA, edgecolor="white", lw=0.4, zorder=4)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("RNA slope\n(SD per 5 kg/m²)"); ax.set_ylabel("Protein slope (SD per 5 kg/m²)"); style_axis(ax, None)
xs = np.linspace(-lim, lim, 200)
axt.fill_between(xs, gaussian_kde(W.RNA_BMI_per5)(xs), color=LIGHT_BLUE, alpha=0.7, lw=0); axt.plot(xs, gaussian_kde(W.RNA_BMI_per5)(xs), color=BLUE, lw=0.9)
axr.fill_betweenx(xs, gaussian_kde(W.Protein_BMI_per5)(xs), color=LIGHT_MAGENTA, alpha=0.7, lw=0); axr.plot(gaussian_kde(W.Protein_BMI_per5)(xs), xs, color=MAGENTA, lw=0.9)
for a_ in (axt, axr):
    a_.set_axis_off()
axt.set_xlim(-lim, lim); axr.set_ylim(-lim, lim); axt.set_ylim(0, None); axr.set_xlim(0, None)
save(fig, "F5", "p01", "rna_vs_protein", R_DIR / "results/all_effects_95HDI.csv")
save_key("F5", "k03", "key_a", [Line2D([], [], marker="o", ls="", color=MAGENTA, mec="white", ms=5), Line2D([], [], marker="o", ls="", color=INK, ms=4),
                                Line2D([], [], marker="o", ls="", color="#B8BBC1", ms=4)],
         ["95% HDI excludes 0 for RNA and protein", "Excludes 0 in one of them", "Neither"], S["k03"]["w"], S["k03"]["h"], ncol=1)

CO = rd(R_DIR / "bootstrap" / "concordance_95CI.csv")
CO = CO[CO.stratum.eq("all") & CO.metric.eq("correlation_of_estimated_slopes") & CO.scope.isin(["bulk", "family"])]
rows = []
for f in FAMS:
    for c_, lab, col in [("celltype_vs_RNA", "cell type and bulk RNA", BLUE), ("celltype_vs_Protein", "cell type and protein", MAGENTA)]:
        r = CO[CO.scope.eq("family") & CO.family.eq(f) & CO.comparison.eq(c_)]
        if len(r): rows.append((f"{FAM_LABEL[f]}: {lab}", r.iloc[0], col))
r = CO[CO.scope.eq("bulk") & CO.comparison.eq("RNA_vs_Protein")]
if len(r): rows.append(("Bulk RNA and protein", r.iloc[0], MAGENTA)); set_value(rp_bench_r=f"{r.estimate.iloc[0]:.2f}")
y = -np.arange(len(rows))
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=58, bottom=13, right=3, top=2); zebra(ax, y)
for yy, (lab, r_, col) in zip(y, rows):
    ax.hlines(yy, r_.ci_95_lower, r_.ci_95_upper, color=col, lw=SIZE["interval_line"], zorder=2)
    ax.scatter([r_.estimate], [yy], s=SIZE["marker"], color=col, edgecolor="white", lw=0.7, zorder=4)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_xlim(-0.2, 1.0)
ax.set_yticks(y, [l for l, _, _ in rows], fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
ax.set_xlabel("Correlation of slopes (95% bootstrap CI)"); style_axis(ax, "x"); row_axis(ax)
save(fig, "F5", "p02", "benchmark_correlations", R_DIR / "bootstrap/concordance_95CI.csv")
save_key("F5", "k01", "key_b", [Line2D([], [], marker="o", color=BLUE, lw=1.6, mec="white", ms=6), Line2D([], [], marker="o", color=MAGENTA, lw=1.6, mec="white", ms=6)],
         ["Cell type and bulk RNA (same RNA assay)", "Comparisons involving protein"], S["k01"]["w"], S["k01"]["h"], ncol=1)

T = rd(R_DIR / "inputs_used" / "sensitivity_targets.csv")
T["label"] = [f"{FAM_PREFIX[f]}: {ct_label(c)} · {sig_label(s)}" for f, c, s in zip(T.family, T.CellType, T.Signature)]
T = T.drop_duplicates("programme").merge(W[["Protein_minus_RNA_BMI_per5"]], left_on="programme", right_index=True).sort_values("Protein_minus_RNA_BMI_per5")
y = -np.arange(len(T))
fig, axes = shared_row_axes(S["p03"]["w"], S["p03"]["h"], left=76, n=2, gap=8, bottom=13)
for k, (ax, q, xl) in enumerate(zip(axes, ["Protein_minus_RNA_BMI_per5", "residual_RNA_Protein_correlation"],
                                     ["Protein − RNA slope (SD per 5 kg/m²)", "Residual RNA–protein correlation"])):
    zebra(ax, y); v = X.loc[[(p, q) for p in T.programme]]
    hdi_forest(ax, y, v.posterior_mean, v.hdi_95_lower, v.hdi_95_upper, v.excl, size=26, lw=1.6)
    ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
    ax.set_yticks(y, T.label if k == 0 else [""] * len(y), fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
    ax.set_xlabel(xl); style_axis(ax, "x"); row_axis(ax)
save(fig, "F5", "p03", "targets_contrast_and_residual_correlation", R_DIR / "results/all_effects_95HDI.csv")

top = W.assign(m=np.maximum(W.RNA_BMI_per5.abs(), W.Protein_BMI_per5.abs())).nlargest(16, "m").sort_values("Protein_minus_RNA_BMI_per5")
y = -np.arange(len(top))
fig, axes = shared_row_axes(S["p04"]["w"], S["p04"]["h"], left=86, n=3, gap=5, bottom=13)
for k, (ax, q, xl) in enumerate(zip(axes, ["RNA_BMI_per5", "Protein_BMI_per5", "Protein_minus_RNA_BMI_per5"], ["RNA slope", "Protein slope", "Protein − RNA"])):
    zebra(ax, y); v = X.loc[[(p, q) for p in top.index]]
    hdi_forest(ax, y, v.posterior_mean, v.hdi_95_lower, v.hdi_95_upper, v.excl, size=24, lw=1.5)
    ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); span = symmetric_span(v.hdi_95_lower, v.hdi_95_upper)
    ax.set_xlim(-span, span); ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3, symmetric=True))
    ax.set_yticks(y, [LAB.get(p, p) for p in top.index] if k == 0 else [""] * len(y), fontsize=FONT - 1); ax.set_ylim(y.min() - 0.6, 0.6)
    ax.set_xlabel(f"{xl} (SD)"); style_axis(ax, "x"); row_axis(ax)
save(fig, "F5", "p04", "top16_gene_sets", R_DIR / "results/all_effects_95HDI.csv")
top.assign(label=[LAB.get(p, p) for p in top.index], n_source_features=[NSRC.get(p, 1) for p in top.index]).to_csv(TABLES / "Fig5d_gene_sets.csv")
h = hdi_handles()
save_key("F5", "k02", "key_cd", h, [x.get_label() for x in h], S["k02"]["w"], S["k02"]["h"], ncol=3)


In [ ]:
from fig_common import *
S = SLOT["Figure 6"]
MC = rd(DA_DIR / "marginal_conditional_feature_comparison.csv")
MC["excl"] = tb(MC.hdi_excludes_zero)
TR = rd(DA_DIR / "interval_transition_counts.csv")
for a in ("phospho", "glyco"):
    t = TR[TR.assay.eq(a)]; mc = MC[MC.assay.eq(a)]
    marg = t[~t.marginal_direction.eq("uncertain")].features.sum()
    ret = t[~t.marginal_direction.eq("uncertain") & ~t.direction.eq("uncertain")].features.sum()
    add = t[t.marginal_direction.eq("uncertain") & ~t.direction.eq("uncertain")].features.sum()
    flips = t[(t.marginal_direction.eq("positive") & t.direction.eq("negative")) | (t.marginal_direction.eq("negative") & t.direction.eq("positive"))].features.sum()
    set_value(**{f"n_{a}_adequate": int(len(mc)), f"{a}_marg_cand": int(marg), f"{a}_retained": int(ret), f"{a}_added": int(add),
                 f"{a}_cand": int(ret + add), f"{a}_sign_flips": int(flips)})

for slot, a in [("p01", "phospho"), ("p02", "glyco")]:
    m = MC[MC.assay.eq(a)]
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=15, bottom=13, right=2, top=2)
    lim = symmetric_span(m.posterior_mean_marginal, m.posterior_mean_conditional, pad=1.05)
    ax.axhline(0, color=INK, lw=0.7, zorder=1); ax.axvline(0, color=INK, lw=0.7, zorder=1)
    ax.plot([-lim, lim], [-lim, lim], color=MID_GREY, ls="--", lw=0.8, zorder=1)
    o = m[~m.excl]; c = m[m.excl]
    ax.scatter(o.posterior_mean_marginal, o.posterior_mean_conditional, s=7, color=BAND_GREY, edgecolor=MID_GREY, lw=0.3, zorder=2)
    ax.scatter(c.posterior_mean_marginal, c.posterior_mean_conditional, s=14, color=dir_colors(c.posterior_mean_conditional),
               edgecolor="white", lw=0.4, zorder=3)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal", adjustable="box")
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3, symmetric=True)); ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(3, symmetric=True))
    ax.set_xlabel("Marginal slope"); ax.set_ylabel("Conditional slope" if slot == "p01" else ""); style_axis(ax, None)
    save(fig, "F6", slot, f"{a}_marginal_vs_conditional", DA_DIR / "marginal_conditional_feature_comparison.csv")
save_key("F6", "k01", "scatter_key",
         [Line2D([], [], marker="o", ls="", color=MAGENTA, mec="white", ms=6), Line2D([], [], marker="o", ls="", color=BLUE, mec="white", ms=6),
          Line2D([], [], marker="o", ls="", color=BAND_GREY, mec=MID_GREY, ms=5), Line2D([], [], color=MID_GREY, ls="--", lw=0.8)],
         ["Conditional HDI excludes zero, positive", "Conditional HDI excludes zero, negative", "Conditional HDI includes zero", "Identity"],
         S["k01"]["w"], S["k01"]["h"], ncol=2)

fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=24, bottom=13, right=8, top=2)
rows, yv = [], []
for k, a in enumerate(["phospho", "glyco"]):
    t = TR[TR.assay.eq(a)]
    vals = [("Retained", t[~t.marginal_direction.eq("uncertain") & ~t.direction.eq("uncertain")].features.sum(), INK),
            ("Lost", t[~t.marginal_direction.eq("uncertain") & t.direction.eq("uncertain")].features.sum(), MID_GREY),
            ("Added", t[t.marginal_direction.eq("uncertain") & ~t.direction.eq("uncertain")].features.sum(), MAGENTA)]
    for j, (lab, v, col) in enumerate(vals):
        yy = -(k * 4.0 + j); ax.barh(yy, v, height=0.7, color=col, zorder=2)
        ax.text(v, yy, f" {int(v)}", va="center", ha="left", fontsize=SIZE["value"]); rows.append(lab); yv.append(yy)
    ax.text(-0.95, -(k * 4.0 + 1), ASSAY_LABEL[a], transform=ax.get_yaxis_transform(), rotation=90, ha="center", va="center",
            fontsize=SIZE["value"], color=ASSAY_COLOR[a])
ax.set_yticks(yv, rows); ax.set_xlim(0, max(ax.get_xlim()[1] * 1.18, 10)); ax.set_xlabel("Candidates (95% HDI\nexcludes zero)")
style_axis(ax, "x"); row_axis(ax)
save(fig, "F6", "p03", "interval_transitions", DA_DIR / "interval_transition_counts.csv")

for slot, a, n in [("p04", "phospho", None), ("p05", "glyco", 20)]:
    c = MC[MC.assay.eq(a) & MC.excl].copy()
    c = c.reindex(c.posterior_mean_conditional.abs().sort_values(ascending=False).index)
    if n: c = c.head(n)
    c = c.sort_values("posterior_mean_conditional"); y = -np.arange(len(c))
    lab = [site_label(g, f) for g, f in zip(c.gene, c.feature_id)]
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=38, bottom=13, right=3, top=2); zebra(ax, y)
    ax.hlines(y + 0.22, c.hdi_95_lower_marginal, c.hdi_95_upper_marginal, color=LINE, lw=1.2, zorder=2)
    ax.scatter(c.posterior_mean_marginal, y + 0.22, s=16, marker="s", color=MID_GREY, edgecolor="white", lw=0.4, zorder=3)
    hdi_forest(ax, y - 0.12, c.posterior_mean_conditional, c.hdi_95_lower_conditional, c.hdi_95_upper_conditional, c.excl, size=24, lw=1.6)
    ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
    ax.set_yticks(y, lab, fontsize=FONT - 1); ax.set_ylim(y.min() - 0.7, 0.7); ax.set_xlabel(PTM_UNIT); style_axis(ax, "x"); row_axis(ax)
    ax.set_ylabel("Phosphosite (gene, site)" if a == "phospho" else "Glycoform (gene, glycan)", color=ASSAY_COLOR[a])
    save(fig, "F6", slot, f"{a}_candidates", DA_DIR / "marginal_conditional_feature_comparison.csv")
    c.assign(label=lab).to_csv(TABLES / f"Fig6_{a}_candidates.csv", index=False)
save_key("F6", "k02", "candidate_key",
         [Line2D([], [], marker="s", color=MID_GREY, mec="white", lw=1.2, ms=5), *hdi_handles()[:2]],
         ["Marginal slope, 95% HDI", "Conditional, positive", "Conditional, negative"], S["k02"]["w"], S["k02"]["h"], ncol=3)


In [ ]:
from fig_common import *
S = SLOT["Figure 7"]
A = ptm_primary(); A = A[A.adequate]
AC = {"phospho": BLUE, "glyco": MAGENTA}

fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=15, bottom=13, right=2, top=3)
for k, a in enumerate(["phospho", "glyco"]):
    g = A[A.assay.eq(a)]; obs, exp = int(g.excl.sum()), 0.05 * len(g)
    ax.bar(k - 0.19, obs, width=0.36, color=AC[a], zorder=2)
    ax.bar(k + 0.19, exp, width=0.36, facecolor=mpl.colors.to_rgba(AC[a], 0.35), edgecolor=AC[a], hatch="////", lw=0.9, zorder=2)
    ax.text(k - 0.19, obs, f"{obs}", ha="center", va="bottom", fontsize=SIZE["value"])
    ax.text(k + 0.19, exp, f"{exp:.0f}", ha="center", va="bottom", fontsize=SIZE["value"], color=MID_GREY)
    set_value(**{f"{a}_obs_excl": obs, f"{a}_exp_excl": round(exp, 2), f"{a}_n_adequate": int(len(g))})
ax.set_xticks([0, 1], [f"Phosphosites\n(n = {int(A.assay.eq('phospho').sum())})", f"Glycoforms\n(n = {int(A.assay.eq('glyco').sum()):,})"])
ax.set_xlim(-0.6, 1.6); ax.set_ylim(0, ax.get_ylim()[1] * 1.1); ax.set_ylabel("Features with 95% HDI excluding 0"); style_axis(ax, "y")
save(fig, "F7", "p01", "observed_vs_expected", DA_DIR / "all_primary_features.csv")

fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=17, bottom=13, right=2, top=3)
for a in ["phospho", "glyco"]:
    v = np.sort(np.clip(A[A.assay.eq(a)].lfsr.to_numpy(), 1e-4, None)); q = 100 * (np.arange(1, len(v) + 1) - 0.5) / len(v)
    ax.plot(q, v, color=AC[a], lw=1.5, zorder=3)
for t_, ls in [(0.05, "--"), (0.01, ":")]:
    ax.axhline(t_, color=INK, ls=ls, lw=0.9, zorder=2); ax.text(99, t_ * 1.12, f"lfsr = {t_}", ha="right", va="bottom", fontsize=SIZE["value"] - 1, color=MID_GREY)
ax.set_yscale("log"); ax.set_ylim(1e-4, 0.7); ax.set_xlim(0, 100)
ax.set_xlabel("Feature percentile within assay\n(smallest lfsr first)"); ax.set_ylabel("lfsr = min(P(β>0), P(β<0))"); style_axis(ax, "y")
save(fig, "F7", "p02", "lfsr_percentile", DA_DIR / "all_primary_features.csv")

fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=17, bottom=13, right=2, top=3)
for lab, v, col, ls in [("phospho", A[A.assay.eq("phospho")].lfsr, AC["phospho"], "-"), ("glyco", A[A.assay.eq("glyco")].lfsr, AC["glyco"], "-"),
                        ("all", A.lfsr, TEAL, "--")]:
    v = np.sort(v.to_numpy()); n = np.arange(1, len(v) + 1); ax.plot(n, np.clip(np.cumsum(v) / n, 1e-4, None), color=col, lw=1.5, ls=ls, zorder=3)
ax.axhline(0.05, color=INK, ls="--", lw=0.9, zorder=2)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_ylim(1e-4, 0.7)
ax.set_xlabel("Features retained (ranked by lfsr)"); ax.set_ylabel("Mean lfsr among retained"); style_axis(ax, "y")
save(fig, "F7", "p03", "mean_lfsr_retained", DA_DIR / "all_primary_features.csv")
save_key("F7", "k01", "key_abc", [Patch(color=BLUE), Patch(color=MAGENTA), Line2D([], [], color=TEAL, ls="--", lw=1.5),
                                  Line2D([], [], color=INK, ls="--", lw=0.9), Patch(facecolor=mpl.colors.to_rgba(MID_GREY, 0.35), edgecolor=MID_GREY, hatch="////")],
         ["Phosphosites", "Glycoforms", "All adequate features (c)", "Reference 0.05", "Expected if all slopes null (5%)"], S["k01"]["w"], S["k01"]["h"], ncol=1)

D = rd(DA_DIR / "candidate_deletion_comparisons.csv").sort_values("gene_primary", ascending=False)
y = -np.arange(len(D)) * 1.0
fig, ax = panel(S["p04"]["w"], S["p04"]["h"], left=36, bottom=13, right=3, top=2)
for yy, r in zip(y, D.itertuples()):
    for dy, m, lo, hi, col in [(0.16, r.posterior_mean_primary, r.hdi_95_lower_primary, r.hdi_95_upper_primary, MID_GREY),
                               (-0.16, r.posterior_mean_deletion, r.hdi_95_lower_deletion, r.hdi_95_upper_deletion, TEAL)]:
        ax.hlines(yy + dy, lo, hi, color=col, lw=1.8, zorder=2); ax.scatter([m], [yy + dy], s=30, color=col, edgecolor="white", lw=0.6, zorder=4)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, [f"{g}\n(without {p})" for g, p in zip(D.gene_primary, D.deleted_patient)])
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel("PTM given protein (SD per 5 kg/m²)"); style_axis(ax, "x"); row_axis(ax)
save(fig, "F7", "p04", "deletion_refits", DA_DIR / "candidate_deletion_comparisons.csv")
save_key("F7", "k02", "key_d", [Line2D([], [], marker="o", color=MID_GREY, lw=1.8, mec="white", ms=6), Line2D([], [], marker="o", color=TEAL, lw=1.8, mec="white", ms=6)],
         ["Full cohort", "Without the named patient"], S["k02"]["w"], S["k02"]["h"], ncol=1)
changed = D[tb(D.interval_class_changed)]
set_value(deletion_genes=", ".join(sorted(D.gene_primary)), deletion_changed=", ".join(changed.gene_primary) or "none")


In [ ]:
from fig_common import *
S = SLOT["Figure S11"]
PF, PC = load_primary()
RF, RC, DG = {}, {}, {}
for f in FAMS:
    d = ROOT / f"{f}__randomgenes"
    RF[f] = pd.read_csv(d / "feature_summary.csv").query("estimand == 'per_5kg_m2'")
    RC[f] = pd.read_csv(d / "celltype_summary.csv").query("estimand == 'per_5kg_m2'")
    DG[f] = json.loads((d / "diagnostics.json").read_text())
lfsr = lambda p: np.minimum(p, 1 - p)
SH = {"immune_coarse": "coarse", "immune_fine": "fine", "nonimmune": "non"}
for f in FAMS:
    pf = PF[PF.family.eq(f)]
    set_value(**{f"real_lfsr_{SH[f]}": int((lfsr(pf.p_positive) < 0.05).sum()), f"rand_lfsr_{SH[f]}": int((lfsr(RF[f].p_positive) < 0.05).sum()),
                 f"real_pos_{SH[f]}": f"{(pf['mean'] > 0).mean():.2f}", f"rand_pos_{SH[f]}": f"{(RF[f]['mean'] > 0).mean():.2f}",
                 f"rand_gate_{SH[f]}": bool(DG[f]["gate_pass"])})
set_value(rand_fine_ess=int(min(DG["immune_fine"]["min_ess_bulk"], DG["immune_fine"]["min_ess_tail"])),
          rand_fine_rhat=f"{DG['immune_fine']['max_rhat']:.3f}")
REAL, RAND = INK, LIGHT_BLUE
xs = np.arange(3); lab3 = ["Immune\ncoarse", "Immune\nfine", "Non-\nimmune"]
for slot, fn, yl, fmt in [("p01", lambda df: (df["mean"] > 0).mean(), "Feature means above zero", "{:.2f}"),
                          ("p02", lambda df: int((lfsr(df.p_positive) < 0.05).sum()), "Features with lfsr < 0.05", "{}")]:
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=16, bottom=14, right=3, top=3)
    for k, f in enumerate(FAMS):
        for dx, df, col in [(-0.18, PF[PF.family.eq(f)], REAL), (0.18, RF[f], RAND)]:
            v = fn(df); ax.bar(k + dx, v, width=0.34, color=col, edgecolor=INK if col == RAND else col, lw=0.6, zorder=2)
            ax.text(k + dx, v, fmt.format(v), ha="center", va="bottom", fontsize=SIZE["value"] - 1)
    ax.set_xticks(xs, lab3); ax.set_ylabel(yl); ax.set_xlim(-0.6, 2.6)
    ax.set_ylim(0, 1.08 if slot == "p01" else max(ax.get_ylim()[1] * 1.15, 1)); style_axis(ax, "y")
    if slot == "p02": ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    save(fig, "S11", slot, "share_positive" if slot == "p01" else "lfsr_counts", ROOT)
save_key("S11", "k01", "key_ab", [Patch(facecolor=REAL), Patch(facecolor=RAND, edgecolor=INK, lw=0.6)],
         ["Real signatures (primary fit)", "Size-matched random gene sets"], S["k01"]["w"], S["k01"]["h"], ncol=2)

fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=17, bottom=14, right=3, top=2)
allv = []
for f in FAMS:
    m = RC[f].merge(PC[PC.family.eq(f)], on="celltype", suffixes=("_r", "_p"))
    ax.scatter(m.mean_p, m.mean_r, s=26, color=FAM_COLOR[f], edgecolor="white", lw=0.6, zorder=3); allv += [m.mean_p, m.mean_r]
lim = symmetric_span(*allv, pad=1.1)
ax.plot([-lim, lim], [-lim, lim], color=MID_GREY, ls="--", lw=0.8); ax.axhline(0, color=INK, lw=0.7); ax.axvline(0, color=INK, lw=0.7)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Real signatures (SD per 5 kg/m²)"); ax.set_ylabel("Random gene sets"); style_axis(ax, None)
save(fig, "S11", "p03", "celltype_real_vs_random", ROOT)
save_key("S11", "k02", "key_c", [Line2D([], [], marker="o", ls="", color=FAM_COLOR[f], mec="white", ms=6) for f in FAMS],
         [FAM_LABEL[f] for f in FAMS], S["k02"]["w"], S["k02"]["h"], ncol=3)

run = sorted(p for p in DRIVERS.glob("run_*") if (p / "logo_summary.csv").exists())[-1]
G = pd.read_csv(run / "logo_summary.csv").sort_values("full_slope"); y = -np.arange(len(G))
fig, ax = panel(S["p04"]["w"], S["p04"]["h"], left=34, bottom=14, right=20, top=2); zebra(ax, y)
ax.hlines(y, G.logo_min, G.logo_max, color=LIGHT_MAGENTA, lw=5, zorder=2)
ax.scatter(G.full_slope, y, s=SIZE["marker"], color=MAGENTA, edgecolor="white", lw=0.7, zorder=4)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_xlim(0, max(G.logo_max) * 1.15)
ax.set_yticks(y, [sig_label(s) for s in G.Signature]); ax.set_ylim(y.min() - 0.6, 0.6)
right_column(ax, y, list(G.most_influential_gene))
ax.set_xlabel("OLS slope (SD per 5 kg/m²)"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S11", "p04", "tam_gene_drivers", run)
save_key("S11", "k03", "key_d", [Line2D([], [], marker="o", ls="", color=MAGENTA, mec="white", ms=6), Line2D([], [], color=LIGHT_MAGENTA, lw=5)],
         ["All genes", "Range leaving out one gene"], S["k03"]["w"], S["k03"]["h"], ncol=1)
set_value(driver_run=run.name)


In [ ]:
from fig_common import *
LAB, NSRC = programme_labels(); E = rp_effects(); X = E.set_index(["programme", "quantity"])
COV = rd(R_DIR / "inputs_used" / "coverage.csv"); COV["inc"] = tb(COV.included)

# ---------------- S13
S = SLOT["Figure S13"]
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=24, bottom=13, right=10, top=2)
y = -np.arange(3)
for yy, f in zip(y, FAMS):
    g = COV[COV.family.eq(f)]; inc, exc = int(g.inc.sum()), int((~g.inc).sum())
    ax.barh(yy, inc, color=FAM_COLOR[f], height=0.6, zorder=2); ax.barh(yy, exc, left=inc, color=BAND_GREY, height=0.6, zorder=2)
    ax.text(inc / 2, yy, str(inc), ha="center", va="center", color="white", fontsize=SIZE["value"])
    ax.text(inc + exc, yy, f" {inc + exc}", ha="left", va="center", fontsize=SIZE["value"])
ax.set_yticks(y, [FAM_LABEL[f] for f in FAMS]); ax.set_xlabel("Cellular features"); ax.set_xlim(0, ax.get_xlim()[1] * 1.15)
style_axis(ax, "x"); row_axis(ax); save(fig, "S13", "p01", "eligibility", R_DIR / "inputs_used/coverage.csv")
set_value(n_rp_eligible=int(COV.inc.sum()), n_rp_features=int(len(COV)))

CO = rd(R_DIR / "bootstrap" / "concordance_95CI.csv")
CO = CO[CO.stratum.eq("all") & CO.metric.eq("sign_agreement") & CO.scope.isin(["bulk", "family"])]
BL = rd(R_DIR / "bootstrap" / "bulk_linear_effects_95CI.csv"); CT = rd(R_DIR / "bootstrap" / "celltype_and_matched_bulk_linear_95CI.csv")
def ppos(v): v = np.asarray(v, float); return float((v > 0).mean())
def baseline(comp, fam):
    if fam is None:
        pa = ppos(BL[BL.quantity.eq("RNA_BMI_per5")].estimate); pb = ppos(BL[BL.quantity.eq("Protein_BMI_per5")].estimate)
    else:
        g = CT[CT.family.eq(fam)]; q = {"celltype": "celltype_BMI_per5", "RNA": "paired_RNA_BMI_per5", "Protein": "paired_Protein_BMI_per5"}
        a, b = comp.split("_vs_"); pa, pb = ppos(g[g.quantity.eq(q[a])].estimate), ppos(g[g.quantity.eq(q[b])].estimate)
    return pa * pb + (1 - pa) * (1 - pb)
CMP = [("celltype_vs_RNA", "Cell-type RNA vs bulk RNA"), ("celltype_vs_Protein", "Cell-type RNA vs bulk protein"), ("RNA_vs_Protein", "Bulk RNA vs bulk protein")]
SHORT = {"celltype_vs_RNA": "Cell type vs RNA", "celltype_vs_Protein": "Cell type vs protein", "RNA_vs_Protein": "RNA vs protein"}
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=56, bottom=13, right=3, top=2)
yy, labs, k = [], [], 0.0
for c, cl in CMP:
    g = CO[CO.comparison.eq(c)]
    for scope, f in [("bulk", None)] + [("family", f) for f in FAMS]:
        r = g[g.scope.eq(scope)] if f is None else g[g.scope.eq(scope) & g.family.eq(f)]
        if not len(r): continue
        r = r.iloc[0]; col = INK if f is None else FAM_COLOR[f]
        ax.hlines(-k, r.ci_95_lower, r.ci_95_upper, color=col, lw=SIZE["interval_line"], zorder=2)
        ax.scatter([r.estimate], [-k], s=SIZE["marker"], color=col, edgecolor="white", lw=0.7, zorder=4, marker="D" if f is None else "o")
        ax.plot([baseline(c, f)] * 2, [-k - 0.3, -k + 0.3], color=INK, lw=1.4, zorder=5)
        yy.append(-k); labs.append(f"{SHORT[c]}, {'all' if f is None else FAM_LABEL[f].lower()}"); k += 1
    k += 0.5
yy = np.array(yy); zebra(ax, yy); ax.axvline(0.5, color=MID_GREY, ls=":", lw=0.8, zorder=1)
ax.set_yticks(yy, labs, fontsize=FONT - 1); ax.set_ylim(yy.min() - 0.7, 0.7); ax.set_xlim(0, 1)
ax.set_xlabel("Sign agreement of BMI slopes"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S13", "p02", "sign_agreement", R_DIR / "bootstrap/concordance_95CI.csv")
save_key("S13", "k01", "key_b", [Line2D([], [], marker="D", ls="", color=INK, mec="white", ms=6), Line2D([], [], marker="|", ls="", color=INK, ms=9, mew=1.4)],
         ["Estimate, patient-bootstrap 95% CI", "Expected from marginal signs"], S["k01"]["w"], S["k01"]["h"], ncol=2)

Z = np.load(R_DIR / "inputs_used" / "bulk_scores.npz", allow_pickle=True)
vals, progs, pats = Z["values"], [str(p) for p in Z["programme_ids"]], [str(p) for p in Z["patient_ids"]]
ax_p = [i for i, n in enumerate(vals.shape) if n == len(progs)][0]; ax_q = [i for i, n in enumerate(vals.shape) if n == len(pats) and i != ax_p][0]
ax_l = [i for i in range(vals.ndim) if i not in (ax_p, ax_q)][0]
V = np.moveaxis(vals, (ax_l, ax_p, ax_q), (0, 1, 2))
cov = rd(R_DIR / "inputs_used" / "patient_covariates.csv").set_index("base_sample_id").reindex(pats)
bmi = cov.bmi.to_numpy(float)
lin = BL[BL.quantity.eq("RNA_BMI_per5")].set_index("programme").estimate.reindex(progs).to_numpy(float)
def slope(Y): return np.array([np.polyfit(bmi[np.isfinite(y)], y[np.isfinite(y)], 1)[0] for y in Y])
rna_layer = int(np.argmax([np.corrcoef(slope(V[l]), lin)[0, 1] for l in range(V.shape[0])]))
prot_layer = 1 - rna_layer if V.shape[0] == 2 else [l for l in range(V.shape[0]) if l != rna_layer][0]
print(f"bulk_scores.npz: layer {rna_layer} = RNA, layer {prot_layer} = protein (matched by benchmark slopes)")
CMAP_BMI = mpl.colors.LinearSegmentedColormap.from_list("bmi", ["#DDE3EE", BLUE, MAGENTA])
def prog_of(fam, ct, sig):
    r = COV[COV.family.eq(fam) & COV.CellType.eq(ct) & COV.Signature.eq(sig)]
    return r.programme.iloc[0]
for slot, (fam, ct, sig) in [("p03", ("nonimmune", "TUMOR_EPITHELIAL", "ECM_Remodeling_Signature")),
                             ("p04", ("immune_fine", "CD8_T_EXHAUSTED", "Metabolic_Imbalance_Signature"))]:
    j = progs.index(prog_of(fam, ct, sig)); r_, p_ = V[rna_layer, j], V[prot_layer, j]
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=16, bottom=13, right=3, top=2)
    ok = np.isfinite(r_) & np.isfinite(p_)
    ax.scatter(r_[ok], p_[ok], c=bmi[ok], cmap=CMAP_BMI, vmin=18, vmax=40, s=18, edgecolor="white", lw=0.4, zorder=3)
    b = np.polyfit(r_[ok], p_[ok], 1); xs = np.linspace(np.nanmin(r_), np.nanmax(r_), 2)
    ax.plot(xs, b[1] + b[0] * xs, color=INK, lw=1.0, zorder=4)
    ax.set_xlabel("RNA score (SD)"); ax.set_ylabel("Protein score (SD)"); style_axis(ax, None)
    save(fig, "S13", slot, f"score_pairs_{ct.lower()}", R_DIR / "inputs_used/bulk_scores.npz")
save_colorbar("S13", "k02", "bmi_colorbar", CMAP_BMI, 18, 40, "BMI (kg/m²)", S["k02"]["w"], S["k02"]["h"], vertical=True)

# ---------------- S14
S = SLOT["Figure S14"]
RS = rd(R_DIR / "results" / "run_status.csv")
RS["cls"] = np.where(RS.variant.eq("primary"), "Primary", np.where(RS.variant.str.contains("without|delet", case=False), "Patient deletion", "Sensitivity"))
CLS = {"Primary": INK, "Sensitivity": BLUE, "Patient deletion": MAGENTA}
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=18, bottom=13, right=3, top=2)
for c, col in CLS.items():
    g = RS[RS.cls.eq(c)]; ax.scatter(g.max_rhat, np.minimum(g.min_ess_bulk, g.min_ess_tail), s=14, color=col, edgecolor="white", lw=0.4, zorder=3)
ax.axvline(1.01, color=INK, ls="--", lw=0.8); ax.axhline(400, color=INK, ls="--", lw=0.8); ax.set_yscale("log")
ax.set_xlabel("Maximum R-hat"); ax.set_ylabel("Minimum ESS"); style_axis(ax, None)
save(fig, "S14", "p01", "rhat_ess", R_DIR / "results/run_status.csv")
save_key("S14", "k01", "key_a", [Line2D([], [], marker="o", ls="", color=c, mec="white", ms=6) for c in CLS.values()], list(CLS), S["k01"]["w"], S["k01"]["h"], ncol=3)
set_value(n_rp_fits=int(len(RS)), rp_max_rhat=f"{RS.max_rhat.max():.4f}", rp_div=int(pd.to_numeric(RS.divergences).sum()))

PC_ = rd(R_DIR / "results" / "all_predictive_checks.csv"); PC_ = PC_[PC_.variant.eq("primary")]
PC_["posterior_predictive_upper_tail_probability"] = pd.to_numeric(PC_.posterior_predictive_upper_tail_probability, errors="coerce")
PC_ = PC_[np.isfinite(PC_.posterior_predictive_upper_tail_probability)]; PC_["fl"] = tb(PC_.flag)
checks = sorted(PC_.check.unique()); yk = -np.arange(len(checks))
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=52, bottom=13, right=3, top=2); zebra(ax, yk)
rng = np.random.default_rng(1)
for yy, c in zip(yk, checks):
    g = PC_[PC_.check.eq(c)]; jit = rng.uniform(-0.25, 0.25, len(g))
    ax.scatter(g.posterior_predictive_upper_tail_probability, yy + jit, s=6, color=np.where(g.fl, MAGENTA, MID_GREY), lw=0, zorder=3)
ax.set_yticks(yk, [c.replace("_", " ").replace("RNA Protein", "RNA–protein").replace("maximum absolute", "max. abs.") for c in checks], fontsize=FONT - 1)
ax.set_xlim(0, 1); ax.set_ylim(yk.min() - 0.6, 0.6); ax.set_xlabel("Upper-tail probability"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S14", "p02", "predictive_checks", R_DIR / "results/all_predictive_checks.csv")
set_value(rp_flags=int(PC_.fl.sum()), rp_checks=int(len(PC_)))

CV = rd(R_DIR / "bootstrap" / "RNA_gene_coverage_sensitivity_95CI.csv").groupby("programme")[["estimate", "ci_95_lower", "ci_95_upper"]].mean()
mt = BL[BL.quantity.eq("RNA_BMI_per5")].set_index("programme").estimate
cv = CV.join(mt.rename("matched"), how="inner"); cv["full"] = cv.matched - cv.estimate
cv["ex"] = (cv.ci_95_lower > 0) | (cv.ci_95_upper < 0)
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=17, bottom=13, right=3, top=2)
lim = symmetric_span(cv.full, cv.matched); ax.plot([-lim, lim], [-lim, lim], color=MID_GREY, ls="--", lw=0.8)
points(ax, cv.full, cv.matched, np.where(cv.ex, MAGENTA, MID_GREY), cv.ex, size=18)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Full-source RNA slope"); ax.set_ylabel("Matched-gene RNA slope"); style_axis(ax, None)
save(fig, "S14", "p03", "matched_vs_full", R_DIR / "bootstrap/RNA_gene_coverage_sensitivity_95CI.csv")
save_key("S14", "k02", "key_c", [Line2D([], [], marker="o", ls="", color=MAGENTA, mec="white", ms=6), Line2D([], [], marker="o", ls="", mfc="white", mec=MID_GREY, ms=6)],
         ["Difference CI excludes zero", "Includes zero"], S["k02"]["w"], S["k02"]["h"], ncol=2)
set_value(n_cov_sets=int(len(cv)), cov_rho=f"{cv.full.rank().corr(cv.matched.rank()):.2f}")

ID = rd(R_DIR / "results" / "influence_deletion_changes.csv")
set_value(del_max_slope=f"{ID[ID.quantity.isin(['RNA_BMI_per5', 'Protein_BMI_per5'])].posterior_mean_change.abs().max():.4f}",
          del_max_corr=f"{ID[ID.quantity.eq('residual_RNA_Protein_correlation')].posterior_mean_change.abs().max():.4f}")
QN = {"RNA_BMI_per5": "RNA", "Protein_BMI_per5": "protein", "Protein_minus_RNA_BMI_per5": "protein − RNA"}
D = ID[ID.quantity.isin(QN)].copy(); D["qo"] = D.quantity.map(list(QN).index); D = D.sort_values(["programme", "qo"])
y = -np.arange(len(D)); fig, ax = panel(S["p04"]["w"], S["p04"]["h"], left=64, bottom=13, right=22, top=2); zebra(ax, y)
ax.hlines(y, D.deletion_hdi_95_lower, D.deletion_hdi_95_upper, color=dir_colors(D.deletion_posterior_mean), lw=1.6, zorder=2)
ax.scatter(D.deletion_posterior_mean, y, s=22, color=dir_colors(D.deletion_posterior_mean), edgecolor="white", lw=0.6, zorder=4)
ax.scatter(D.primary_posterior_mean, y, s=60, marker="|", color=INK, lw=1.4, zorder=5)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, [f"{LAB.get(p, p).split(' · ')[-1]}, {QN[q]}" for p, q in zip(D.programme, D.quantity)], fontsize=FONT - 2)
right_column(ax, y, [f"−{s}" for s in D.base_sample_id], header="Deleted")
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel("Slope after deletion (SD per 5 kg/m²)"); style_axis(ax, "x"); row_axis(ax)
save(fig, "S14", "p04", "deletion_refits", R_DIR / "results/influence_deletion_changes.csv")
save_key("S14", "k03", "key_d", [Line2D([], [], marker="o", color=MID_GREY, lw=1.6, mec="white", ms=6), Line2D([], [], marker="|", ls="", color=INK, ms=9, mew=1.4)],
         ["After deletion, 95% HDI", "Original mean"], S["k03"]["w"], S["k03"]["h"], ncol=2)

# ---------------- S15
S = SLOT["Figure S15"]
T = rd(R_DIR / "inputs_used" / "sensitivity_targets.csv")
T["label"] = [f"{FAM_PREFIX[f]}: {ct_label(c)} · {sig_label(s)}" for f, c, s in zip(T.family, T.CellType, T.Signature)]
T = T.set_index("programme").loc[lambda d: ~d.index.duplicated()]
y = -np.arange(len(T)); fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=70, bottom=13, right=4, top=2); zebra(ax, y)
for dy, q, col in [(0.16, "RNA_BMI_per5", BLUE), (-0.16, "Protein_BMI_per5", MAGENTA)]:
    v = X.loc[[(p, q) for p in T.index]]
    hdi_forest(ax, y + dy, v.posterior_mean, v.hdi_95_lower, v.hdi_95_upper, v.excl, colors=[col] * len(v), size=24, lw=1.6)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, T.label); ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(RP_UNIT); style_axis(ax, "x"); row_axis(ax)
save(fig, "S15", "p01", "targets_rna_protein", R_DIR / "results/all_effects_95HDI.csv")
save_key("S15", "k01", "key_a", [Line2D([], [], marker="o", color=BLUE, lw=1.6, mec="white", ms=6), Line2D([], [], marker="o", color=MAGENTA, lw=1.6, mec="white", ms=6),
          Line2D([], [], marker="o", color=MID_GREY, lw=1.6, mfc="white", mec=MID_GREY, ms=6)],
         ["RNA", "Protein", "Open marker: 95% HDI includes zero"], S["k01"]["w"], S["k01"]["h"], ncol=3)

SE = rd(R_DIR / "results" / "sensitivity_effects_95HDI.csv"); SE = SE[SE.programme.isin(T.index)]
def vcat(v):
    v = str(v)
    if v == "primary": return "Primary"
    if v.startswith("primary__without") or "without" in v: return None
    if v == "fixed_plex": return "Fixed plex"
    if v.startswith("export_subset"): return "Purity subset"
    if v.startswith("wes_purity") or v.startswith("wes_adjusted"): return "Purity adjusted"
    if v.startswith("stage_subset"): return "Stage subset"
    if v.startswith("stage_adjusted"): return "Stage adjusted"
    if v.startswith("fraction_subset"): return "RNA-fraction subset"
    if v.startswith("fraction_adjusted"): return "RNA-fraction adjusted"
    return v.replace("_", " ").capitalize()
ALL = pd.concat([E[E.programme.isin(T.index)], SE], ignore_index=True); ALL["cat"] = ALL.variant.map(vcat); ALL = ALL[ALL.cat.notna()]
ALL["excl"] = (ALL.hdi_95_lower > 0) | (ALL.hdi_95_upper < 0)
CATS = [c for c in ["Primary", "Fixed plex", "Purity subset", "Purity adjusted", "Stage subset", "Stage adjusted", "RNA-fraction subset",
                    "RNA-fraction adjusted"] if c in set(ALL.cat)] + sorted(set(ALL.cat) - {"Primary", "Fixed plex", "Purity subset", "Purity adjusted",
                    "Stage subset", "Stage adjusted", "RNA-fraction subset", "RNA-fraction adjusted"})
span = symmetric_span(ALL[ALL.quantity.isin(["RNA_BMI_per5", "Protein_BMI_per5"])].posterior_mean)
for slot, q in [("p02", "Protein_BMI_per5"), ("p03", "RNA_BMI_per5")]:
    sub = ALL[ALL.quantity.eq(q)].drop_duplicates(["programme", "cat"])
    M_ = sub.pivot(index="programme", columns="cat", values="posterior_mean").reindex(index=T.index, columns=CATS)
    R_ = sub.pivot(index="programme", columns="cat", values="excl").reindex(index=T.index, columns=CATS).fillna(False)
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=50 if slot == "p02" else 3, bottom=33, right=2, top=2)
    heat(ax, M_.to_numpy(float), span, [l.split(": ", 1)[-1] for l in T.label] if slot == "p02" else [""] * len(T), CATS, rings=R_.to_numpy(bool),
         fontsize=FONT - 2)
    save(fig, "S15", slot, f"sensitivity_{q}", R_DIR / "results/sensitivity_effects_95HDI.csv")
save_colorbar("S15", "k02", "colorbar", DIVERGING, -span, span, "Posterior mean (SD per 5 kg/m²)", S["k02"]["w"], S["k02"]["h"], vertical=True)
save_key("S15", "k03", "key_bc", [Line2D([], [], marker="o", ls="", mfc="white", mec=INK, ms=6), Patch(facecolor=EMPTY_CELL, edgecolor="#9A9DA2")],
         ["95% HDI excludes zero", "Not modelled"], S["k03"]["w"], S["k03"]["h"], ncol=2)
variants = sorted(SE.variant.unique())
print("RNA sensitivity variants:", variants)


In [ ]:
from fig_common import *
CS = rd(X_DIR / "calibration_summary_recomputed.csv")
SCN = ["null", "protein_only", "conditional_signal", "heavy_tails", "sbc"]
SCL_ = {"null": "Null", "protein_only": "Protein only", "conditional_signal": "Conditional signal", "heavy_tails": "Heavy tails", "sbc": "SBC"}
QS = ["beta_ptm", "beta_protein", "beta_conditional", "coupling", "rho", "nu"]
QL = {"beta_ptm": "β PTM (marginal)", "beta_protein": "β protein", "beta_conditional": "β PTM | protein", "coupling": "protein–PTM coupling",
      "rho": "residual correlation ρ", "nu": "Student-t ν"}
SCOPE = [("All completed", BLUE, 0.15), ("Numerically adequate", MAGENTA, -0.15)]
missing = sorted(set(SCN) - set(CS.scenario))
if missing: print("WARNING scenarios missing from calibration table:", missing)
for fig_no, metric, xlab in [("S16", "coverage", "Coverage"), ("S17", "bias", "Bias"), ("S18", "rmse", "RMSE")]:
    S = SLOT[f"Figure {fig_no}"]
    for k, q in enumerate(QS):
        slot = f"p0{k + 1}"; y = -np.arange(len(SCN))
        fig, ax = panel(S[slot]["w"], S[slot]["h"], left=34 if k % 2 == 0 else 6, bottom=13, right=3, top=2); zebra(ax, y)
        for scope, col, dy in SCOPE:
            for yy, sc in zip(y, SCN):
                r = CS[CS.scope.eq(scope) & CS.scenario.eq(sc) & CS.quantity.eq(q)]
                if not len(r): continue
                r = r.iloc[0]
                if metric == "coverage": m, lo, hi = r.coverage, r.coverage_low, r.coverage_high
                elif metric == "bias": m, lo, hi = r.bias, r.bias - 1.96 * r.bias_mcse, r.bias + 1.96 * r.bias_mcse
                else: m, lo, hi = r.rmse, np.nan, np.nan
                if np.isfinite(lo): ax.hlines(yy + dy, lo, hi, color=col, lw=1.6, zorder=2)
                ax.scatter([m], [yy + dy], s=24, color=col, edgecolor="white", lw=0.6, zorder=4)
        if metric == "coverage": ax.axvline(0.95, color=INK, ls="--", lw=0.9, zorder=1); ax.set_xlim(min(0.75, ax.get_xlim()[0]), 1.0)
        elif metric == "bias": ax.axvline(0, color=INK, ls="--", lw=0.9, zorder=1)
        else: ax.set_xlim(0, ax.get_xlim()[1] * 1.08)
        ax.set_yticks(y, [SCL_[s] for s in SCN] if k % 2 == 0 else [""] * len(y)); ax.set_ylim(y.min() - 0.6, 0.6)
        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
        ax.set_xlabel(f"{xlab}, {QL[q]}"); style_axis(ax, "x"); row_axis(ax)
        save(fig, fig_no, slot, f"{metric}_{q}", X_DIR / "calibration_summary_recomputed.csv")
    h = [Line2D([], [], marker="o", color=c, lw=1.6 if metric != "rmse" else 0, mec="white", ms=6) for _, c, _ in SCOPE]
    labs = ["All completed replicates", "Numerically adequate replicates"]
    if metric != "rmse":
        h.append(Line2D([], [], color=INK, ls="--", lw=0.9)); labs.append("Nominal 0.95" if metric == "coverage" else "Zero")
    save_key(fig_no, "k01", "key", h, labs, S["k01"]["w"], S["k01"]["h"], ncol=len(h))

S = SLOT["Figure S19"]; SB = rd(X_DIR / "sbc_rank_bins.csv")
for k, q in enumerate(QS):
    g = SB[SB.quantity.eq(q)].sort_values("bin_left"); x = (g.bin_left + g.bin_right) / 2; w = (g.bin_right - g.bin_left) * 0.9
    fig, ax = panel(S[f"p0{k + 1}"]["w"], S[f"p0{k + 1}"]["h"], left=12, bottom=13, right=2, top=2)
    ax.fill_between(x, g.reference_low, g.reference_high, color=BAND_GREY, step="mid", zorder=1)
    ax.bar(x, g["count"] - g.failed, width=w, color=BLUE, zorder=2); ax.bar(x, g.failed, bottom=g["count"] - g.failed, width=w, color=MAGENTA, zorder=2)
    ax.plot(x, g.expected, color=INK, ls="--", lw=0.9, zorder=3)
    ax.set_xlabel(f"Rank, {QL[q]}"); ax.set_xticks([]); ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(3, integer=True)); style_axis(ax, "y")
    save(fig, "S19", f"p0{k + 1}", f"sbc_{q}", X_DIR / "sbc_rank_bins.csv")
save_key("S19", "k01", "key_a", [Patch(color=BLUE), Patch(color=MAGENTA), Line2D([], [], color=INK, ls="--", lw=0.9), Patch(color=BAND_GREY)],
         ["Numerically adequate", "Numerical failure", "Expected count", "Pointwise 95% reference"], S["k01"]["w"], S["k01"]["h"], ncol=4)
CC = rd(X_DIR / "calibration_completion.csv")
CC = CC.set_index("scenario").reindex([s for s in SCN if s in set(CC.scenario)]).reset_index()
fig, ax = panel(S["p07"]["w"], S["p07"]["h"], left=34, bottom=13, right=8, top=2); y = -np.arange(len(CC))
ax.barh(y, CC.adequate, color=BLUE, height=0.6, zorder=2); ax.barh(y, CC.failed, left=CC.adequate, color=MAGENTA, height=0.6, zorder=2)
for yy, r in zip(y, CC.itertuples()): ax.text(r.completed, yy, f" {r.adequate}/{r.completed}", va="center", fontsize=SIZE["value"])
ax.set_yticks(y, [SCL_[s] for s in CC.scenario]); ax.set_xlim(0, CC.completed.max() * 1.3); ax.set_xlabel("Simulated datasets")
style_axis(ax, "x"); row_axis(ax); save(fig, "S19", "p07", "completion", X_DIR / "calibration_completion.csv")
save_key("S19", "k02", "key_b", [Patch(color=BLUE), Patch(color=MAGENTA)], ["Numerically adequate", "Failed numerical checks"],
         S["k02"]["w"], S["k02"]["h"], ncol=1)
set_value(sim_completed=int(CC.completed.sum()), sim_adequate=int(CC.adequate.sum()))


In [ ]:
from fig_common import *
A = ptm_primary()
VARN = {"primary": "Primary", "fixed_plex": "Fixed plex", "stage_subset_base": "Stage subset", "stage_adjusted": "Stage adjusted",
        "wes_subset_base": "Purity subset", "wes_adjusted": "Purity adjusted", "mutation_adjusted": "Mutation adjusted",
        "exclude_recorded_weight_loss": "No recorded weight loss", "exclude_adenosquamous": "No adenosquamous"}
VORDER = list(VARN)

S = SLOT["Figure S20"]
for slot, a in [("p01", "phospho"), ("p02", "glyco")]:
    g = A[A.adequate & A.assay.eq(a)].sort_values("posterior_mean"); x = np.arange(len(g))
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=17 if slot == "p01" else 8, bottom=13, right=3, top=2)
    ax.fill_between(x, g.hdi_95_lower, g.hdi_95_upper, color=BAND_GREY, lw=0, zorder=1)
    ax.plot(x, g.posterior_mean, color=INK, lw=0.9, zorder=2)
    c = g.excl.to_numpy(); ax.scatter(x[c], g.posterior_mean[c], s=10, color=dir_colors(g.posterior_mean[c]), lw=0, zorder=3)
    ax.axhline(0, color=INK, lw=0.7, zorder=1); ax.set_xlim(-1, len(g))
    ax.set_xlabel(f"{ASSAY_LABEL[a]}, ranked (n = {len(g)})"); ax.set_ylabel(PTM_UNIT if slot == "p01" else ""); style_axis(ax, "y")
    save(fig, "S20", slot, f"ranked_{a}", DA_DIR / "all_primary_features.csv")
save_key("S20", "k01", "key_a", [Line2D([], [], color=INK, lw=0.9), Patch(color=BAND_GREY),
                                 Line2D([], [], marker="o", ls="", color=MAGENTA, ms=5), Line2D([], [], marker="o", ls="", color=BLUE, ms=5)],
         ["Posterior mean", "95% HDI", "HDI excludes zero, positive", "HDI excludes zero, negative"], S["k01"]["w"], S["k01"]["h"], ncol=4)
CD = rd(DA_DIR / "candidates_with_diagnostics.csv")
sens_keys = set(rd(X_DIR / "conditional_sensitivity_effects.csv").key) | set(rd(DA_DIR / "available_continuous_sensitivities.csv").key)
CD["sens"] = CD.key.isin(sens_keys)
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=18, bottom=13, right=3, top=2)
for a in ["phospho", "glyco"]:
    g = CD[CD.assay.eq(a)]
    points(ax, g.max_rhat, np.minimum(g.min_ess_bulk, g.min_ess_tail), np.array([ASSAY_COLOR[a]] * len(g)), g.sens.to_numpy(), size=22)
ax.axvline(1.01, color=INK, ls="--", lw=0.8); ax.axhline(400, color=INK, ls="--", lw=0.8)
ax.set_xlabel("Maximum R-hat"); ax.set_ylabel("Minimum ESS"); style_axis(ax, None)
save(fig, "S20", "p03", "candidate_diagnostics", DA_DIR / "candidates_with_diagnostics.csv")
save_key("S20", "k02", "key_b", [Line2D([], [], marker="o", ls="", color=BLUE, mec="white", ms=6), Line2D([], [], marker="o", ls="", color=MAGENTA, mec="white", ms=6),
                                 Line2D([], [], marker="o", ls="", mfc="white", mec=MID_GREY, ms=6)],
         ["Phosphosite, sensitivity fits", "Glycoform, sensitivity fits", "No sensitivity fits"], S["k02"]["w"], S["k02"]["h"], ncol=1)
set_value(n_candidates=int(len(CD)), n_cand_sens=int(CD.sens.sum()))

S = SLOT["Figure S21"]
for slot, gene in [("p01", "CTSD"), ("p02", "HLA-DRA"), ("p03", "LAMC1"), ("p04", "POSTN")]:
    g = rd(B_DIR / f"within_{gene}_plotted_values.csv"); g = g[tb(g.numerical_checks_pass)].sort_values("posterior_mean")
    y = -np.arange(len(g)); show = len(g) <= 20
    fig, ax = panel(S[slot]["w"], S[slot]["h"], left=36 if show else 12, bottom=13, right=3, top=2)
    if show: zebra(ax, y)
    hdi_forest(ax, y, g.posterior_mean, g.hdi_95_lower, g.hdi_95_upper, tb(g.hdi_excludes_zero), size=16 if not show else 22, lw=1.1 if not show else 1.5)
    ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_ylim(y.min() - 0.7, 0.7)
    if show: ax.set_yticks(y, [site_label("", f).strip() for f in g.feature_id], fontsize=FONT - 2)
    else: ax.set_yticks([])
    ax.set_ylabel(f"{gene} glycoform (glycan composition)" if show else f"{gene} glycoforms, ranked (n = {len(g)})", color=MAGENTA)
    ax.set_xlabel(PTM_UNIT); style_axis(ax, "x"); row_axis(ax)
    save(fig, "S21", slot, f"within_{gene}", B_DIR / f"within_{gene}_plotted_values.csv")
h = hdi_handles(); save_key("S21", "k01", "key", h, [x.get_label() for x in h], S["k01"]["w"], S["k01"]["h"], ncol=1)

S = SLOT["Figure S22"]
CSE = rd(X_DIR / "conditional_sensitivity_effects.csv"); CSE["excl"] = tb(CSE.hdi_excludes_zero); CSE["ok"] = tb(CSE.numerical_checks_pass)
ids = CSE.drop_duplicates("key")[["key", "gene", "feature_id", "assay"]].sort_values(["gene", "feature_id"]).reset_index(drop=True)
ids["E"] = [f"E{i + 1:02d}" for i in range(len(ids))]; ids.to_csv(TABLES / "FigS22_E_identifiers.csv", index=False)
EMAP = dict(zip(ids.key, ids.E)); span = symmetric_span(CSE.posterior_mean)
cols = [v for v in VORDER if v in set(CSE.variant)]
Mv = CSE.pivot_table(index="key", columns="variant", values="posterior_mean", aggfunc="first").reindex(index=ids.key, columns=cols)
Ex = CSE.pivot_table(index="key", columns="variant", values="excl", aggfunc="first").reindex(index=ids.key, columns=cols).fillna(False).astype(bool)
Ok = CSE.pivot_table(index="key", columns="variant", values="ok", aggfunc="first").reindex(index=ids.key, columns=cols).fillna(True).astype(bool)
fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=26, bottom=38, right=2, top=2)
heat(ax, Mv.to_numpy(float), span, [f"{e} {g}" for e, g in zip(ids.E, ids.gene)], [VARN[v] for v in cols],
     rings=(Ex & Ok).to_numpy(), crosses=(~Ok).to_numpy(), fontsize=FONT - 2)
save(fig, "S22", "p01", "candidate_variant_heatmap", X_DIR / "conditional_sensitivity_effects.csv")
save_colorbar("S22", "k01", "colorbar", DIVERGING, -span, span, PTM_UNIT, S["k01"]["w"], S["k01"]["h"], vertical=True)
save_key("S22", "k02", "key_a", [Line2D([], [], marker="o", ls="", mfc="white", mec=INK, ms=6), Line2D([], [], marker="x", ls="", color=INK, ms=6),
                                 Patch(facecolor=EMPTY_CELL, edgecolor="#9A9DA2")],
         ["95% HDI excludes zero", "Numerical failure", "Not fitted"], S["k02"]["w"], S["k02"]["h"], ncol=1)
ICS = rd(X_DIR / "interval_classification_summary.csv")
CMPN = {f"primary → {v}": VARN[v] for v in VORDER[1:]}
CMPN.update({"stage_subset_base → stage_adjusted": "Stage adj. vs subset", "wes_subset_base → wes_adjusted": "Purity adj. vs subset",
             "primary → stage_subset_base": "Stage subset", "primary → wes_subset_base": "Purity subset"})
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=36, bottom=13, right=3, top=2); y = -np.arange(len(ICS))
ax.barh(y, ICS.same_class, color=INK, height=0.6, zorder=2)
ax.barh(y, ICS.changed_class, left=ICS.same_class, color=MAGENTA, height=0.6, zorder=2)
ax.barh(y, ICS.numerical_failures, left=ICS.same_class + ICS.changed_class, color=BAND_GREY, height=0.6, zorder=2)
ax.set_yticks(y, [CMPN.get(c, c) for c in ICS.comparison], fontsize=FONT - 2); ax.set_xlabel("Features (of 24)")
style_axis(ax, "x"); row_axis(ax); save(fig, "S22", "p02", "interval_classification", X_DIR / "interval_classification_summary.csv")
save_key("S22", "k03", "key_b", [Patch(color=INK), Patch(color=MAGENTA), Patch(color=BAND_GREY)], ["Same class", "Changed", "Numerical failure"],
         S["k03"]["w"], S["k03"]["h"], ncol=3)

MCC = rd(X_DIR / "matched_conditional_comparisons.csv")
order = list(ids.key); y = -np.arange(len(order))
for fig_no in ("S23", "S24"):
    S = SLOT[f"Figure {fig_no}"]
    for k, comp in enumerate(COMPARISONS[f"Figure {fig_no}"]):
        vb, va = [x.strip() for x in comp.split("→")]
        g = MCC[MCC.variant_base.eq(vb) & MCC.variant_alt.eq(va)].drop_duplicates("key").set_index("key").reindex(order)
        if g.posterior_mean_alt.notna().sum() == 0: print(f"WARNING {fig_no}: no rows for {comp}")
        fig, ax = panel(S[f"p0{k + 1}"]["w"], S[f"p0{k + 1}"]["h"], left=31 if k % 2 == 0 else 4, bottom=13, right=3, top=2); zebra(ax, y)
        for side, col, dy in [("base", BLUE, 0.17), ("alt", MAGENTA, -0.17)]:
            ok = tb(g[f"numerical_checks_pass_{side}"]) & g[f"posterior_mean_{side}"].notna().to_numpy(); ex = tb(g[f"hdi_excludes_zero_{side}"])
            m, lo, hi = g[f"posterior_mean_{side}"].to_numpy(float), g[f"hdi_95_lower_{side}"].to_numpy(float), g[f"hdi_95_upper_{side}"].to_numpy(float)
            ax.hlines(y[ok] + dy, lo[ok], hi[ok], color=col, lw=1.3, zorder=2)
            points(ax, m[ok], y[ok] + dy, np.array([col] * ok.sum()), ex[ok], size=16)
            ax.scatter(m[~ok], y[~ok] + dy, marker="x", s=18, color=MID_GREY, lw=1.0, zorder=3)
        ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_ylim(y.min() - 0.6, 0.6)
        ax.set_yticks(y, [f"{EMAP[x]} {g_}" for x, g_ in zip(order, ids.gene)] if k % 2 == 0 else [""] * len(y), fontsize=FONT - 2)
        ax.set_xlabel(f"{CMPN.get(comp, comp)} (SD per 5 kg/m²)"); style_axis(ax, "x"); row_axis(ax)
        if k % 2 == 0: ax.set_ylabel("Candidate feature (E code, gene)", color=INK)
        save(fig, fig_no, f"p0{k + 1}", comp.replace(" → ", "_to_"), X_DIR / "matched_conditional_comparisons.csv")
    save_key(fig_no, "k01", "key", [Line2D([], [], marker="o", color=BLUE, lw=1.3, mec="white", ms=6), Line2D([], [], marker="o", color=MAGENTA, lw=1.3, mec="white", ms=6),
                                    Line2D([], [], marker="o", color=MID_GREY, lw=1.3, mfc="white", mec=MID_GREY, ms=6), Line2D([], [], marker="x", ls="", color=MID_GREY, ms=6)],
             ["Reference fit", "Sensitivity fit", "Open: 95% HDI includes zero", "Failed numerical checks"], S["k01"]["w"], S["k01"]["h"], ncol=4)


In [ ]:
from fig_common import *
PE = rd(P_DIR / "all_effects_95HDI.csv"); PE["excl"] = tb(PE.hdi_excludes_zero)
prim = PE[PE.variant.eq("primary") & PE.term.eq("bmi_5_within")]
st = prim[prim.gene.eq("STAT3") & prim.assay.eq("phospho") & prim.feature_id.str.contains(r"_(?:Y705|S727)$")]
QN = {"Protein": "Total STAT3 protein", "PTM": "Phosphosite, marginal", "PTM_given_Protein": "Phosphosite | protein"}

# ---------------- S25 a: STAT3; b: ASAH1 glycoforms
S = SLOT["Figure S25"]
rows = [(f"{fid.rsplit('_', 1)[-1]}: {QN[q]}", r) for fid in sorted(st.feature_id.unique()) for q in QN
        for _, r in st[st.feature_id.eq(fid) & st.quantity.eq(q)].iterrows()]
y = -np.arange(len(rows)); fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=56, bottom=13, right=3, top=2); zebra(ax, y)
hdi_forest(ax, y, [r.posterior_mean for _, r in rows], [r.hdi_95_lower for _, r in rows], [r.hdi_95_upper for _, r in rows],
           [r.excl for _, r in rows], size=24, lw=1.6)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_yticks(y, [l for l, _ in rows], fontsize=FONT - 1)
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(PTM_UNIT); style_axis(ax, "x"); row_axis(ax); ax.set_ylabel("STAT3 site: quantity", color=BLUE)
save(fig, "S25", "p01", "stat3_sites", P_DIR / "all_effects_95HDI.csv")
asah = prim[prim.gene.eq("ASAH1") & prim.assay.eq("glyco") & prim.quantity.eq("PTM_given_Protein")].sort_values("feature_id").reset_index(drop=True)
asah["A"] = [f"A{i + 1:02d}" for i in range(len(asah))]; asah[["A", "key", "feature_id"]].to_csv(TABLES / "FigS25_A_identifiers.csv", index=False)
AMAP = dict(zip(asah.key, asah.A)); set_value(n_asah1=int(len(asah)))
b = asah.sort_values("posterior_mean"); y = -np.arange(len(b))
fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=32, bottom=13, right=3, top=2); zebra(ax, y)
hdi_forest(ax, y, b.posterior_mean, b.hdi_95_lower, b.hdi_95_upper, b.excl, size=22, lw=1.5)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, [f"{a} {site_label('', f).strip()}" for a, f in zip(b.A, b.feature_id)], fontsize=FONT - 2)
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(PTM_UNIT); style_axis(ax, "x"); row_axis(ax); ax.set_ylabel("ASAH1 glycoform (A code, glycan)", color=MAGENTA)
save(fig, "S25", "p02", "asah1_glycoforms", P_DIR / "all_effects_95HDI.csv")
h = hdi_handles(); save_key("S25", "k01", "key", h, [x.get_label() for x in h], S["k01"]["w"], S["k01"]["h"], ncol=1)

S = SLOT["Figure S26"]
VARN = {"fixed_plex": "Fixed plex", "stage_subset_base": "Stage subset", "stage_adjusted": "Stage adjusted", "wes_subset_base": "Purity subset",
        "wes_adjusted": "Purity adjusted", "mutation_adjusted": "Mutation adjusted", "exclude_recorded_weight_loss": "No weight loss",
        "exclude_adenosquamous": "No adenosquamous"}
AC = rd(DA_DIR / "available_continuous_sensitivities.csv"); AC = AC[AC.gene.eq("STAT3")].sort_values(["feature_id", "variant_comparison"])
y = -np.arange(len(AC)); fig, ax = panel(S["p01"]["w"], S["p01"]["h"], left=50, bottom=13, right=3, top=2); zebra(ax, y)
ax.hlines(y + 0.17, AC.hdi_95_lower_base, AC.hdi_95_upper_base, color=LINE, lw=1.2, zorder=2)
ax.scatter(AC.posterior_mean_base, y + 0.17, s=16, facecolor="white", edgecolor=MID_GREY, lw=1.0, zorder=3)
hdi_forest(ax, y - 0.13, AC.posterior_mean_comparison, AC.hdi_95_lower_comparison, AC.hdi_95_upper_comparison,
           tb(AC.hdi_excludes_zero_comparison), size=20, lw=1.4)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, [f"{f.rsplit('_', 1)[-1]}: {VARN.get(v, v)}" for f, v in zip(AC.feature_id, AC.variant_comparison)], fontsize=FONT - 2)
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel(PTM_UNIT); style_axis(ax, "x"); row_axis(ax); ax.set_ylabel("STAT3 site: sensitivity fit", color=BLUE)
save(fig, "S26", "p01", "stat3_sensitivity", DA_DIR / "available_continuous_sensitivities.csv")
CT = rd(DA_DIR / "categorical_target_contrasts.csv"); CT = CT[CT.gene.eq("STAT3")].sort_values(["feature_id", "term"])
y = -np.arange(len(CT)); fig, ax = panel(S["p02"]["w"], S["p02"]["h"], left=42, bottom=13, right=3, top=2); zebra(ax, y)
hdi_forest(ax, y, CT.posterior_mean, CT.hdi_95_lower, CT.hdi_95_upper, tb(CT.hdi_excludes_zero), size=22, lw=1.5)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1)
ax.set_yticks(y, [f"{f.rsplit('_', 1)[-1]}: {'obese' if t.startswith('obese') else 'overweight'}" for f, t in zip(CT.feature_id, CT.term)])
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel("Difference vs normal weight (SD)"); style_axis(ax, "x"); row_axis(ax); ax.set_ylabel("STAT3 site: BMI group", color=BLUE)
save(fig, "S26", "p02", "stat3_categorical", DA_DIR / "categorical_target_contrasts.csv")
save_key("S26", "k01", "key_ab", [Line2D([], [], marker="o", color=MID_GREY, lw=1.2, mfc="white", mec=MID_GREY, ms=6), *hdi_handles()[:1],
                                  Line2D([], [], marker="o", color=BLUE, lw=1.6, mfc="white", mec=BLUE, ms=6)],
         ["Reference fit (a)", "Sensitivity fit, HDI excludes zero", "Sensitivity fit, HDI includes zero"], S["k01"]["w"], S["k01"]["h"], ncol=1)
BB = rd(P_DIR / "bootstrap_target_linear_benchmark.csv")
st_keys = set(st.key); bb = pd.concat([BB[BB.key.isin(st_keys)].assign(lab=lambda d: [f"STAT3 {f.rsplit('_', 1)[-1]}: {QN[q]}" for f, q in zip(d.feature_id, d.quantity)]),
                                       BB[BB.key.isin(AMAP) & BB.quantity.eq("PTM_given_Protein")].assign(lab=lambda d: [f"ASAH1 {AMAP[k]}" for k in d.key])])
bb = bb.sort_values("lab"); y = -np.arange(len(bb)); ex = (bb.ci_95_lower > 0) | (bb.ci_95_upper < 0)
fig, ax = panel(S["p03"]["w"], S["p03"]["h"], left=66, bottom=13, right=3, top=2); zebra(ax, y)
hdi_forest(ax, y, bb.estimate, bb.ci_95_lower, bb.ci_95_upper, ex, size=20, lw=1.4)
ax.axvline(0, color=INK, lw=SIZE["zero_line"], zorder=1); ax.set_yticks(y, bb.lab, fontsize=FONT - 2)
ax.set_ylim(y.min() - 0.6, 0.6); ax.set_xlabel("Linear benchmark slope (SD per 5 kg/m² within plex)"); style_axis(ax, "x"); row_axis(ax); ax.set_ylabel("Feature: quantity", color=INK)
save(fig, "S26", "p03", "bootstrap_benchmark", P_DIR / "bootstrap_target_linear_benchmark.csv")
save_key("S26", "k02", "key_c", hdi_handles("Positive, 95% CI excludes zero", "Negative, 95% CI excludes zero")[:2] +
         [Line2D([], [], marker="o", color=MID_GREY, lw=1.6, mfc="white", mec=MID_GREY, ms=6)],
         ["Positive, 95% CI excludes zero", "Negative, 95% CI excludes zero", "95% CI includes zero"], S["k02"]["w"], S["k02"]["h"], ncol=3)
n_ex = int(tb(AC.hdi_excludes_zero_comparison).sum() + tb(CT.hdi_excludes_zero).sum())
set_value(stat3_note="No STAT3 interval excluded zero in any variant." if n_ex == 0 else f"{n_ex} STAT3 sensitivity intervals excluded zero.")


In [ ]:
import io, json, re
from pathlib import Path
import pandas as pd
from PIL import Image
from lxml import etree
from pptx import Presentation
from pptx.util import Mm, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_AUTO_SIZE, MSO_ANCHOR
from pptx.opc.constants import RELATIONSHIP_TYPE as RT
from pptx.opc.package import Part
from pptx.opc.packuri import PackURI
from pptx.oxml.xmlchemy import OxmlElement
from docx import Document
from docx.shared import Pt as DPt
from fig_common import OUT, TIFF_DIR, SVG_LOCAL, SVG_DRIVE

L = json.loads((OUT / "layout.json").read_text()); PAGE, FIGS = L["page"], L["figures"]
CAP = json.loads((OUT / "captions.json").read_text())
VAL = json.loads((OUT / "values.json").read_text()) if (OUT / "values.json").exists() else {}
MAN = pd.read_csv(OUT / "manifest.csv")
DECKS = {"main": OUT / "Main_Figures_r10.pptx", "supp": OUT / "Supplementary_Figures_r10.pptx"}
problems = []

def caption_text(fig):
    c = CAP[fig]; body = " ".join(c["panels"])
    missing = sorted(set(re.findall(r"{(\w+)}", body + c["title"])) - set(VAL))
    if missing:
        problems.append(f"{fig}: caption values missing {missing}")
    fill = lambda s: re.sub(r"{(\w+)}", lambda m: str(VAL.get(m.group(1), "{" + m.group(1) + "}")), s)
    return fill(c["title"]), fill(body)

def attach_svg(pic, slide, svg_path):
    blob = Path(svg_path).read_bytes()
    pkg = slide.part.package
    names = {str(p.partname) for p in pkg.iter_parts()}; n = 1
    while f"/ppt/media/r10_svg_{n}.svg" in names:
        n += 1
    part = Part(PackURI(f"/ppt/media/r10_svg_{n}.svg"), "image/svg+xml", pkg, blob)
    rid = slide.part.relate_to(part, RT.IMAGE)
    ext_list, ext = OxmlElement("a:extLst"), OxmlElement("a:ext")
    ext.set("uri", "{96DAC541-7B7A-43D3-8B79-37D633B846F1}")
    node = etree.Element("{http://schemas.microsoft.com/office/drawing/2016/SVG/main}svgBlip",
                         nsmap={"asvg": "http://schemas.microsoft.com/office/drawing/2016/SVG/main"})
    node.set("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed", rid)
    ext.append(node); ext_list.append(ext); pic._element.blipFill.blip.append(ext_list)

def textbox(slide, x, y, w, h, runs, size, anchor=MSO_ANCHOR.TOP):
    tb = slide.shapes.add_textbox(Mm(x), Mm(y), Mm(w), Mm(h)); tf = tb.text_frame
    tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
    tf.word_wrap, tf.auto_size, tf.vertical_anchor = True, MSO_AUTO_SIZE.NONE, anchor
    p = tf.paragraphs[0]
    for text, bold in runs:
        r = p.add_run(); r.text = text; r.font.name = "Arial"; r.font.size = Pt(size); r.font.bold = bold
        r.font.color.rgb = RGBColor(0, 0, 0)
    return tb

def overlap(a, b):
    return not (a[0] + a[2] <= b[0] or b[0] + b[2] <= a[0] or a[1] + a[3] <= b[1] or b[1] + b[3] <= a[1])

legends = []
for deck, path in DECKS.items():
    def _order(f):
        m = re.search(r"\(continued (\d+)\)", f); return (int(re.sub(r"\D", "", FIGS[f]["number"]) or 0), int(m.group(1)) if m else 1)
    figs = sorted([f for f, d in FIGS.items() if d["deck"] == deck], key=_order)
    if not figs:
        continue
    prs = Presentation(); prs.slide_width, prs.slide_height = Mm(PAGE["width_mm"]), Mm(PAGE["height_mm"])
    for fig in figs:
        d = FIGS[fig]; tag = ("F" if deck == "main" else "S") + d["number"]
        slide = prs.slides.add_slide(prs.slide_layouts[6]); boxes = []
        for s in d["slots"]:
            row = MAN[(MAN.figure == tag) & (MAN.slot == s["slot"])]
            if row.empty:
                problems.append(f"{fig} {s['slot']}: no file in manifest"); continue
            row = row.iloc[-1]
            for dim, key in (("w", "width_mm"), ("h", "height_mm")):
                if abs(row[key] / s[dim] - 1) > 0.02:
                    problems.append(f"{fig} {s['slot']}: drawn {row[key]} mm vs slot {s[dim]} mm ({dim})")
            svg = SVG_LOCAL / row.file if (SVG_LOCAL / row.file).exists() else SVG_DRIVE / row.file
            im = Image.open(TIFF_DIR / row.tiff); px = (round(s["w"] / 25.4 * 300), round(s["h"] / 25.4 * 300))
            buf = io.BytesIO(); im.resize(px, Image.LANCZOS).save(buf, format="PNG", dpi=(300, 300)); buf.seek(0)
            pic = slide.shapes.add_picture(buf, Mm(s["x"]), Mm(s["y"]), Mm(s["w"]), Mm(s["h"]))
            attach_svg(pic, slide, svg)
            boxes.append((s["slot"], (s["x"], s["y"], s["w"], s["h"])))
            if s.get("letter"):
                textbox(slide, s["x"] - 4.5, s["y"] - 1.0, 5, 5, [(s["letter"], True)], PAGE["letter_pt"])
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                if overlap(boxes[i][1], boxes[j][1]):
                    problems.append(f"{fig}: {boxes[i][0]} overlaps {boxes[j][0]}")
        title, body = caption_text(fig)
        cy = d["caption_y"]; ch = PAGE["height_mm"] - PAGE["margin_mm"] + 5 - cy
        label = fig.split(" (")[0]
        textbox(slide, PAGE["margin_mm"], cy, PAGE["width_mm"] - 2 * PAGE["margin_mm"], ch,
                [(f"{label}. {title} ", True), (body, False)], PAGE["caption_pt"])
        n_chars = len(title) + len(body) + 12
        need = n_chars / ((PAGE["width_mm"] - 30) / (PAGE["caption_pt"] * 0.353 * 0.5)) * PAGE["caption_pt"] * 0.353 * 1.25
        if need > ch:
            problems.append(f"{fig}: caption may not fit ({need:.0f} mm needed, {ch:.0f} mm available)")
        letters = {s["letter"] for s in d["slots"] if s.get("letter")}
        cited = set()
        for grp in re.findall(r"\(([a-h](?:\s*[,–]\s*[a-h])*)[\);]", body):
            for a, b in re.findall(r"([a-h])–([a-h])", grp):
                cited |= set("abcdefgh"["abcdefgh".index(a):"abcdefgh".index(b) + 1])
            cited |= set(re.findall(r"[a-h]", grp))
        if letters - cited:
            problems.append(f"{fig}: panels {sorted(letters - cited)} not described in the caption")
        if "continued" not in fig:
            legends.append((fig, title, body))
        elif CAP[fig].get("append") and legends:
            f0, t0, b0 = legends[-1]; legends[-1] = (f0, t0, b0 + " " + body)
    prs.save(path); print("saved", path)

doc = Document()
for fig, title, body in legends:
    p = doc.add_paragraph(); r = p.add_run(f"{fig}. {title} "); r.bold = True; r.font.size = DPt(10)
    r2 = p.add_run(body); r2.font.size = DPt(10)
doc.save(OUT / "figure_legends_r10.docx"); print("saved", OUT / "figure_legends_r10.docx")
print("\nCHECKS:", "all passed" if not problems else "")
for p in problems:
    print(" -", p)


In [ ]:
import zipfile, shutil, re
from pathlib import Path
import pandas as pd
from fig_common import OUT, SVG_LOCAL, SVG_DRIVE, TIFF_DIR
MAN = pd.read_csv(OUT / "manifest.csv")
def folder(tag):
    return f"Main_Figures/Figure_{tag[1:]}" if tag.startswith("F") else f"Supplementary_Figures/Figure_S{tag[1:]}"
ORG = OUT / "organized"
if ORG.exists(): shutil.rmtree(ORG)
z = Path("/content/r10_figures.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as fz:
    for _, r in MAN.sort_values(["figure", "slot"]).iterrows():
        svg = SVG_LOCAL / r.file if (SVG_LOCAL / r.file).exists() else SVG_DRIVE / r.file
        d = folder(r.figure)
        fz.write(svg, f"{d}/{r.file}")
        (ORG / d).mkdir(parents=True, exist_ok=True)
        shutil.copy2(svg, ORG / d / r.file)
        if (TIFF_DIR / r.tiff).exists(): shutil.copy2(TIFF_DIR / r.tiff, ORG / d / r.tiff)
    for n in ["Main_Figures_r10.pptx", "Supplementary_Figures_r10.pptx", "figure_legends_r10.docx",
              "manifest.csv", "layout.json", "captions.json", "values.json"]:
        if (OUT / n).exists(): fz.write(OUT / n, n)
print("folders:", sorted({folder(t) for t in MAN.figure}))
from google.colab import files; files.download(str(z))
